# Reasoning-trace trajectory analysis (cell level + pooled + compute level)

**Type:** observational research analysis. No LLM inference, no new labels, no causal claims.

**Comparison unit:** the *cell* = `benchmark::question_id` x `compute_level` (5 samples). Cells of one question are
not independent, so every pooled number is built as **cell -> question (equal weight) -> pooled**, and all
confidence intervals come from a **cluster bootstrap that resamples questions**.

**Length** is measured in *words* (whitespace tokens after `<br>` -> space and zero-width characters removed).
Token counts are supplementary only.

Everything is exploratory: effect sizes are reported with CIs and n, with no multiplicity correction.

Sections: 1 Config -> 2 Load -> 3 Data quality -> 4 Segment reconstruction -> 5 Features -> 6 Cells and groups ->
7 Part A (cell level) -> 8 Part B (pooled) -> 9 Part C (compute level) -> 10 Textual analyses ->
11 Statistics and robustness -> 12 Findings.

## 1. Config and imports

In [ ]:
import os, re, json, math, zlib, warnings, hashlib, difflib
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import roc_auc_score
from IPython.display import display, Markdown

# ------------------------------------------------------------------ the config (only these two paths) ---------
CSV_PATH = os.environ.get("TRACE_CSV", "/kaggle/input/reasoning-traces/traces_labeled.csv")
OUT_DIR = os.environ.get("TRACE_OUT", "/kaggle/working/trajectory_analysis")

# ------------------------------------------------------------------ analysis parameters (defaults, no edits needed)
SEED = 0
MIN_ELIGIBLE = 4            # eligible samples needed for a cell (spec: >= 4)
SPREAD_SD, SPREAD_RATIO = 1.5, 1.5   # ALL-CORRECT spread rule: max-min >= SPREAD_SD*SD and max/min >= SPREAD_RATIO
N_BOOT = 2000               # question-level bootstrap resamples
N_PERM = 500                # permutation-baseline repetitions
N_INLINE = 3                # representative cells shown inline per group
SAVE_ALL_CELL_FIGURES = os.environ.get("TRACE_FAST") != "1"   # True = one figure + one text table for every eligible cell
REL_BINS = 10               # relative-position bins (deciles)
ABS_BIN_WORDS = 100         # absolute-position bin width in words
ABS_MIN_TRACES = 20         # an absolute bin is drawn only if at least this many traces reach it
REP_SIM = 0.60              # TF-IDF cosine at/above which a segment is a "near repeat" of an earlier segment
CP_SIM = 0.30               # cosine below which a long-trace segment has "no counterpart" in the short trace
DIV_SIM = 0.35              # text-divergence threshold (segment has no counterpart in the other trace)
MIN_Q_NGRAM = 15            # minimum number of questions supporting a reported n-gram

ROLES = ["setup", "reasoning", "doubt", "verification", "correction"]
KEY_ROLES = ["doubt", "verification", "correction"]
STATUSES = ["none", "candidate", "final"]
TEMPORAL = ["new", "continuation"]
REQUIRED_COLS = ["question_id", "benchmark", "difficulty_tier", "task_type", "compute_level", "sample_index",
                 "raw_model_output", "thinking_text", "gold", "is_correct", "tokens_used", "thinking_tokens",
                 "output_tokens", "hit_response_marker", "was_truncated", "hit_marker_empty_output",
                 "rtl_status", "rtl_n_segments", "rtl_labels_json"]

for sub in ["cell_figures", "cell_figures/normalized", "pooled_figures", "tables", "tables/cell_text"]:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)
FIG_DIR, POOL_DIR, TAB_DIR = (os.path.join(OUT_DIR, s) for s in ["cell_figures", "pooled_figures", "tables"])
RNG = np.random.default_rng(SEED)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 80)
sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
# expected and harmless: nanmean over bins/questions that no trace reaches
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="All-NaN slice")

# One colour per discourse role, identical in every figure.
ROLE_COLORS = {"setup": "#9aa5b1", "reasoning": "#4c78a8", "doubt": "#f58518", "verification": "#54a24b",
               "correction": "#e45756"}


def save_table(df, name, index=True):
    path = os.path.join(TAB_DIR, name)
    df.to_csv(path, index=index)
    return path


def save_fig(fig, name, folder=None, dpi=130):
    path = os.path.join(folder or POOL_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    return path

### Helper functions (text normalisation, statistics)
Statistics follow the hierarchy *cell -> question -> pooled*: functions below take a **question-level** table
(one row per question) and resample **questions** with replacement.

In [ ]:
ZW_RE = re.compile("[\u200b\u200c\u200d\u2060\ufeff]")
BR_RE = re.compile(r"<br\s*/?>", re.I)
MARK_RE = re.compile("\u27e6S(\\d+)\u27e7")


def clean_text(s):
    """<br> -> space, zero-width characters removed. Used for word counts and text analyses only."""
    return ZW_RE.sub("", BR_RE.sub(" ", s))


def word_count(s):
    return len(clean_text(s).split())


def norm_answer(x):
    """Light answer normalisation (case, \\boxed{}, $, spaces, thousands commas, trailing period; numbers canonical)."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().lower()
    s = re.sub(r"\\boxed\s*\{(.*)\}", r"\1", s)
    s = re.sub(r"\\text\s*\{(.*?)\}", r"\1", s)
    s = s.replace("\\dfrac", "\\frac").replace("\\left", "").replace("\\right", "")
    s = re.sub(r"\\[,!; ]", "", s).replace("$", "").replace(" ", "")
    s = re.sub(r"(?<=\d),(?=\d{3}(\D|$))", "", s)
    s = s.rstrip(".")
    if s == "":
        return None
    try:
        return "%.10g" % float(s)
    except ValueError:
        return s


def is_refinement(a, b):
    """Two different normalised answers where one contains the other (e.g. 'x=3' vs '3'): refinement/format."""
    return a != b and (a in b or b in a)


def nanpct(x, q):
    """Percentile ignoring NaN; NaN (no warning) when nothing finite is left."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    return np.percentile(x, q) if len(x) else np.nan


def paired_summary(qdf, cols=None, B=None, seed=SEED):
    """Question-level summary of per-question differences: mean, bootstrap CI, dz (+CI), sign-flip p, Wilcoxon p."""
    rng = np.random.default_rng(seed)
    B = B or N_BOOT
    cols = list(qdf.columns) if cols is None else cols
    rows = []
    for c in cols:
        v = qdf[c].dropna().to_numpy(float)
        n = len(v)
        if n < 3:
            rows.append(dict(feature=c, n_questions=n))
            continue
        idx = rng.integers(0, n, (B, n))
        vi = v[idx]
        bm = vi.mean(1)
        sd = v.std(ddof=1)
        bsd = vi.std(1, ddof=1)
        with np.errstate(divide="ignore", invalid="ignore"):
            bdz = np.where(bsd > 0, bm / bsd, np.nan)
        dz = v.mean() / sd if sd > 0 else np.nan
        flips = rng.choice([-1.0, 1.0], (B, n))
        p_sf = (1 + (np.abs((flips * v).mean(1)) >= abs(v.mean()) - 1e-12).sum()) / (B + 1)
        p_w = np.nan if np.all(v == 0) else stats.wilcoxon(v).pvalue
        rows.append(dict(feature=c, n_questions=n, mean=v.mean(), ci_lo=np.percentile(bm, 2.5),
                         ci_hi=np.percentile(bm, 97.5), dz=dz, dz_lo=nanpct(bdz, 2.5), dz_hi=nanpct(bdz, 97.5),
                         p_signflip=p_sf, p_wilcoxon=p_w))
    return pd.DataFrame(rows).set_index("feature")


def boot_mean_ci(v, B=None, seed=SEED):
    v = np.asarray(v, float)
    v = v[~np.isnan(v)]
    if len(v) < 3:
        return np.nan, np.nan, np.nan, len(v)
    rng = np.random.default_rng(seed)
    bm = v[rng.integers(0, len(v), (B or N_BOOT, len(v)))].mean(1)
    return v.mean(), np.percentile(bm, 2.5), np.percentile(bm, 97.5), len(v)


def boot_matrix_ci(X, B=300, seed=SEED):
    """Column-wise mean and bootstrap CI over the ROWS (questions) of a 2-D array that may contain NaN."""
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    means = np.nanmean(X, axis=0) if n else np.full(X.shape[1], np.nan)
    boots = np.empty((B, X.shape[1]))
    for b in range(B):
        boots[b] = np.nanmean(X[rng.integers(0, n, n)], axis=0)
    return means, np.nanpercentile(boots, 2.5, axis=0), np.nanpercentile(boots, 97.5, axis=0)


def weighted_ecdf(vals, w):
    vals, w = np.asarray(vals, float), np.asarray(w, float)
    ok = ~np.isnan(vals)
    vals, w = vals[ok], w[ok]
    o = np.argsort(vals)
    return vals[o], np.cumsum(w[o]) / w.sum()


def safe_name(s):
    return re.sub(r"[^A-Za-z0-9._-]+", "-", str(s)).strip("-")[:80]


def hier_mean(df, feats, cell_col="cell_id", q_col="qkey"):
    """trace -> cell mean -> question mean (NaN skipped at each level). Returns question-level DataFrame."""
    c = df.groupby(cell_col, sort=False)[feats].mean()
    cq = df.drop_duplicates(cell_col).set_index(cell_col)[q_col]
    c[q_col] = cq.reindex(c.index).to_numpy()
    return c.groupby(q_col, sort=False)[feats].mean()

## 2. Load and inspect
The CSV is read as text (so ids, answers and booleans survive untouched). Only the string `''` is treated as
missing; `'None'`/`'NA'` stay as they are because they can be legitimate gold answers. Source columns are never
renamed; derived columns get new names.

In [ ]:
raw = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False, na_values=[""], encoding="utf-8")
missing_cols = [c for c in REQUIRED_COLS if c not in raw.columns]
assert not missing_cols, "Missing required columns: %s" % missing_cols
extra_cols = [c for c in raw.columns if c not in REQUIRED_COLS]
print("rows: %d  columns: %d  extra (unused) columns: %s" % (len(raw), raw.shape[1], extra_cols))


def parse_bool(s):
    m = {"true": 1.0, "1": 1.0, "1.0": 1.0, "yes": 1.0, "y": 1.0, "t": 1.0,
         "false": 0.0, "0": 0.0, "0.0": 0.0, "no": 0.0, "n": 0.0, "f": 0.0}
    return s.str.strip().str.lower().map(m)


df = raw.copy()
df["trace_id"] = np.arange(len(df))
df["qkey"] = df["benchmark"].astype(str) + "::" + df["question_id"].astype(str)
df["cell_id"] = df["qkey"] + "@" + df["compute_level"].astype(str)
df["correct"] = parse_bool(df["is_correct"])            # 1.0 / 0.0 / NaN (unknown)
df["trunc"] = parse_bool(df["was_truncated"])           # 1.0 / 0.0 / NaN (unknown)
df["empty_out"] = parse_bool(df["hit_marker_empty_output"])
df["hit_marker"] = parse_bool(df["hit_response_marker"])
for c in ["tokens_used", "thinking_tokens", "output_tokens", "rtl_n_segments", "sample_index"]:
    df[c + "_num"] = pd.to_numeric(df[c], errors="coerce")
unparsed = {c: int(df[s].isna().sum() - df[c].isna().sum()) for c, s in
            [("is_correct", "correct"), ("was_truncated", "trunc")]}
print("values that could not be parsed as booleans:", unparsed)

display(df[REQUIRED_COLS].isna().sum().rename("n_missing").to_frame().T)
for c in ["benchmark", "difficulty_tier", "task_type", "compute_level", "rtl_status"]:
    print("\n" + c); print(df[c].value_counts(dropna=False).to_string())
print("\nis_correct:", df["correct"].value_counts(dropna=False).to_dict(),
      "| was_truncated:", df["trunc"].value_counts(dropna=False).to_dict())
display(df[["tokens_used_num", "thinking_tokens_num", "output_tokens_num", "rtl_n_segments_num"]].describe().T)

## 3. Data quality (row level, before reconstruction)
Checks on the raw table: duplicates, the 5-samples-per-cell assumption, metadata consistency, text artifacts.
Reconstruction-specific checks (JSON, label matching) follow in section 4.

In [ ]:
KEY_COLS = ["benchmark", "question_id", "compute_level", "sample_index"]
df["dup_key"] = df.duplicated(KEY_COLS, keep=False)
dq = []


def add_dq(check, n_rows, note=""):
    dq.append(dict(check=check, n_rows=int(n_rows), share=n_rows / len(df), note=note))


add_dq("rows total", len(df))
add_dq("duplicate (benchmark, question_id, compute_level, sample_index) key", df["dup_key"].sum(),
       "all rows sharing a duplicated key are excluded (ambiguous)")
add_dq("empty thinking_text", df["thinking_text"].isna().sum())
add_dq("empty rtl_labels_json", df["rtl_labels_json"].isna().sum())
add_dq("correctness unknown (is_correct unparseable/missing)", df["correct"].isna().sum())
add_dq("truncation status unknown", df["trunc"].isna().sum())
add_dq("gold missing", df["gold"].isna().sum())
tt = df["thinking_text"].fillna("")
add_dq("thinking_text contains <br> tags", tt.str.contains(BR_RE).sum(), "replaced by a space for counting")
add_dq("thinking_text contains zero-width characters", tt.str.contains(ZW_RE).sum(), "removed for counting")
ENC_RE = re.compile("(?:\u00c3.|\u00e2\u20ac.|\u00c2\u00a0|\ufffd)")
add_dq("thinking_text shows encoding artifacts (mojibake / U+FFFD)", tt.str.contains(ENC_RE).sum(),
       "reported only; text left untouched")
n_open = tt.str.count("\u27e6")
n_mark = tt.str.count(MARK_RE)
add_dq("thinking_text with malformed markers (opening bracket count != valid marker count)", (n_open != n_mark).sum())
add_dq("thinking_text with no marker at all", ((n_mark == 0) & df["thinking_text"].notna()).sum())

# 5-samples-per-cell assumption (rows per cell, before any filtering)
cell_rows = df.groupby("cell_id").agg(n_rows=("trace_id", "size"), n_unique_idx=("sample_index", "nunique"))
print("rows per cell (expected 5):\n", cell_rows["n_rows"].value_counts().sort_index().to_string())
off5 = cell_rows[cell_rows["n_rows"] != 5]
print("cells with != 5 rows: %d of %d" % (len(off5), len(cell_rows)))
save_table(off5, "cells_not_five_rows.csv")
add_dq("rows in cells that do not have exactly 5 rows", df["cell_id"].isin(off5.index).sum(),
       "kept; the 4-sample rule is applied later after exclusions")

# metadata consistency within a question
per_q = df.groupby("qkey").agg(n_tier=("difficulty_tier", "nunique"), n_task=("task_type", "nunique"))
add_dq("questions with several difficulty_tier values", (per_q["n_tier"] > 1).sum(), "units are questions, not rows")
add_dq("questions with several task_type values", (per_q["n_task"] > 1).sum(), "units are questions, not rows")
ids_multi = df.groupby("question_id")["benchmark"].nunique()
add_dq("question_id values that occur in more than one benchmark", (ids_multi > 1).sum(),
       "units are ids; qkey = benchmark::question_id keeps them apart")
dq_df = pd.DataFrame(dq)
display(dq_df)
print("\ntruncation x correctness (rows):")
display(pd.crosstab(df["trunc"].fillna(-1), df["correct"].fillna(-1)))

## 4. Segment reconstruction and annotation matching
Rules (from the spec):
* `thinking_text` is split on the `⟦S<n>⟧` markers; the text after a marker belongs to that segment; original text
  is kept untouched (`text_raw`), cleaned text (`text_clean`) is used only for word counts and text analyses.
* Annotations are matched to segments **by `segment_id`** (the integer inside the id), never by position, and the
  **text order is preserved**. Nothing is sorted by id and no label is ever invented.
* A trace counts as *labeled* only if the marker ids in the text and the annotation ids are the same set, there
  are no duplicate ids on either side, the JSON is valid, and **every segment has a valid `discourse_role`**.
  Invalid `answer_status` / `temporal_role` values are set to missing (not guessed) and reported.
* Length of a trace = sum of its segment word counts (text before the first marker is reported separately).

In [ ]:
def parse_seg_num(seg_id):
    m = re.search(r"(\d+)", str(seg_id))
    return int(m.group(1)) if m else None


seg_rows, trace_flags = [], []
for tid, text, lj, n_rtl in zip(df["trace_id"], df["thinking_text"], df["rtl_labels_json"], df["rtl_n_segments_num"]):
    fl = dict(trace_id=tid, no_text=text is None or text != text, json_missing=lj is None or lj != lj,
              json_malformed=False, n_text_seg=0, n_json=0, dup_ids_text=False, dup_ids_json=False,
              ids_mismatch=False, count_mismatch=False, rtl_n_vs_text=False, rtl_n_vs_json=False,
              order_differs=False, bad_role=0, bad_status=0, bad_temporal=0, missing_keys=0, extra_keys=0,
              nonstandard_id=0, empty_segments=0, preamble_words=0)
    ann_by_num, ann_nums = {}, []
    if not fl["json_missing"]:
        try:
            ann = json.loads(lj)
            if not isinstance(ann, list) or not all(isinstance(a, dict) for a in ann):
                fl["json_malformed"] = True
                ann = []
        except json.JSONDecodeError:
            fl["json_malformed"] = True
            ann = []
        for a in ann:
            n = parse_seg_num(a.get("segment_id"))
            if n is None:
                fl["nonstandard_id"] += 1
                continue
            if not re.fullmatch("⟦S\\d+⟧", str(a.get("segment_id"))):
                fl["nonstandard_id"] += 1
            if n in ann_by_num:
                fl["dup_ids_json"] = True
            else:
                ann_by_num[n] = a
            ann_nums.append(n)
            keys = set(a)
            fl["missing_keys"] += len({"segment_id", "discourse_role", "answer_status", "answer_value",
                                       "temporal_role"} - keys)
            fl["extra_keys"] += len(keys - {"segment_id", "discourse_role", "answer_status", "answer_value",
                                            "temporal_role"})
        fl["n_json"] = len(ann)
    if not fl["no_text"]:
        ms = list(MARK_RE.finditer(text))
        fl["n_text_seg"] = len(ms)
        text_nums = [int(m.group(1)) for m in ms]
        fl["dup_ids_text"] = len(set(text_nums)) != len(text_nums)
        if ms:
            fl["preamble_words"] = word_count(text[:ms[0].start()])
        for i, m in enumerate(ms):
            end = ms[i + 1].start() if i + 1 < len(ms) else len(text)
            raw_seg = text[m.end():end]
            a = ann_by_num.get(int(m.group(1)))
            role = status = value = temporal = None
            if a is not None:
                role, status, temporal = a.get("discourse_role"), a.get("answer_status"), a.get("temporal_role")
                value = a.get("answer_value")
                if role not in ROLES:
                    fl["bad_role"] += 1
                if status not in STATUSES:
                    fl["bad_status"] += 1
                    status = None
                if temporal not in TEMPORAL:
                    fl["bad_temporal"] += 1
                    temporal = None
                if value is not None and not isinstance(value, str):
                    value = str(value)
            clean = clean_text(raw_seg)
            w = len(clean.split())
            fl["empty_segments"] += int(w == 0)
            seg_rows.append((tid, i, int(m.group(1)), m.group(0), raw_seg, clean, w, role, status, value, temporal,
                             a is not None))
        fl["ids_mismatch"] = set(text_nums) != set(ann_by_num)
        fl["order_differs"] = (not fl["ids_mismatch"]) and text_nums != ann_nums
        fl["count_mismatch"] = fl["n_text_seg"] != fl["n_json"]
        if n_rtl == n_rtl:
            fl["rtl_n_vs_text"] = int(n_rtl) != fl["n_text_seg"]
            fl["rtl_n_vs_json"] = int(n_rtl) != fl["n_json"]
    trace_flags.append(fl)

SEG_COLS = ["trace_id", "seg_pos", "seg_num", "seg_marker", "text_raw", "text_clean", "words", "role",
            "answer_status", "answer_value", "temporal_role", "matched"]
seg_all = pd.DataFrame(seg_rows, columns=SEG_COLS)
flags = pd.DataFrame(trace_flags).set_index("trace_id")
flags["labels_ok"] = (~flags["no_text"] & ~flags["json_missing"] & ~flags["json_malformed"] & (flags["n_text_seg"] > 0)
                      & ~flags["ids_mismatch"] & ~flags["dup_ids_text"] & ~flags["dup_ids_json"]
                      & (flags["bad_role"] == 0))
# cumulative word positions in TEXT ORDER (never sorted by id)
seg_all["end_word"] = seg_all.groupby("trace_id")["words"].cumsum()
seg_all["start_word"] = seg_all["end_word"] - seg_all["words"]
trace_words = seg_all.groupby("trace_id")["words"].sum()
flags["words_total"] = trace_words.reindex(flags.index).fillna(0).astype(int)

# Independent check: word count of the whole text with markers replaced by spaces must equal the sum of segment words
# (text before the first marker excluded).
chk = df.set_index("trace_id")["thinking_text"].dropna()
first_mark = chk.map(lambda s: (MARK_RE.search(s).start() if MARK_RE.search(s) else len(s)))
indep = pd.Series({t: word_count(MARK_RE.sub(" ", s[first_mark[t]:])) for t, s in chk.items()})
assert (indep.reindex(flags.index).fillna(0).astype(int) == flags["words_total"]).all(), \
    "segment widths do not sum to the independently counted trace length"
# order preserved: within each trace seg_pos is 0..n-1 and start_word is non-decreasing
assert (seg_all.groupby("trace_id")["seg_pos"].apply(lambda s: (s.to_numpy() == np.arange(len(s))).all())).all()
assert (seg_all.groupby("trace_id")["start_word"].apply(lambda s: (np.diff(s.to_numpy()) >= 0).all())).all()
print("assertions passed: segment widths sum to trace length; text order preserved (%d traces, %d segments)"
      % (flags.shape[0], len(seg_all)))

### Annotation / reconstruction quality report

In [ ]:
n = len(flags)
rep = [
    ("rows", n),
    ("JSON missing/empty", flags["json_missing"].sum()),
    ("JSON malformed (parse error or not a list of objects)", flags["json_malformed"].sum()),
    ("segment count in text != annotation count", flags["count_mismatch"].sum()),
    ("segment-id set in text != annotation id set", flags["ids_mismatch"].sum()),
    ("duplicate segment ids in text", flags["dup_ids_text"].sum()),
    ("duplicate segment ids in JSON", flags["dup_ids_json"].sum()),
    ("annotation order differs from text order (matched by id anyway)", flags["order_differs"].sum()),
    ("rtl_n_segments != segments in text", flags["rtl_n_vs_text"].sum()),
    ("rtl_n_segments != annotations in JSON", flags["rtl_n_vs_json"].sum()),
    ("traces with unexpected discourse_role value", (flags["bad_role"] > 0).sum()),
    ("traces with unexpected answer_status value", (flags["bad_status"] > 0).sum()),
    ("traces with unexpected temporal_role value", (flags["bad_temporal"] > 0).sum()),
    ("traces with missing annotation keys", (flags["missing_keys"] > 0).sum()),
    ("traces with extra annotation keys", (flags["extra_keys"] > 0).sum()),
    ("traces with non-standard segment_id format", (flags["nonstandard_id"] > 0).sum()),
    ("traces with >= 1 empty (zero-word) segment", (flags["empty_segments"] > 0).sum()),
    ("traces with words before the first marker (excluded from length)", (flags["preamble_words"] > 0).sum()),
    ("LABELED traces (all checks passed, all roles valid)", flags["labels_ok"].sum()),
]
recon_report = pd.DataFrame(rep, columns=["check", "n_traces"])
recon_report["share"] = recon_report["n_traces"] / n
display(recon_report)
save_table(recon_report, "reconstruction_quality.csv", index=False)
print("\nrtl_status x labeled:")
display(pd.crosstab(df.set_index("trace_id")["rtl_status"].fillna("NA"), flags["labels_ok"]))
print("segment-level label counts (labeled traces only):")
seg_lab = seg_all[seg_all["trace_id"].isin(flags.index[flags["labels_ok"]])]
display(pd.concat({c: seg_lab[c].value_counts(dropna=False) for c in ["role", "answer_status", "temporal_role"]}))

### Eligibility and exclusion funnel
A sample is **eligible** if it is labeled, complete (not truncated), has known correctness, and its key is not
duplicated. Zero-word traces are also dropped (no width to draw). The funnel is applied in the order below and every
exclusion is counted. Truncated traces are kept in the tables so that they can be analysed separately
(section 11) and in the sensitivity checks.

In [ ]:
tr = df.set_index("trace_id").join(flags)
steps = [
    ("not a duplicate key", ~tr["dup_key"]),
    ("labeled (valid JSON, ids match, all roles valid)", tr["labels_ok"]),
    ("has words", tr["words_total"] > 0),
    ("correctness known", tr["correct"].notna()),
    ("truncation status known", tr["trunc"].notna()),
    ("complete (not truncated)", tr["trunc"] == 0),
]
alive = pd.Series(True, index=tr.index)
funnel = [dict(step="all rows", failing_this_criterion=np.nan, removed=0, remaining=len(tr))]
for name, ok in steps:
    removed = int((alive & ~ok).sum())
    funnel.append(dict(step=name, failing_this_criterion=int((~ok).sum()), removed=removed,
                       remaining=int((alive & ok).sum())))
    tr.loc[alive & ~ok, "excl_reason"] = name
    alive &= ok
funnel = pd.DataFrame(funnel)
display(funnel)
save_table(funnel, "exclusion_funnel.csv", index=False)
tr["eligible"] = alive
# "base_ok" = everything except the truncation requirement (used for truncated-trace analyses / sensitivity)
tr["base_ok"] = (~tr["dup_key"]) & tr["labels_ok"] & (tr["words_total"] > 0) & tr["correct"].notna()
print("eligible traces: %d | base_ok (incl. truncated): %d | truncated among base_ok: %d"
      % (tr["eligible"].sum(), tr["base_ok"].sum(), (tr["base_ok"] & (tr["trunc"] == 1)).sum()))

## 5. Feature preparation
Features are computed for **all labeled, non-duplicate traces with known correctness** (including truncated ones, so
that sensitivity checks can switch them on); the `eligible` flag decides which traces enter the main analyses.

Positions of answer events are measured at the **end of the segment that first carries them** ("the answer has been
stated by here"). Text-based features are *cues and similarity scores*, not labels.

In [ ]:
F = tr.loc[tr["base_ok"], ["benchmark", "question_id", "qkey", "cell_id", "compute_level", "sample_index",
                           "difficulty_tier", "task_type", "correct", "trunc", "eligible", "empty_out", "gold",
                           "raw_model_output", "thinking_tokens_num", "words_total"]].sort_index().copy()
F = F.rename(columns={"words_total": "words"})
S = seg_all[seg_all["trace_id"].isin(F.index)].sort_values(["trace_id", "seg_pos"]).reset_index(drop=True)
assert np.array_equal(S["trace_id"].unique(), F.index.to_numpy()), "every feature trace needs >= 1 segment"
F["n_seg"] = S.groupby("trace_id").size()
F["log_words"] = np.log(F["words"])
F["mean_seg_words"] = F["words"] / F["n_seg"]
F["med_seg_words"] = S.groupby("trace_id")["words"].median()
S["twords"] = S["trace_id"].map(F["words"])
S["rel_start"], S["rel_end"] = S["start_word"] / S["twords"], S["end_word"] / S["twords"]
S["rel_mid"] = (S["rel_start"] + S["rel_end"]) / 2
S["role_code"] = S["role"].map({r: i for i, r in enumerate(ROLES)})
offs = np.r_[0, np.cumsum(S.groupby("trace_id", sort=True).size().to_numpy())]   # contiguous slice per trace
tid_arr = F.index.to_numpy()
print("feature table: %d traces (%d eligible), %d segments" % (len(F), F["eligible"].sum(), len(S)))

# ---- role composition (word shares / segment shares) ------------------------------------------------------------
role_words = S.groupby(["trace_id", "role"])["words"].sum().unstack(fill_value=0).reindex(columns=ROLES, fill_value=0)
role_segs = S.groupby(["trace_id", "role"]).size().unstack(fill_value=0).reindex(columns=ROLES, fill_value=0)
for r in ROLES:
    F["words_" + r] = role_words[r].reindex(F.index).fillna(0)
    F["share_" + r] = F["words_" + r] / F["words"]
    F["segshare_" + r] = role_segs[r].reindex(F.index).fillna(0) / F["n_seg"]

# ---- merged phase runs ------------------------------------------------------------------------------------------
S["run_break"] = S["role"].ne(S.groupby("trace_id")["role"].shift())
S["run_idx"] = S.groupby("trace_id")["run_break"].cumsum() - 1
R = S.groupby(["trace_id", "run_idx"], sort=True).agg(
    role=("role", "first"), n_segs=("seg_pos", "size"), words=("words", "sum"), start_word=("start_word", "min"),
    end_word=("end_word", "max"), first_seg=("seg_pos", "min"), last_seg=("seg_pos", "max")).reset_index()
R["twords"] = R["trace_id"].map(F["words"])
R["rel_start"], R["rel_end"] = R["start_word"] / R["twords"], R["end_word"] / R["twords"]
R["prev_role"] = R.groupby("trace_id")["role"].shift(1)
R["next_role"] = R.groupby("trace_id")["role"].shift(-1)
F["n_runs"] = R.groupby("trace_id").size()
F["role_changes"] = F["n_runs"] - 1
F["changes_per100w"] = F["role_changes"] / F["words"] * 100
F["segs_per_run"] = F["n_seg"] / F["n_runs"]
F["mean_run_words"] = F["words"] / F["n_runs"]
for r in ROLES:
    cnt = R[R["role"] == r].groupby("trace_id").size()
    F["runs_" + r] = cnt.reindex(F.index).fillna(0)
for r in KEY_ROLES:
    F["runs100_" + r] = F["runs_" + r] / F["words"] * 100
    first = R[R["role"] == r].groupby("trace_id").agg(w=("start_word", "min"), rel=("rel_start", "min"))
    F["first_%s_words" % r] = first["w"].reindex(F.index)
    F["first_%s_rel" % r] = first["rel"].reindex(F.index)
    F["has_" + r] = F["first_%s_words" % r].notna().astype(float)
for a, b, nm in [("doubt", "verification", "doubt_to_verif"), ("doubt", "correction", "doubt_to_corr"),
                 ("verification", "doubt", "verif_to_doubt")]:
    cnt = R[(R["role"] == a) & (R["next_role"] == b)].groupby("trace_id").size()
    F[nm + "100"] = cnt.reindex(F.index).fillna(0) / F["words"] * 100
last_seg = S.groupby("trace_id").tail(1).set_index("trace_id")
F["last_role"], F["last_seg_words"] = last_seg["role"], last_seg["words"]

# ---- lexical cues (counts of surface phrases per segment; cues, not labels) -------------------------------------
CUES = {
    "wait": r"\bwait\b", "hmm": r"\bhmm+\b", "maybe": r"\bmaybe\b", "actually": r"\bactually\b",
    "let_me_check": r"\blet me (?:check|double[- ]?check|verify|re-?check)\b|\bdouble[- ]?check",
    "alternatively": r"\balternatively\b|\bor perhaps\b|\bon the other hand\b",
    "restart": r"\bstart over\b|\bstart again\b|\btry again\b|\bfrom scratch\b|\blet me redo\b|\blet'?s redo\b|\bre-?do\b",
}
HEDGES = ["wait", "hmm", "maybe", "actually", "let_me_check", "alternatively"]
low = S["text_clean"].str.lower()
for k, p in CUES.items():
    S["cue_" + k] = low.str.count(p)
S["cue_hedge"] = S[["cue_" + k for k in HEDGES]].sum(axis=1)
cue_sum = S.groupby("trace_id")[["cue_" + k for k in CUES] + ["cue_hedge"]].sum()
for k in CUES:
    F["cue_%s_100w" % k] = cue_sum["cue_" + k].reindex(F.index) / F["words"] * 100
F["hedge_100w"] = cue_sum["cue_hedge"].reindex(F.index) / F["words"] * 100     # wait+hmm+maybe+actually+let me check+alternatively

# ---- TF-IDF similarity between segments (global vocabulary, per-trace comparison) -------------------------------
tfidf = TfidfVectorizer(lowercase=True, stop_words="english", token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_]+\b",
                        min_df=2, max_features=60000, sublinear_tf=True, dtype=np.float32)
X = tfidf.fit_transform(S["text_clean"])          # rows aligned with S (row i = S.iloc[i]); L2-normalised
print("TF-IDF matrix:", X.shape, "| vocabulary:", len(tfidf.vocabulary_))
sim_prev = np.full(len(S), np.nan)     # cosine with the immediately preceding segment
sim_np = np.full(len(S), np.nan)       # max cosine with any NON-adjacent earlier segment (i <= j-2)
for ti in range(len(tid_arr)):
    a, b = offs[ti], offs[ti + 1]
    m = b - a
    if m < 2:
        continue
    Xt = X[a:b]
    G = (Xt @ Xt.T).toarray().astype(float)
    ok = np.asarray(Xt.getnnz(axis=1)) > 0
    G[~ok, :] = np.nan
    G[:, ~ok] = np.nan
    j = np.arange(1, m)
    sim_prev[a + 1:b] = G[j, j - 1]
    if m >= 3:
        P = np.where(np.tril(np.ones((m, m), bool), -2), G, np.nan)
        Pm = np.where(np.isnan(P), -np.inf, P).max(axis=1)
        Pm[np.isinf(Pm)] = np.nan
        sim_np[a:b] = Pm
S["sim_prev"], S["sim_nonadj_max"] = sim_prev, sim_np

# ---- novelty, repeated n-grams / sentences / equations / numbers, incremental compression -----------------------
TOK_RE = re.compile(r"[a-z0-9]+(?:[.'\-][a-z0-9]+)*")
NUM_RE = re.compile(r"\d+(?:\.\d+)?")
SENT_RE = re.compile(r"(?<=[.!?])\s+|\n+")
cols = ["n_content", "n_new", "n4", "n4_rep", "n_num", "n_num_rep", "n_sent", "n_sent_rep", "n_eq", "n_eq_rep"]
tx = {c: np.zeros(len(S)) for c in cols}
inc_comp = np.full(len(S), np.nan)
texts = S["text_clean"].to_numpy()
comp_ratio, max_sent_rep, top_sent = [], [], []
for ti in range(len(tid_arr)):
    a, b = offs[ti], offs[ti + 1]
    seen_c, seen4, seen_n, seen_eq = set(), set(), set(), set()
    seen_s = Counter()
    co = zlib.compressobj(6)
    for k in range(a, b):
        t = texts[k]
        lw = t.lower()
        toks = TOK_RE.findall(lw)
        content = {w for w in toks if w.isalpha() and len(w) >= 3 and w not in ENGLISH_STOP_WORDS}
        tx["n_content"][k] = len(content)
        tx["n_new"][k] = len(content - seen_c)
        seen_c |= content
        for i in range(len(toks) - 3):
            g = " ".join(toks[i:i + 4])
            tx["n4"][k] += 1
            if g in seen4:
                tx["n4_rep"][k] += 1
            else:
                seen4.add(g)
        for x in NUM_RE.findall(lw):
            if len(x) >= 2:
                tx["n_num"][k] += 1
                if x in seen_n:
                    tx["n_num_rep"][k] += 1
                else:
                    seen_n.add(x)
        for sent in SENT_RE.split(t):
            st = TOK_RE.findall(sent.lower())
            if len(st) < 4:
                continue
            key = " ".join(st)
            tx["n_sent"][k] += 1
            if seen_s[key] > 0:
                tx["n_sent_rep"][k] += 1
            seen_s[key] += 1
            if "=" in sent:
                eq = re.sub(r"\s+", "", sent.lower())
                tx["n_eq"][k] += 1
                if eq in seen_eq:
                    tx["n_eq_rep"][k] += 1
                else:
                    seen_eq.add(eq)
        bts = t.encode("utf-8")
        out = len(co.compress(bts)) + len(co.flush(zlib.Z_SYNC_FLUSH))
        if len(bts) >= 80:
            inc_comp[k] = out / len(bts)
    full = " ".join(texts[a:b]).encode("utf-8")
    comp_ratio.append(len(zlib.compress(full, 6)) / max(len(full), 1))
    max_sent_rep.append(max(seen_s.values()) if seen_s else 0)
    top_sent.append(seen_s.most_common(1)[0][0] if seen_s else "")
for c in cols:
    S[c] = tx[c]
S["novelty"] = np.where(S["n_content"] >= 3, S["n_new"] / S["n_content"].replace(0, np.nan), np.nan)
S["rep4_share"] = S["n4_rep"] / S["n4"].replace(0, np.nan)
S["inc_comp"] = inc_comp
S["near_repeat"] = (S["sim_nonadj_max"] >= REP_SIM).astype(float)
tsum = S.groupby("trace_id")[cols].sum()
F["novelty"] = tsum["n_new"].reindex(F.index) / tsum["n_content"].reindex(F.index).replace(0, np.nan)
F["rep4_share"] = tsum["n4_rep"].reindex(F.index) / tsum["n4"].reindex(F.index).replace(0, np.nan)
F["num_repeat_share"] = tsum["n_num_rep"].reindex(F.index) / tsum["n_num"].reindex(F.index).replace(0, np.nan)
F["sent_repeat_share"] = tsum["n_sent_rep"].reindex(F.index) / tsum["n_sent"].reindex(F.index).replace(0, np.nan)
F["eq_repeat_share"] = tsum["n_eq_rep"].reindex(F.index) / tsum["n_eq"].reindex(F.index).replace(0, np.nan)
F["comp_ratio"] = pd.Series(comp_ratio, index=F.index)
F["max_sent_repeat"] = pd.Series(max_sent_rep, index=F.index)
F["top_sentence"] = pd.Series(top_sent, index=F.index)
long_seg = S[S["words"] >= 8]
F["rep_sim_mean"] = long_seg.groupby("trace_id")["sim_nonadj_max"].mean().reindex(F.index)
F["near_repeat_wshare"] = (S["words"] * S["near_repeat"]).groupby(S["trace_id"]).sum().reindex(F.index) / F["words"]
LOOP_SENT, LOOP_REP4, LOOP_MAXREP = 0.20, 0.50, 3     # descriptive loop flag (not a label)
F["loop_flag"] = ((F["sent_repeat_share"] >= LOOP_SENT) | (F["rep4_share"] >= LOOP_REP4)
                  | (F["max_sent_repeat"] >= LOOP_MAXREP)).astype(float)

# ---- answer development -----------------------------------------------------------------------------------------
S["ans_norm"] = S["answer_value"].map(norm_answer)
F["gold_norm"] = F["gold"].map(norm_answer)
ev = S[S["answer_status"].isin(["candidate", "final"]) | S["ans_norm"].notna()]
ans_rows = []
for tid, g in ev.groupby("trace_id", sort=False):
    tw, n_seg_t = F.at[tid, "words"], F.at[tid, "n_seg"]
    pos, status, val = g["seg_pos"].to_numpy(), g["answer_status"].to_numpy(), g["ans_norm"].to_numpy()
    endw = dict(zip(pos, g["end_word"].to_numpy()))
    is_c, is_f = status == "candidate", status == "final"
    first_ans_pos = pos[is_c | is_f][0] if (is_c | is_f).any() else None
    first_c_pos = pos[is_c][0] if is_c.any() else None
    first_f_pos = pos[is_f][0] if is_f.any() else None
    vals = [(p, v) for p, v in zip(pos, val) if v is not None]
    repl_pos, refine_pos, prev = [], [], None
    for p, v in vals:
        if prev is not None and v != prev:
            (refine_pos if is_refinement(prev, v) else repl_pos).append(p)
        prev = v
    last_val = vals[-1][1] if vals else None
    stab_pos = None
    if vals:
        i = len(vals) - 1
        while i > 0 and vals[i - 1][1] == last_val:
            i -= 1
        stab_pos = vals[i][0]
    gold_n = F.at[tid, "gold_norm"]
    gold_pos = [p for p, v in vals if v == gold_n and gold_n is not None]
    distinct = len({v for _, v in vals})
    # does any value after the first `final` segment differ from the value stated at that segment?
    fval = next((v for p, v in vals if p == first_f_pos), None) if first_f_pos is not None else None
    post_final_change = float(fval is not None and any(p > first_f_pos and v != fval for p, v in vals))
    row = dict(trace_id=tid, first_ans_seg=first_ans_pos, first_cand_seg=first_c_pos, first_final_seg=first_f_pos,
               first_ans_end=endw.get(first_ans_pos), first_cand_end=endw.get(first_c_pos),
               first_final_end=endw.get(first_f_pos), n_value_events=len(vals), n_distinct_values=distinct,
               n_replace=len(repl_pos), n_refine=len(refine_pos), replace_segs=repl_pos, refine_segs=refine_pos,
               last_val=last_val, stab_seg=stab_pos, gold_ever=float(bool(gold_pos)),
               gold_first_seg=gold_pos[0] if gold_pos else None, cand_repeat_n=len(vals) - distinct,
               n_final_segs=int(is_f.sum()),
               final_no_value=int(((status == "final") & np.array([v is None for v in val])).sum()),
               cand_no_value=int(((status == "candidate") & np.array([v is None for v in val])).sum()),
               value_status_none=int((~(is_c | is_f) & np.array([v is not None for v in val])).sum()),
               final_before_cand=float(first_f_pos is not None and first_c_pos is not None and first_f_pos < first_c_pos),
               post_final_change=post_final_change)
    if stab_pos is not None:
        row["stab_end"] = endw[stab_pos]
        row["words_after_stab"] = tw - endw[stab_pos]
        row["segs_after_stab"] = n_seg_t - 1 - stab_pos
    row["last_matches_gold"] = float(last_val is not None and gold_n is not None and last_val == gold_n)
    row["gold_lost"] = float(bool(gold_pos) and not row["last_matches_gold"])
    ans_rows.append(row)
A = pd.DataFrame(ans_rows).set_index("trace_id")
F = F.join(A.drop(columns=["replace_segs", "refine_segs"]), how="left")
F["has_answer"] = F["first_ans_seg"].notna().astype(float)
F["first_ans_rel"] = F["first_ans_end"] / F["words"]
F["first_cand_rel"] = F["first_cand_end"] / F["words"]
F["first_final_rel"] = F["first_final_end"] / F["words"]
F["stab_rel"] = F["stab_end"] / F["words"]
F["words_before_first_ans"] = F["first_ans_end"]
F["words_after_first_ans"] = F["words"] - F["first_ans_end"]
F["share_after_first_ans"] = F["words_after_first_ans"] / F["words"]
F["share_after_stab"] = F["words_after_stab"] / F["words"]
F["words_after_first_final"] = F["words"] - F["first_final_end"]
F["share_after_first_final"] = F["words_after_first_final"] / F["words"]
F["n_replace_100w"] = F["n_replace"].fillna(0) / F["words"] * 100
F["any_replace"] = (F["n_replace"].fillna(0) > 0).astype(float)
no_val = F["last_val"].isna()          # no annotated answer value -> gold comparisons are undefined, not "0"
for c in ["gold_ever", "gold_lost", "last_matches_gold"]:
    F.loc[no_val, c] = np.nan
A_lists = A[["replace_segs", "refine_segs"]]   # positions of answer-change events (used by the figures)

# ---- thinking vs output: heuristic answer extraction from raw_model_output --------------------------------------
def last_boxed(s):
    i = s.rfind("\\boxed{")
    if i < 0:
        return None
    j, depth, start = i + 7, 1, i + 7
    while j < len(s) and depth:
        depth += (s[j] == "{") - (s[j] == "}")
        j += 1
    return s[start:j - 1] if depth == 0 else None


ANS_RE = re.compile(r"(?:final answer|the answer|answer)\s*(?:is|:)\s*[:\-]?\s*\**\s*([^\n]+)", re.I)


def extract_output_answer(raw_out):
    if raw_out is None or raw_out != raw_out:
        return None, "none"
    s = raw_out.split("</think>")[-1] if "</think>" in raw_out else raw_out
    tail = s[-600:]
    b = last_boxed(tail)
    if b:
        return b, "boxed"
    m = list(ANS_RE.finditer(tail))
    if m:
        return re.sub(r"[*$]", "", m[-1].group(1)).strip(), "answer_phrase"
    return None, "none"


# ---- post-answer composition: words in segments strictly AFTER the first answer / first final, by role ----------
for tag, col in [("postans", "first_ans_seg"), ("postfinal", "first_final_seg")]:
    after = S["seg_pos"] > S["trace_id"].map(F[col])          # NaN -> False (trace has no such event)
    pw = S[after].groupby(["trace_id", "role"])["words"].sum().unstack(fill_value=0).reindex(columns=ROLES, fill_value=0)
    for r in ROLES:
        F["%s_words_%s" % (tag, r)] = pw[r].reindex(F.index).fillna(0)

# ---- answer pattern per trace (descriptive categories) ---------------------------------------------------------
F["ans_pattern"] = np.select(
    [F["last_val"].isna(), F["gold_lost"] == 1, F["n_replace"].fillna(0) > 0],
    ["no answer value", "gold candidate later replaced", "answer replaced"], default="stable")
for k, lab in [("stable", "stable"), ("replaced", "answer replaced"), ("goldlost", "gold candidate later replaced"),
               ("noanswer", "no answer value")]:
    F["pat_" + k] = (F["ans_pattern"] == lab).astype(float)

ex = F["raw_model_output"].map(extract_output_answer)
F["out_ans"] = ex.map(lambda t: norm_answer(t[0]))
F["out_method"] = ex.map(lambda t: t[1])
F["out_matches_gold"] = np.where(F["out_ans"].notna() & F["gold_norm"].notna(), (F["out_ans"] == F["gold_norm"]).astype(float), np.nan)
F["out_matches_last_cand"] = np.where(F["out_ans"].notna() & F["last_val"].notna(), (F["out_ans"] == F["last_val"]).astype(float), np.nan)
F = F.drop(columns=["raw_model_output"]).copy()      # .copy() defragments the wide frame
save_table(F.drop(columns=["top_sentence"]), "trace_features.csv")
print("trace-level features:", F.shape)

### Annotation coverage / consistency report (answer fields)
How reliable are the answer annotations? `final` is an annotation status, **not** correctness. The agreement
between `is_correct` and "last annotated answer equals gold" is the key sanity check.

In [ ]:
E = F[F["eligible"]]
ar = [
    ("eligible traces", len(E)),
    ("with >= 1 answer candidate/final annotation", int(E["has_answer"].sum())),
    ("with a `final` segment", int(E["first_final_seg"].notna().sum())),
    ("with several `final` segments", int((E["n_final_segs"] > 1).sum())),
    ("`final` appears before any `candidate`", int((E["final_before_cand"] == 1).sum())),
    ("segments with status final/candidate but answer_value null",
     int(E["final_no_value"].fillna(0).sum() + E["cand_no_value"].fillna(0).sum())),
    ("segments with an answer_value but status none", int(E["value_status_none"].fillna(0).sum())),
    ("first segment annotated `continuation`", int(S[(S["seg_pos"] == 0) & (S["temporal_role"] == "continuation")].shape[0])),
    ("traces with >= 1 observed answer replacement", int((E["n_replace"] > 0).sum())),
    ("traces with >= 1 observed refinement/format change", int((E["n_refine"] > 0).sum())),
]
cov = pd.DataFrame(ar, columns=["check", "n"])
cov["share_of_eligible"] = cov["n"] / len(E)
display(cov)
save_table(cov, "annotation_coverage.csv", index=False)
chk = E[E["last_val"].notna()]
agree = pd.crosstab(chk["correct"].map({1.0: "is_correct=1", 0.0: "is_correct=0"}),
                    chk["last_matches_gold"].map({1.0: "last answer == gold", 0.0: "last answer != gold"}))
print("is_correct vs (last annotated answer == gold), traces with an annotated value:")
display(agree)
print("agreement rate: %.3f" % ((chk["correct"] == chk["last_matches_gold"]).mean()))
print("\nP(correct | has `final`) vs P(correct | no `final`) -- `final` status is not correctness:")
display(E.groupby(E["first_final_seg"].notna())["correct"].agg(["mean", "size"]))
print("\noutput-answer extraction (heuristic) by method:")
display(E["out_method"].value_counts().to_frame("n"))
ok_out = E[E["out_ans"].notna()]
print("extracted output answer agrees with is_correct: %.3f (n=%d)" % (
    (ok_out["out_matches_gold"] == ok_out["correct"]).mean(), len(ok_out)))
print("words vs thinking_tokens Spearman rho (supplementary): %.3f" %
      stats.spearmanr(E["words"], E["thinking_tokens_num"], nan_policy="omit")[0])

## 6. Cell definition and group assignment
A **cell** = `benchmark::question_id` x `compute_level` (5 samples). Only *eligible* samples count. A cell needs
`MIN_ELIGIBLE` (4) eligible samples, otherwise it is `TOO-FEW`.

| Group | Rule (per cell) |
|---|---|
| **ALL-CORRECT** | all eligible samples correct **and** real length spread: `max-min >= 1.5*SD` and `max/min >= 1.5` |
| ALL-CORRECT-NARROW | all correct but not enough spread (kept as a reference set, not used for long-vs-short) |
| **MIXED** | >= 1 correct and >= 1 incorrect eligible sample |
| **ALL-WRONG** | all eligible samples incorrect |

The group rule is a function, so the sensitivity section can re-run it with other thresholds.

In [ ]:
def build_cells(Fe, min_elig=MIN_ELIGIBLE, spread_sd=SPREAD_SD, min_ratio=SPREAD_RATIO):
    """One row per cell with counts, length statistics and the group label (Fe = eligible traces)."""
    cells = Fe.groupby("cell_id", sort=False).agg(
        qkey=("qkey", "first"), benchmark=("benchmark", "first"), question_id=("question_id", "first"),
        compute_level=("compute_level", "first"), n_elig=("words", "size"), n_correct=("correct", "sum"),
        w_min=("words", "min"), w_max=("words", "max"), w_mean=("words", "mean"), w_sd=("words", "std"))
    cells["n_wrong"] = cells["n_elig"] - cells["n_correct"]
    cells["w_range"] = cells["w_max"] - cells["w_min"]
    cells["w_ratio"] = cells["w_max"] / cells["w_min"]
    cells["spread_ok"] = (cells["w_range"] >= spread_sd * cells["w_sd"]) & (cells["w_ratio"] >= min_ratio)
    few = cells["n_elig"] < min_elig
    cells["group"] = np.select(
        [few, (cells["n_wrong"] == 0) & cells["spread_ok"], cells["n_wrong"] == 0, cells["n_correct"] == 0],
        ["TOO-FEW", "ALL-CORRECT", "ALL-CORRECT-NARROW", "ALL-WRONG"], default="MIXED")
    return cells


def make_pairs(Fe, cell_ids):
    """Shortest and longest eligible trace per cell (ties broken by sample_index)."""
    sub = Fe[Fe["cell_id"].isin(cell_ids)].reset_index()
    asc = sub.sort_values(["cell_id", "words", "sample_index"]).groupby("cell_id").head(1).set_index("cell_id")
    desc = (sub.sort_values(["cell_id", "words", "sample_index"], ascending=[True, False, False])
            .groupby("cell_id").head(1).set_index("cell_id"))
    return pd.DataFrame({"short_id": asc["trace_id"], "long_id": desc["trace_id"].reindex(asc.index)})


E = F[F["eligible"]]
cells = build_cells(E)
qinfo = df.groupby("qkey").agg(difficulty_tier=("difficulty_tier", lambda s: s.mode().iloc[0] if s.notna().any() else None),
                               task_type=("task_type", lambda s: s.mode().iloc[0] if s.notna().any() else None))
cells["difficulty_tier"] = cells["qkey"].map(qinfo["difficulty_tier"])
cells["task_type"] = cells["qkey"].map(qinfo["task_type"])
F["cell_group"] = F["cell_id"].map(cells["group"])
E = F[F["eligible"]]
GROUPS = ["ALL-CORRECT", "ALL-CORRECT-NARROW", "MIXED", "ALL-WRONG", "TOO-FEW"]
save_table(cells, "cells.csv")

n_cells_all = df["cell_id"].nunique()
print("cells in raw data: %d | cells with >= 1 eligible sample: %d | cells with >= %d eligible samples: %d"
      % (n_cells_all, len(cells), MIN_ELIGIBLE, (cells["group"] != "TOO-FEW").sum()))
print("eligible samples per cell:", cells["n_elig"].value_counts().sort_index().to_dict())
overall = cells.groupby("group").agg(cells=("qkey", "size"), questions=("qkey", "nunique"), traces=("n_elig", "sum"))
display(overall.reindex(GROUPS).fillna(0).astype(int))

def composition(by):
    rows = [dict(by=k, group=g, cells=len(s), questions=s["qkey"].nunique(), traces=int(s["n_elig"].sum()))
            for (k, g), s in cells.groupby([by, "group"])]
    return pd.DataFrame(rows).pivot(index="by", columns="group", values=["cells", "questions", "traces"]).fillna(0).astype(int)

for by in ["benchmark", "difficulty_tier", "compute_level"]:
    comp = composition(by)
    print("\ncomposition by", by, "(cells / questions / traces per group)")
    display(comp)
    save_table(comp, "composition_by_%s.csv" % by)

# compute-level order: numeric if possible, else low<medium<high, else by mean length
cl = sorted(df["compute_level"].dropna().unique(), key=str)
num = pd.to_numeric(pd.Series(cl), errors="coerce")
if num.notna().all():
    COMPUTE_ORDER = [c for _, c in sorted(zip(num, cl))]
elif set(map(str.lower, cl)) <= {"minimal", "low", "medium", "high", "xhigh", "max"}:
    rank = {"minimal": 0, "low": 1, "medium": 2, "high": 3, "xhigh": 4, "max": 5}
    COMPUTE_ORDER = sorted(cl, key=lambda c: rank[c.lower()])
else:
    COMPUTE_ORDER = list(E.groupby("compute_level")["words"].mean().sort_values().index)
print("compute-level order used:", COMPUTE_ORDER)

# paired structures
ac_ids = cells.index[cells["group"] == "ALL-CORRECT"]
ac_pairs = make_pairs(E, ac_ids)
print("ALL-CORRECT cells (long vs short pairs): %d" % len(ac_pairs))
group_of_trace = F["cell_group"]

### Representative cells
For the inline figures, each group shows the cells closest to the group median (standardised log mean length and
spread), one per question. Every eligible cell still gets its own saved figure and text table in Part A.

In [ ]:
def representative(group, k=N_INLINE):
    c = cells[cells["group"] == group].copy()
    if c.empty:
        return []
    feats = pd.DataFrame({"len": np.log(c["w_mean"]), "ratio": np.log(c["w_ratio"].clip(lower=1)),
                          "mix": c["n_correct"] / c["n_elig"]})
    z = (feats - feats.median()) / feats.std(ddof=0).replace(0, 1)
    c["dist"] = np.sqrt((z ** 2).sum(axis=1))
    c = c.sort_values(["dist"]).drop_duplicates("qkey")
    return list(c.index[:k])


REP_CELLS = {g: representative(g) for g in ["ALL-CORRECT", "MIXED", "ALL-WRONG"]}
print({g: v for g, v in REP_CELLS.items()})

## 7. Part A: cell-level analysis
### 7.1 Trajectory figures
* One row per eligible sample of the cell; **x = cumulative word position on one absolute scale shared by all rows**.
* Each segment is a block whose **width = its word count** (a 500-word trace is 5x as wide as a 100-word trace),
  drawn in the original text order, never sorted and never rescaled per trace. Zero-word segments get no width (thin tick).
* Colour = `discourse_role` (same in every figure). Rows are ordered by total words (**shortest at the top**).
* Markers above a row (they never change block widths) sit at the *end* of the segment that first carries the event:
  first answer candidate, first final, observed answer replacement / refinement. A black triangle below a row marks the
  *algorithmic divergence candidate* (section 7.2; needs manual reading, no causal meaning).
* A **normalised (0-100 %)** variant is saved separately for the representative cells and is titled as normalised.

In [ ]:
seg_ix = S.groupby("trace_id").indices                      # trace_id -> positions in S (contiguous, text order)
S_start, S_words, S_end = S["start_word"].to_numpy(), S["words"].to_numpy(), S["end_word"].to_numpy()
S_role = S["role"].to_numpy()
cell_members = E.groupby("cell_id").groups                   # cell_id -> eligible trace_ids
RUN_ROLES = R.groupby("trace_id")["role"].apply(list)
RUN_START = R.groupby("trace_id")["start_word"].apply(lambda s: s.to_numpy())
RUN_FIRSTSEG = R.groupby("trace_id")["first_seg"].apply(lambda s: s.to_numpy())
MARKERS = {"cand": dict(marker="v", color="#08306b", label="first answer candidate"),
           "final": dict(marker="D", color="#6a3d9a", label="first final"),
           "repl": dict(marker="X", color="#d62728", label="observed answer replacement"),
           "refine": dict(marker="P", color="#17becf", label="answer refinement/format change")}
DIV_MARK = dict(marker="^", color="black", label="divergence candidate (algorithmic)")
div_marks = {}      # trace_id -> word position of the divergence candidate (filled in 7.2)


def plot_cell(cid, normalized=False, path=None, show=False, dpi=90):
    ids = list(cell_members[cid])
    rows = E.loc[ids].sort_values(["words", "sample_index"])
    n = len(rows)
    fig, ax = plt.subplots(figsize=(12, 0.78 * n + 2.4))
    xmax = 100.0 if normalized else float(rows["words"].max())
    used_markers = set()
    for k, (tid, r) in enumerate(rows.iterrows()):
        y = n - 1 - k                                       # shortest row at the top
        ix = seg_ix[tid]
        st, wd, rl = S_start[ix], S_words[ix], S_role[ix]
        # assertions: widths sum to the trace length; text order preserved (starts non-decreasing, first at 0)
        assert wd.sum() == r["words"] and st[0] == 0 and np.all(np.diff(st) >= 0)
        sc = 100.0 / r["words"] if normalized else 1.0
        pos = wd > 0
        ax.broken_barh(list(zip(st[pos] * sc, wd[pos] * sc)), (y - 0.38, 0.76),
                       facecolors=[ROLE_COLORS[x] for x in rl[pos]], edgecolors="white", linewidth=0.3)
        for s0, x in zip(st[~pos], rl[~pos]):                # zero-word segment: thin tick, no width
            ax.vlines(s0 * sc, y - 0.38, y + 0.38, colors=ROLE_COLORS[x], linewidth=0.8)
        if tid in A.index:
            evs = []
            if pd.notna(r["first_cand_end"]):
                evs.append(("cand", r["first_cand_end"]))
            if pd.notna(r["first_final_end"]):
                evs.append(("final", r["first_final_end"]))
            evs += [("repl", S_end[ix[p]]) for p in A_lists.at[tid, "replace_segs"]]
            evs += [("refine", S_end[ix[p]]) for p in A_lists.at[tid, "refine_segs"]]
            for kind, xw in evs:
                m = MARKERS[kind]
                dy = 0.52 if kind in ("cand", "final") else 0.70      # stagger so events at the same x stay visible
                ax.scatter([xw * sc], [y + dy], marker=m["marker"], s=34, color=m["color"], clip_on=False, zorder=5)
                used_markers.add(kind)
        if tid in div_marks:
            ax.scatter([div_marks[tid] * sc], [y - 0.55], marker="^", s=40, color="black", clip_on=False, zorder=5)
            used_markers.add("div")
    ax.set_yticks(range(n))
    labels = [("s%s · %s · %dw · %d seg" % (r["sample_index"], "correct" if r["correct"] == 1 else "INCORRECT",
                                           r["words"], r["n_seg"])) for _, r in rows.iloc[::-1].iterrows()]
    ax.set_yticklabels(labels, fontsize=8.5)
    for lab, (_, r) in zip(ax.get_yticklabels(), rows.iloc[::-1].iterrows()):
        lab.set_color("#1b7837" if r["correct"] == 1 else "#b2182b")
    ax.set_xlim(0, xmax * 1.02)
    ax.set_ylim(-0.9, n - 0.1 + 0.7)
    ax.set_xlabel("position in the trace, % of its own length (each row rescaled)" if normalized else
                  "cumulative word position (one absolute scale shared by all rows)")
    grp = cells.at[cid, "group"]
    ax.set_title(("NORMALISED 0-100 % · " if normalized else "") + "%s · group %s · rows ordered by total words "
                 "(shortest at top)" % (cid, grp), fontsize=10)
    ax.grid(axis="y", visible=False)
    handles = [plt.Rectangle((0, 0), 1, 1, color=ROLE_COLORS[x]) for x in ROLES]
    names = list(ROLES)
    for kind in ["cand", "final", "repl", "refine"]:
        if kind in used_markers:
            handles.append(plt.Line2D([], [], marker=MARKERS[kind]["marker"], color=MARKERS[kind]["color"], ls=""))
            names.append(MARKERS[kind]["label"])
    if "div" in used_markers:
        handles.append(plt.Line2D([], [], marker="^", color="black", ls=""))
        names.append(DIV_MARK["label"])
    ax.legend(handles, names, ncol=4, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.16), frameon=False)
    if path:
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)


def cell_text_table(cid):
    """Segment-level inspection table for all eligible samples of a cell (original segment text untouched)."""
    parts = []
    for tid in E.loc[list(cell_members[cid])].sort_values(["words", "sample_index"]).index:
        ix = seg_ix[tid]
        t = S.iloc[ix][["seg_pos", "seg_marker", "role", "answer_status", "answer_value", "temporal_role", "words",
                        "start_word", "end_word", "rel_start", "text_raw"]].copy()
        t.insert(0, "is_correct", E.at[tid, "correct"])
        t.insert(0, "sample_index", E.at[tid, "sample_index"])
        t.insert(0, "cell_id", cid)
        t["marker_first_cand"] = t["seg_pos"] == E.at[tid, "first_cand_seg"]
        t["marker_first_final"] = t["seg_pos"] == E.at[tid, "first_final_seg"]
        t["marker_answer_replace"] = t["seg_pos"].isin(A_lists.at[tid, "replace_segs"]) if tid in A.index else False
        t["marker_div_candidate"] = False
        parts.append(t)
    return pd.concat(parts, ignore_index=True)

### 7.2 Divergence candidates (algorithmic; needs manual reading)
For a compared pair (MIXED: the *length-matched* wrong/correct pair; ALL-CORRECT: longest vs shortest) two
independent cues are computed:
1. **phase divergence** - the first merged-run index where the two role sequences differ (plus the overall
   role-sequence similarity from `difflib`);
2. **text divergence** - the first segment (with >= 5 words) whose best TF-IDF cosine to *any* segment of the other
   trace is below `DIV_SIM`, and so is the next informative one (two in a row, to avoid single odd segments).
Surrounding text is printed. These are **candidates for a human to read, not causes** of the outcome.

In [ ]:
def text_divergence(ta, tb):
    """First segment of trace ta without a counterpart in tb (two consecutive informative segments below DIV_SIM)."""
    ia, ib = seg_ix[ta], seg_ix[tb]
    Xa, Xb = X[ia[0]:ia[-1] + 1], X[ib[0]:ib[-1] + 1]
    best = np.asarray((Xa @ Xb.T).max(axis=1).todense()).ravel()
    informative = (np.asarray(Xa.getnnz(axis=1)) > 0) & (S_words[ia] >= 5)
    cand = np.where(informative)[0]
    flag = best[cand] < DIV_SIM
    for k in range(len(cand) - 1):
        if flag[k] and flag[k + 1]:
            return int(cand[k])
    return None


def context(tid, seg_pos, width=1, clip=260):
    if seg_pos is None:
        return ""
    ix = seg_ix[tid]
    out = []
    for p in range(max(0, seg_pos - width), min(len(ix), seg_pos + width + 1)):
        row = S.iloc[ix[p]]
        out.append("%s[%s%s] %s" % (">>" if p == seg_pos else "  ", row["role"], "|" + row["answer_status"] if
                                    row["answer_status"] in ("candidate", "final") else "", row["text_clean"][:clip]))
    return "\n".join(out)


def divergence_record(cid, ta, tb, kind, label_a, label_b):
    ra, rb = RUN_ROLES[ta], RUN_ROLES[tb]
    k = next((i for i, (x, y) in enumerate(zip(ra, rb)) if x != y), min(len(ra), len(rb)) if len(ra) != len(rb) else None)
    wa, wb = F.at[ta, "words"], F.at[tb, "words"]
    pa = (RUN_START[ta][k] if k is not None and k < len(ra) else wa) if k is not None else None
    pb = (RUN_START[tb][k] if k is not None and k < len(rb) else wb) if k is not None else None
    sa, sb = text_divergence(ta, tb), text_divergence(tb, ta)
    ta_w = S_start[seg_ix[ta][sa]] if sa is not None else None
    tb_w = S_start[seg_ix[tb][sb]] if sb is not None else None
    # text to print: the text-divergence segment if found, otherwise the first segment of the diverging phase run
    ctx_a, basis_a = (sa, "text") if sa is not None else ((int(RUN_FIRSTSEG[ta][k]) if k is not None and k < len(ra) else None), "phase")
    ctx_b, basis_b = (sb, "text") if sb is not None else ((int(RUN_FIRSTSEG[tb][k]) if k is not None and k < len(rb) else None), "phase")
    return dict(cell_id=cid, pair=kind, id_a=ta, id_b=tb, label_a=label_a, label_b=label_b, words_a=wa, words_b=wb,
                phase_div_run=k, phase_div_words_a=pa, phase_div_words_b=pb,
                phase_div_rel_a=None if pa is None else pa / wa, phase_div_rel_b=None if pb is None else pb / wb,
                role_seq_similarity=difflib.SequenceMatcher(None, ra, rb, autojunk=False).ratio(),
                text_div_seg_a=sa, text_div_seg_b=sb, text_div_words_a=ta_w, text_div_words_b=tb_w,
                text_div_rel_a=None if ta_w is None else ta_w / wa, text_div_rel_b=None if tb_w is None else tb_w / wb,
                first_ans_end_a=F.at[ta, "first_ans_end"], first_ans_end_b=F.at[tb, "first_ans_end"],
                context_basis_a=basis_a, context_basis_b=basis_b,
                context_a=context(ta, ctx_a), context_b=context(tb, ctx_b),
                note="algorithmic candidate, needs manual reading")


div_rows = []
for cid in cells.index[cells["group"] == "MIXED"]:
    sub = E.loc[list(cell_members[cid])]
    w, c = sub[sub["correct"] == 0], sub[sub["correct"] == 1]
    # length-matched pair: wrong/correct combination with the smallest |log(words ratio)|
    best = min(((abs(np.log(a / b)), ia, ib) for ia, a in w["words"].items() for ib, b in c["words"].items()))
    div_rows.append(divergence_record(cid, best[1], best[2], "wrong_vs_correct(length-matched)", "wrong", "correct"))
for cid, pr in ac_pairs.iterrows():
    div_rows.append(divergence_record(cid, pr["long_id"], pr["short_id"], "long_vs_short", "long", "short"))
div_df = pd.DataFrame(div_rows)
div_df["precedes_first_ans_a"] = np.where(div_df["text_div_words_a"].notna() & div_df["first_ans_end_a"].notna(),
                                          (div_df["text_div_words_a"] < div_df["first_ans_end_a"]).astype(float), np.nan)
for _, r in div_df.iterrows():          # marker position: text evidence if present, otherwise the phase divergence
    for tid, tw, pw in [(r["id_a"], r["text_div_words_a"], r["phase_div_words_a"]),
                        (r["id_b"], r["text_div_words_b"], r["phase_div_words_b"])]:
        pos = tw if pd.notna(tw) else pw
        if pd.notna(pos):
            div_marks[tid] = float(pos)
save_table(div_df.drop(columns=["id_a", "id_b"]), "divergence_candidates.csv", index=False)
print("divergence candidates computed for %d pairs (%d MIXED, %d ALL-CORRECT)" % (
    len(div_df), (div_df["pair"].str.startswith("wrong")).sum(), (div_df["pair"] == "long_vs_short").sum()))

### 7.3 One figure and one text table for every eligible cell
Files: `cell_figures/cell_<qkey>_<compute>.png` and `tables/cell_text/<same stem>.csv`. A short hash is appended only
where two cells would otherwise share a file name. `tables/cell_index.csv` maps cells to files.

In [ ]:
cell_ids = list(cells.index[cells["group"] != "TOO-FEW"])
stems = {c: "cell_%s_%s" % (safe_name(cells.at[c, "qkey"]), safe_name(cells.at[c, "compute_level"])) for c in cell_ids}
dups = pd.Series(stems).duplicated(keep=False)
for c in dups[dups].index:
    stems[c] += "-" + hashlib.md5(c.encode()).hexdigest()[:6]
index_rows = []
for c in cell_ids:
    fpath = os.path.join(FIG_DIR, stems[c] + ".png")
    tpath = os.path.join(TAB_DIR, "cell_text", stems[c] + ".csv")
    if SAVE_ALL_CELL_FIGURES:
        plot_cell(c, path=fpath)
        tt_ = cell_text_table(c)
        for _, r in div_df[div_df["cell_id"] == c].iterrows():
            for tid, sg in [(r["id_a"], r["text_div_seg_a"]), (r["id_b"], r["text_div_seg_b"])]:
                if pd.notna(sg):
                    tt_.loc[(tt_["sample_index"] == E.at[tid, "sample_index"]) & (tt_["seg_pos"] == sg),
                            "marker_div_candidate"] = True
        tt_.to_csv(tpath, index=False)
    index_rows.append(dict(cell_id=c, group=cells.at[c, "group"], figure=fpath, text_table=tpath))
cell_index = pd.DataFrame(index_rows)
save_table(cell_index, "cell_index.csv", index=False)
n_png = len([f for f in os.listdir(FIG_DIR) if f.endswith(".png")])
print("eligible cells: %d | cell figures on disk: %d | cell text tables: %d" % (
    len(cell_ids), n_png, len(os.listdir(os.path.join(TAB_DIR, "cell_text")))))
if SAVE_ALL_CELL_FIGURES:
    assert n_png == len(cell_ids), "every eligible cell must have exactly one saved figure"

### Representative cells (inline)
Absolute-scale figures first, then the **normalised** variant of the same cell (saved separately under
`cell_figures/normalized/`).

In [ ]:
for g, lst in REP_CELLS.items():
    print("=== %s: representative cells %s" % (g, lst))
    for c in lst:
        plot_cell(c, show=True, path=os.path.join(FIG_DIR, stems[c] + ".png"), dpi=110)
        plot_cell(c, normalized=True, show=(c == lst[0]),
                  path=os.path.join(FIG_DIR, "normalized", stems[c] + "_normalized.png"), dpi=110)
first_cell = REP_CELLS["MIXED"][0] if REP_CELLS["MIXED"] else cell_ids[0]
print("text-inspection table for", first_cell, "(first 12 segments, text clipped for display)")
ti = pd.read_csv(os.path.join(TAB_DIR, "cell_text", stems[first_cell] + ".csv")) if SAVE_ALL_CELL_FIGURES else cell_text_table(first_cell)
display(ti.head(12).assign(text_raw=lambda d: d["text_raw"].astype(str).str.slice(0, 90)))

### 7.4 Feature sets and reusable cell-level table builders
*Length-free* features are shares, rates per 100 words or relative positions, so they do not change just because a
trace is longer. *Length-dependent* features are listed separately and flagged. All builders take an eligible-trace
table and a cell table, so the robustness section can call them with other thresholds.

In [ ]:
LEN_FREE = (["share_" + r for r in ROLES] + ["segshare_" + r for r in KEY_ROLES]
            + ["changes_per100w", "segs_per_run"] + ["runs100_" + r for r in KEY_ROLES]
            + ["doubt_to_verif100", "doubt_to_corr100", "verif_to_doubt100"]
            + ["first_ans_rel", "first_final_rel", "stab_rel", "share_after_first_ans", "share_after_first_final",
               "n_replace_100w", "any_replace", "gold_ever", "gold_lost"]
            + ["novelty", "rep4_share", "near_repeat_wshare", "rep_sim_mean", "num_repeat_share", "sent_repeat_share",
               "eq_repeat_share", "comp_ratio", "loop_flag"]
            + ["hedge_100w"] + ["cue_%s_100w" % k for k in CUES])
LEN_DEP = ["words", "log_words", "n_seg", "mean_seg_words", "n_runs", "words_before_first_ans",
           "words_after_first_ans", "words_after_stab"]
OUTCOME_LINKED = ["gold_ever", "gold_lost", "last_matches_gold"]   # defined with the gold answer -> tied to correctness
ALL_FEATS = LEN_FREE + LEN_DEP
KEY_FEATS = ["share_doubt", "share_verification", "share_correction", "share_reasoning", "changes_per100w",
             "first_ans_rel", "share_after_first_ans", "n_replace_100w", "rep4_share",
             "near_repeat_wshare", "hedge_100w", "log_words"]


def long_short_table(Fe, pairs, feats=ALL_FEATS):
    """Per ALL-CORRECT cell: long minus short (d_<feature>) plus the specific decompositions asked for in the spec."""
    L = Fe.loc[pairs["long_id"]].reset_index(drop=True)
    Sh = Fe.loc[pairs["short_id"]].reset_index(drop=True)
    out = pd.DataFrame(index=pairs.index)
    for f in feats:
        out["d_" + f] = (L[f] - Sh[f]).to_numpy()
    extra = (L["words"] - Sh["words"]).to_numpy()
    out["words_short"], out["words_long"] = Sh["words"].to_numpy(), L["words"].to_numpy()
    out["ratio"] = L["words"].to_numpy() / Sh["words"].to_numpy()
    out["extra_words"] = extra
    for r in ROLES:                                         # which roles hold the extra words
        out["xw_" + r] = (L["words_" + r] - Sh["words_" + r]).to_numpy()
        out["xshare_" + r] = out["xw_" + r] / extra
    with np.errstate(divide="ignore", invalid="ignore"):    # more segments or longer segments?
        out["seg_count_share"] = (np.log(L["n_seg"] / Sh["n_seg"]) / np.log(L["words"] / Sh["words"])).to_numpy()
    both_ans = (L["first_ans_end"].notna() & Sh["first_ans_end"].notna()).to_numpy()
    out["x_before_ans"] = np.where(both_ans, (L["words_before_first_ans"] - Sh["words_before_first_ans"]).to_numpy(), np.nan)
    out["x_after_ans"] = np.where(both_ans, (L["words_after_first_ans"] - Sh["words_after_first_ans"]).to_numpy(), np.nan)
    out["xshare_before_ans"] = out["x_before_ans"] / extra
    out["xshare_after_ans"] = out["x_after_ans"] / extra
    d_rel = (L["first_ans_rel"] - Sh["first_ans_rel"]).to_numpy()
    out["ans_point"] = np.select([np.isnan(d_rel), np.abs(d_rel) <= 0.05, d_rel > 0], ["no answer annotation", "same point (+-5%)",
                                                                                         "later in long"], "earlier in long")
    out["same_last_answer"] = np.where(L["last_val"].notna() & Sh["last_val"].notna(),
                                       (L["last_val"] == Sh["last_val"]).to_numpy().astype(float), np.nan)
    post = L[["postans_words_" + r for r in ROLES]].to_numpy()
    chk_share = post[:, 2:].sum(1) / np.where(post.sum(1) > 0, post.sum(1), np.nan)
    more_repl = (L["n_replace"].fillna(0) > Sh["n_replace"].fillna(0)).to_numpy()
    more_ref = (L["n_refine"].fillna(0) > Sh["n_refine"].fillna(0)).to_numpy()
    out["extra_type"] = np.select(
        [np.isnan(out["xshare_before_ans"]), out["xshare_before_ans"] >= 0.6, more_repl, more_ref, chk_share >= 0.5],
        ["no answer annotation", "extra mainly before first answer", "revises answer", "refines answer / format",
         "checks (verification/doubt/correction)"], default="continues reasoning / restates")
    return out


def mixed_cell_diffs(Fe, cells_df, feats=ALL_FEATS):
    """Per MIXED cell: mean over incorrect samples minus mean over correct samples (within-cell, paired)."""
    ids = cells_df.index[cells_df["group"] == "MIXED"]
    m = Fe[Fe["cell_id"].isin(ids)]
    g = m.groupby(["cell_id", "correct"])[feats].mean()
    w, c = g.xs(0.0, level="correct"), g.xs(1.0, level="correct")
    d = w - c.reindex(w.index)
    d.insert(0, "qkey", cells_df["qkey"].reindex(d.index).to_numpy())
    d.insert(1, "compute_level", cells_df["compute_level"].reindex(d.index).to_numpy())
    return d


def to_question_level(cell_tab, feats):
    return cell_tab.groupby("qkey", sort=False)[feats].mean()


ac_tab = long_short_table(E, ac_pairs)
ac_tab.insert(0, "qkey", cells["qkey"].reindex(ac_tab.index).to_numpy())
ac_tab.insert(1, "compute_level", cells["compute_level"].reindex(ac_tab.index).to_numpy())
mx_tab = mixed_cell_diffs(E, cells)
save_table(ac_tab, "A_allcorrect_long_vs_short_per_cell.csv")
save_table(mx_tab, "A_mixed_wrong_minus_correct_per_cell.csv")
print("ALL-CORRECT cells: %d | MIXED cells: %d | ALL-WRONG cells: %d" % (
    len(ac_tab), len(mx_tab), (cells["group"] == "ALL-WRONG").sum()))

### 7.5 ALL-CORRECT cells: why is the longest correct trace longer than the shortest?
Per cell, long minus short. Questions from the spec: which roles hold the extra words; more segments or longer
segments; extra before or after the first answer; same answer point; does the extra text refine, verify, revise or
change nothing (rule-based label, descriptive only)? Pooled, question-weighted statistics follow in Part B.

In [ ]:
print("ALL-CORRECT cells (n=%d, %d questions)" % (len(ac_tab), ac_tab["qkey"].nunique()))
display(ac_tab[["words_short", "words_long", "ratio", "extra_words"]].describe().T)
print("\nwhere do the extra words sit? (median per-cell share of the extra words, by role)")
display(ac_tab[["xshare_" + r for r in ROLES] + ["xshare_before_ans", "xshare_after_ans"]].median().to_frame("median share"))
print("segment-count share of the length gap (1 = all extra length from more segments, 0 = from longer segments):")
print(ac_tab["seg_count_share"].describe().round(3).to_dict())
for col in ["ans_point", "extra_type"]:
    print("\n" + col + " (cells):")
    display(ac_tab[col].value_counts().to_frame("cells").assign(share=lambda d: d["cells"] / d["cells"].sum()))
print("long and short end on the same last answer in %.0f%% of cells with annotations" %
      (100 * ac_tab["same_last_answer"].mean()))

if len(ac_tab):
    fig, ax = plt.subplots(figsize=(11, max(3, 0.22 * min(len(ac_tab), 45) + 1.5)))
    plot = ac_tab.sort_values("ratio", ascending=False).head(45)
    left = np.zeros(len(plot))
    for r in ROLES:
        v = plot["xw_" + r].clip(lower=0).to_numpy()
        ax.barh(range(len(plot)), v, left=left, color=ROLE_COLORS[r], label=r)
        left += v
    ax.set_yticks(range(len(plot)))
    ax.set_yticklabels([c.replace("::", " ") for c in plot.index], fontsize=6)
    ax.invert_yaxis()
    ax.set_xlabel("extra words in the longest vs the shortest correct trace (words, by role; negative parts not drawn)")
    ax.set_title("ALL-CORRECT cells: where the extra words sit (top 45 cells by length ratio)")
    ax.legend(ncol=5, fontsize=8, loc="lower right")
    save_fig(fig, "A_allcorrect_extra_words_by_role_cells.png")
    plt.show()
    plt.close(fig)

### 7.6 MIXED cells: correct vs incorrect samples of the same question and compute level
Within-cell contrasts (mean of the incorrect samples minus mean of the correct samples). Covered here: length,
role shares, phase order and transitions, first-answer position, answer-change patterns (stable / replaced / gold
candidate later lost), the roles that precede an answer change, and the algorithmic divergence candidates.
Note: the gold-linked categories (`gold_ever`, `gold_lost`, "gold candidate later replaced") use the correct answer, so
they are related to the outcome *by construction*; they describe how a wrong answer came about, they are not
independent predictors and are excluded from the pattern-transfer feature set in 7.8.

In [ ]:
print("MIXED cells: %d (%d questions); samples per side: median %d correct / %d incorrect" % (
    len(mx_tab), mx_tab["qkey"].nunique(), cells[cells["group"] == "MIXED"]["n_correct"].median(),
    cells[cells["group"] == "MIXED"]["n_wrong"].median()))
cell_sign = pd.DataFrame({f: (mx_tab[f] > 0).mean() for f in KEY_FEATS}, index=["share of cells where incorrect > correct"]).T
display(cell_sign.assign(median_diff=mx_tab[KEY_FEATS].median()))

# answer-change patterns (question level: share of each pattern among the incorrect / correct samples of MIXED cells)
mxE = E[E["cell_group"] == "MIXED"]
pat = (mxE.assign(wrong=(mxE["correct"] == 0).map({True: "incorrect", False: "correct"}))
       .groupby(["qkey", "wrong"])["ans_pattern"].value_counts(normalize=True).unstack(fill_value=0))
pat_q = pat.groupby(level="wrong").mean()
print("\nanswer-development pattern among samples of MIXED cells (mean over questions of within-question shares):")
display(pat_q.round(3))
save_table(pat_q, "A_mixed_answer_patterns.csv")

# what precedes an observed answer replacement? roles in the 2 segments before the event
ev_rows = []
for tid in mxE.index:
    for p in A_lists.at[tid, "replace_segs"] if tid in A.index else []:
        ix = seg_ix[tid]
        pre = S_role[ix[max(0, p - 2):p]]
        ev_rows.append(dict(trace_id=tid, qkey=E.at[tid, "qkey"], correct=E.at[tid, "correct"], own_role=S_role[ix[p]],
                            pre_doubt=float("doubt" in pre), pre_verification=float("verification" in pre),
                            pre_correction=float("correction" in pre), pre_any_key=float(any(x in KEY_ROLES for x in pre))))
ev_df = pd.DataFrame(ev_rows)
if len(ev_df):
    ev_q = ev_df.groupby(["correct", "qkey"])[["pre_doubt", "pre_verification", "pre_correction", "pre_any_key"]].mean()
    rows = []
    for cval, name in [(1.0, "correct samples"), (0.0, "incorrect samples")]:
        for col in ev_q.columns:
            m, lo, hi, nq = boot_mean_ci(ev_q.xs(cval, level="correct")[col])
            rows.append(dict(samples=name, feature=col, mean=m, ci_lo=lo, ci_hi=hi, n_questions=nq,
                             n_events=int((ev_df["correct"] == cval).sum())))
    print("\nroles within the 2 segments BEFORE an observed answer replacement (question-level, cluster bootstrap CI):")
    display(pd.DataFrame(rows).set_index(["samples", "feature"]).round(3))
    save_table(pd.DataFrame(rows), "A_mixed_roles_before_answer_change.csv", index=False)
else:
    print("no answer replacements observed in MIXED cells")

# divergence candidates (length-matched wrong/correct pairs)
dv = div_df[div_df["pair"].str.startswith("wrong")]
print("\ndivergence candidates, MIXED length-matched pairs (n=%d):" % len(dv))
print("  text divergence found in the wrong trace: %.0f%% | median relative position: %.2f"
      % (100 * dv["text_div_words_a"].notna().mean(), dv["text_div_rel_a"].astype(float).median()))
print("  text divergence of the wrong trace lies BEFORE its first answer: %.0f%% (of those with both)"
      % (100 * dv["precedes_first_ans_a"].mean()))
print("  median role-sequence similarity of the pair: %.2f" % dv["role_seq_similarity"].median())
print("\nthree examples (algorithmic candidates, need manual reading; '>>' marks the candidate segment):")
for _, r in dv[dv["text_div_seg_a"].notna()].head(3).iterrows():
    print("\n--- %s | wrong trace (%d words) vs correct trace (%d words) | role-sequence similarity %.2f" % (
        r["cell_id"], r["words_a"], r["words_b"], r["role_seq_similarity"]))
    print("WRONG trace around its divergence candidate (basis: %s):\n%s" % (r["context_basis_a"], r["context_a"]))
    print("CORRECT trace around its divergence candidate (basis: %s):\n%s" % (r["context_basis_b"], r["context_b"] or "  (no candidate)"))

### 7.7 ALL-WRONG cells: similar or different trajectories?
Within a cell, pairwise similarity of the samples' trajectories: (i) role-sequence similarity of the merged phase runs
(`difflib` ratio) and (ii) half the L1 distance between role word-share vectors (0 = same composition). MIXED cells
are split into their incorrect and correct sides so the three situations can be compared. Same wrong answer?
Answer changes? Repetition? Recurring phase sequences? (Between-group differences are confounded with question
difficulty; see the notes in 7.8.)

In [ ]:
def pair_similarity(ids):
    ids = list(ids)
    if len(ids) < 2:
        return np.nan, np.nan
    roles = [RUN_ROLES[t] for t in ids]
    shares = E.loc[ids, ["share_" + r for r in ROLES]].to_numpy()
    sims, l1 = [], []
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            sims.append(difflib.SequenceMatcher(None, roles[i], roles[j], autojunk=False).ratio())
            l1.append(np.abs(shares[i] - shares[j]).sum() / 2)
    return float(np.mean(sims)), float(np.mean(l1))


def modal_answer_stats(ids):
    vals = [E.at[t, "last_val"] for t in ids if pd.notna(E.at[t, "last_val"])]
    if len(vals) < 2:
        return dict(n_with_value=len(vals), modal_share=np.nan, n_distinct=len(set(vals)), all_same=np.nan)
    cnt = Counter(vals)
    return dict(n_with_value=len(vals), modal_share=cnt.most_common(1)[0][1] / len(vals), n_distinct=len(cnt),
                all_same=float(len(cnt) == 1))


sim_rows = []
for cid in cell_ids:
    sub = E.loc[list(cell_members[cid])]
    grp = cells.at[cid, "group"]
    if grp == "MIXED":
        for side, ids in [("MIXED-incorrect samples", sub.index[sub["correct"] == 0]),
                          ("MIXED-correct samples", sub.index[sub["correct"] == 1])]:
            s, l = pair_similarity(ids)
            sim_rows.append(dict(cell_id=cid, qkey=cells.at[cid, "qkey"], set=side, role_seq_sim=s, role_share_l1=l,
                                 **modal_answer_stats(ids) if "incorrect" in side else {}))
    else:
        s, l = pair_similarity(sub.index)
        sim_rows.append(dict(cell_id=cid, qkey=cells.at[cid, "qkey"], set=grp, role_seq_sim=s, role_share_l1=l,
                             **modal_answer_stats(sub.index) if grp == "ALL-WRONG" else {}))
sim_df = pd.DataFrame(sim_rows)
save_table(sim_df, "A_within_cell_trajectory_similarity.csv", index=False)
rows = []
for sname, sub in sim_df.groupby("set"):
    qs = sub.groupby("qkey")[["role_seq_sim", "role_share_l1"]].mean()
    r1, r2 = boot_mean_ci(qs["role_seq_sim"]), boot_mean_ci(qs["role_share_l1"])
    rows.append(dict(set=sname, n_cells=len(sub), n_questions=len(qs), role_seq_similarity=r1[0],
                     seq_ci=(round(r1[1], 3), round(r1[2], 3)), role_share_L1=r2[0], L1_ci=(round(r2[1], 3), round(r2[2], 3))))
display(pd.DataFrame(rows).set_index("set").round(3))
aw = sim_df[sim_df["set"].isin(["ALL-WRONG", "MIXED-incorrect samples"]) & sim_df["modal_share"].notna()]
rows = []
for sname, sub in aw.groupby("set"):
    qs = sub.groupby("qkey")[["modal_share", "n_distinct", "all_same"]].mean()
    rows.append(dict(set=sname, n_cells=len(sub), n_questions=len(qs), modal_wrong_answer_share=qs["modal_share"].mean(),
                     distinct_final_answers=qs["n_distinct"].mean(), cells_with_one_shared_answer=qs["all_same"].mean()))
print("\nDo incorrect samples of a cell converge on the SAME wrong final answer? (cells with >= 2 annotated final values)")
display(pd.DataFrame(rows).set_index("set").round(3))

grp_map = {"ALL-WRONG": E["cell_group"] == "ALL-WRONG",
           "MIXED, incorrect samples": (E["cell_group"] == "MIXED") & (E["correct"] == 0),
           "MIXED, correct samples": (E["cell_group"] == "MIXED") & (E["correct"] == 1),
           "ALL-CORRECT (+narrow)": E["cell_group"].isin(["ALL-CORRECT", "ALL-CORRECT-NARROW"])}
chk_feats = ["any_replace", "n_replace_100w", "gold_ever", "gold_lost", "rep4_share", "sent_repeat_share", "loop_flag",
             "near_repeat_wshare", "changes_per100w"]
rows = {name: hier_mean(E[m], chk_feats).mean() for name, m in grp_map.items()}
print("\nanswer changes and repetition by group (mean of question-level means):")
display(pd.DataFrame(rows).T.round(3))


def ngram_counts(ids, n):
    rows = {t: Counter(">".join(RUN_ROLES[t][i:i + n]) for i in range(len(RUN_ROLES[t]) - n + 1)) for t in ids}
    return pd.DataFrame.from_dict(rows, orient="index").fillna(0)


aw_ids, ac_ids_all = E.index[grp_map["ALL-WRONG"]], E.index[grp_map["ALL-CORRECT (+narrow)"]]
ng = ngram_counts(list(aw_ids) + list(ac_ids_all), 3)
rate = ng.div(E.loc[ng.index, "words"], axis=0) * 100
qk = E.loc[ng.index, "qkey"]
r_aw = rate.loc[aw_ids].groupby(qk.loc[aw_ids]).mean().mean()
r_ac = rate.loc[ac_ids_all].groupby(qk.loc[ac_ids_all]).mean().mean()
top = pd.DataFrame({"per100w_ALL-WRONG": r_aw, "per100w_ALL-CORRECT": r_ac}).sort_values("per100w_ALL-WRONG", ascending=False).head(10)
top["ratio"] = top.iloc[:, 0] / top.iloc[:, 1].replace(0, np.nan)
print("\nmost frequent 3-run phase sequences in ALL-WRONG traces (per 100 words, mean of question means):")
display(top.round(3))

### 7.8 MIXED -> ALL-WRONG pattern transfer
**Step 1** - inside MIXED cells, find which structural patterns separate incorrect from correct (paired within the
cell, aggregated per question; effect size `dz`). **Step 2** - check whether the same patterns appear in ALL-WRONG
cells, with ALL-CORRECT as the other reference: Cohen's `d` between question-level means (ALL-WRONG minus
ALL-CORRECT), also computed within each compute level and averaged, and the position of ALL-WRONG between the
MIXED-correct and MIXED-incorrect means (0 = like MIXED-correct, 1 = like MIXED-incorrect).
Descriptive and exploratory. **Caveat:** ALL-WRONG and ALL-CORRECT cells come from different questions (typically
different difficulty), so the group contrast is confounded in a way the within-cell MIXED contrast is not; this is
a consistency check on direction and size, not a classifier and not evidence of cause.

In [ ]:
PATTERN_FEATS = ["share_doubt", "share_verification", "share_correction", "share_reasoning", "share_setup",
                 "changes_per100w", "runs100_doubt", "runs100_verification", "runs100_correction",
                 "doubt_to_verif100", "doubt_to_corr100", "verif_to_doubt100", "first_ans_rel", "share_after_first_ans",
                 "n_replace_100w", "any_replace", "rep4_share", "near_repeat_wshare", "sent_repeat_share",
                 "loop_flag", "hedge_100w", "novelty", "comp_ratio", "mean_seg_words"]


def cohens_d_ci(a, b, B=1000, seed=SEED):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    if len(a) < 3 or len(b) < 3:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)

    def d(x, y):
        sp = np.sqrt(((len(x) - 1) * x.var(ddof=1) + (len(y) - 1) * y.var(ddof=1)) / (len(x) + len(y) - 2))
        return (x.mean() - y.mean()) / sp if sp > 0 else np.nan

    boots = [d(a[rng.integers(0, len(a), len(a))], b[rng.integers(0, len(b), len(b))]) for _ in range(B)]
    return d(a, b), nanpct(boots, 2.5), nanpct(boots, 97.5)


def transfer_table(Fe, cells_df, feats=PATTERN_FEATS):
    mx = mixed_cell_diffs(Fe, cells_df, feats + ["log_words"])
    eff = paired_summary(to_question_level(mx, feats + ["log_words"]))
    g = Fe["cell_group"]
    sets = {"aw": Fe[g == "ALL-WRONG"], "ac": Fe[g.isin(["ALL-CORRECT", "ALL-CORRECT-NARROW"])],
            "mxw": Fe[(g == "MIXED") & (Fe["correct"] == 0)], "mxc": Fe[(g == "MIXED") & (Fe["correct"] == 1)]}
    qm = {k: hier_mean(v, feats + ["log_words"]) for k, v in sets.items()}
    rows = []
    for f in feats + ["log_words"]:
        d_ac = cohens_d_ci(qm["aw"][f], qm["ac"][f])
        # same contrast inside each compute level, averaged (controls compute, not difficulty)
        per_level = []
        for lv in COMPUTE_ORDER:
            a = hier_mean(sets["aw"][sets["aw"]["compute_level"] == lv], [f])[f] if (sets["aw"]["compute_level"] == lv).any() else []
            b = hier_mean(sets["ac"][sets["ac"]["compute_level"] == lv], [f])[f] if (sets["ac"]["compute_level"] == lv).any() else []
            if len(a) >= 5 and len(b) >= 5:
                per_level.append(cohens_d_ci(a, b, B=200)[0])
        gap = qm["mxw"][f].mean() - qm["mxc"][f].mean()
        meaningful = f in eff.index and abs(eff.at[f, "dz"]) >= 0.2      # position is unstable when the MIXED gap is ~0
        pos = (qm["aw"][f].mean() - qm["mxc"][f].mean()) / gap if meaningful and abs(gap) > 1e-12 else np.nan
        rows.append(dict(feature=f, dz_mixed=eff.at[f, "dz"] if f in eff.index else np.nan,
                         dz_lo=eff.at[f, "dz_lo"] if f in eff.index else np.nan,
                         dz_hi=eff.at[f, "dz_hi"] if f in eff.index else np.nan,
                         d_allwrong_vs_allcorrect=d_ac[0], d_lo=d_ac[1], d_hi=d_ac[2],
                         d_within_compute_level=np.mean(per_level) if per_level else np.nan,
                         allwrong_position_between_mixed_means=pos,
                         mean_ALLCORRECT=qm["ac"][f].mean(), mean_MIXED_correct=qm["mxc"][f].mean(),
                         mean_MIXED_incorrect=qm["mxw"][f].mean(), mean_ALLWRONG=qm["aw"][f].mean(),
                         n_q_allwrong=qm["aw"][f].notna().sum(), n_q_allcorrect=qm["ac"][f].notna().sum(),
                         n_q_mixed=int(eff.at[f, "n_questions"]) if f in eff.index else 0))
    return pd.DataFrame(rows).set_index("feature")


transfer = transfer_table(E, cells)
save_table(transfer, "A_mixed_to_allwrong_transfer.csv")
display(transfer[["dz_mixed", "dz_lo", "dz_hi", "d_allwrong_vs_allcorrect", "d_lo", "d_hi", "d_within_compute_level",
                  "allwrong_position_between_mixed_means", "n_q_mixed", "n_q_allwrong", "n_q_allcorrect"]].round(3))
ok = transfer.drop(index="log_words").dropna(subset=["dz_mixed", "d_allwrong_vs_allcorrect"])
sign_agree = (np.sign(ok["dz_mixed"]) == np.sign(ok["d_allwrong_vs_allcorrect"])).mean() if len(ok) else np.nan
rho = stats.spearmanr(ok["dz_mixed"], ok["d_allwrong_vs_allcorrect"])[0] if len(ok) > 3 else np.nan
big = ok[ok["dz_mixed"].abs() >= 0.2]
print("length-free features compared: %d | direction agrees (MIXED effect vs ALL-WRONG-minus-ALL-CORRECT): %.0f%% | "
      "Spearman rho between the two effect-size columns: %.2f" % (len(ok), 100 * sign_agree, rho))
if len(big):
    print("restricted to features with |dz_mixed| >= 0.2 (n=%d): direction agrees for %.0f%%" % (
        len(big), 100 * (np.sign(big["dz_mixed"]) == np.sign(big["d_allwrong_vs_allcorrect"])).mean()))
print("question counts by group and difficulty tier (shows the confounding):")
qg = pd.concat({"ALL-WRONG": E[grp_map["ALL-WRONG"]].drop_duplicates("qkey").set_index("qkey")["difficulty_tier"],
                "ALL-CORRECT (+narrow)": E[grp_map["ALL-CORRECT (+narrow)"]].drop_duplicates("qkey").set_index("qkey")["difficulty_tier"]})
display(qg.groupby(level=0).value_counts().unstack(fill_value=0))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax = axes[0]
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(0, color="grey", lw=0.8)
ax.errorbar(ok["dz_mixed"], ok["d_allwrong_vs_allcorrect"], xerr=[ok["dz_mixed"] - ok["dz_lo"], ok["dz_hi"] - ok["dz_mixed"]],
            yerr=[ok["d_allwrong_vs_allcorrect"] - ok["d_lo"], ok["d_hi"] - ok["d_allwrong_vs_allcorrect"]],
            fmt="o", color="#4c78a8", ecolor="#c8d3e0", ms=4, lw=0.8)
for f, r in ok.iterrows():
    if abs(r["dz_mixed"]) >= 0.2 or abs(r["d_allwrong_vs_allcorrect"]) >= 0.8:
        ax.annotate(f, (r["dz_mixed"], r["d_allwrong_vs_allcorrect"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("MIXED cells: incorrect minus correct, paired effect size dz (question level)")
ax.set_ylabel("ALL-WRONG minus ALL-CORRECT, Cohen d (question-level means)")
ax.set_title("Do MIXED-cell patterns reappear in ALL-WRONG cells? (length-free features)")
ax = axes[1]
top_f = ok.reindex(ok["dz_mixed"].abs().sort_values(ascending=False).index).head(10).index
z = pd.DataFrame({k: transfer.loc[top_f, c] for k, c in [("ALL-CORRECT", "mean_ALLCORRECT"), ("MIXED correct", "mean_MIXED_correct"),
                                                          ("MIXED incorrect", "mean_MIXED_incorrect"), ("ALL-WRONG", "mean_ALLWRONG")]})
z = (z.sub(z.mean(axis=1), axis=0)).div(z.std(axis=1).replace(0, 1), axis=0)
for k, col in zip(z.columns, ["#54a24b", "#9ecae9", "#f58518", "#e45756"]):
    ax.scatter(z[k], range(len(z)), label=k, color=col, s=45, zorder=3)
ax.set_yticks(range(len(z)))
ax.set_yticklabels(z.index, fontsize=8)
ax.invert_yaxis()
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("group mean, standardised across the four groups (per feature)")
ax.set_title("Group profiles for the 10 features with the largest MIXED effect")
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "A_mixed_to_allwrong_transfer.png")
plt.show()
plt.close(fig)

## 8. Part B: pooled analyses across all questions (the universal pattern)
Hierarchy everywhere: **trace -> cell -> question (equal weight) -> pooled**. Differences are always *within a cell*
(long - short, incorrect - correct), averaged per question first, and only then summarised across questions with a
cluster bootstrap over questions. The per-cell figures from Part A are already saved; this part produces one
aggregated figure per analysis.

### 8.0 Position profiles (arrays reused below)
For each trace, the words of every role are distributed over **10 relative-position bins** (deciles of the trace) and
over **fixed 100-word absolute bins**. A segment that straddles a bin edge is split by word overlap.

In [ ]:
nF = len(F)
code = F.index.get_indexer(S["trace_id"]) * 5 + S["role_code"].to_numpy()
twords_seg = S["twords"].to_numpy(float)
REL = np.zeros((nF, 5, REL_BINS))
for b in range(REL_BINS):
    lo, hi = b / REL_BINS * twords_seg, (b + 1) / REL_BINS * twords_seg
    ov = np.clip(np.minimum(S_end, hi) - np.maximum(S_start, lo), 0, None)
    REL[:, :, b] = np.bincount(code, weights=ov, minlength=nF * 5).reshape(nF, 5)
REL_SH = REL / (F["words"].to_numpy(float)[:, None, None] / REL_BINS)         # role share inside each decile
assert np.allclose(REL.sum(axis=(1, 2)), F["words"].to_numpy()), "relative bins must add up to the trace length"

ABS_NB = int(np.ceil(E["words"].quantile(0.99) / ABS_BIN_WORDS))
ABS = np.zeros((nF, 5, ABS_NB))
for b in range(ABS_NB):
    lo, hi = b * ABS_BIN_WORDS, (b + 1) * ABS_BIN_WORDS
    ov = np.clip(np.minimum(S_end, hi) - np.maximum(S_start, lo), 0, None)
    ABS[:, :, b] = np.bincount(code, weights=ov, minlength=nF * 5).reshape(nF, 5)
ABS_PRESENT = ABS.sum(axis=1)                                                 # words of the trace inside each bin
with np.errstate(divide="ignore", invalid="ignore"):                           # bins that are < half filled are ignored
    ABS_SH = np.where(ABS_PRESENT[:, None, :] >= ABS_BIN_WORDS / 2, ABS / ABS_PRESENT[:, None, :], np.nan)
print("relative bins: %d | absolute bins: %d x %d words (covers the 99th percentile of eligible trace lengths)"
      % (REL_BINS, ABS_NB, ABS_BIN_WORDS))

COND = {"ALL-CORRECT short": ac_pairs["short_id"].to_numpy(), "ALL-CORRECT long": ac_pairs["long_id"].to_numpy(),
        "MIXED correct": E.index[(E["cell_group"] == "MIXED") & (E["correct"] == 1)].to_numpy(),
        "MIXED incorrect": E.index[(E["cell_group"] == "MIXED") & (E["correct"] == 0)].to_numpy(),
        "ALL-WRONG": E.index[E["cell_group"] == "ALL-WRONG"].to_numpy()}
COND_COLORS = {"ALL-CORRECT short": "#9ecae9", "ALL-CORRECT long": "#08519c", "MIXED correct": "#74c476",
               "MIXED incorrect": "#f58518", "ALL-WRONG": "#d62728"}
print({k: len(v) for k, v in COND.items()})


def hier_q_array(arr2d, ids):
    """trace -> cell mean -> question mean for a 2-D array aligned with F (NaN skipped). Returns (n_questions, k)."""
    pos = F.index.get_indexer(ids)
    d = pd.DataFrame(arr2d[pos], columns=range(arr2d.shape[1]))
    d["cell"] = F["cell_id"].to_numpy()[pos]
    c = d.groupby("cell", sort=False).mean()
    qmap = pd.Series(F["qkey"].to_numpy()[pos], index=d["cell"].to_numpy()).groupby(level=0).first()
    c["q"] = qmap.reindex(c.index).to_numpy()
    return c.groupby("q", sort=False).mean().to_numpy()


def paired_profile_diff(arr2d, kind):
    """Within-cell paired difference of a 2-D array (long-short or incorrect-correct), averaged per question."""
    if kind == "long-short":
        diff = arr2d[F.index.get_indexer(ac_pairs["long_id"])] - arr2d[F.index.get_indexer(ac_pairs["short_id"])]
        d = pd.DataFrame(diff, columns=range(arr2d.shape[1]))
        d["q"] = cells["qkey"].reindex(ac_pairs.index).to_numpy()
        return d.groupby("q", sort=False).mean().to_numpy()
    m = E[E["cell_group"] == "MIXED"]
    d = pd.DataFrame(arr2d[F.index.get_indexer(m.index)], columns=range(arr2d.shape[1]))
    d["cell"], d["correct"] = m["cell_id"].to_numpy(), m["correct"].to_numpy()
    g = d.groupby(["cell", "correct"]).mean()
    diff = g.xs(0.0, level="correct") - g.xs(1.0, level="correct").reindex(g.xs(0.0, level="correct").index)
    diff["q"] = cells["qkey"].reindex(diff.index).to_numpy()
    return diff.groupby("q", sort=False).mean().to_numpy()


def forest(ax, eff, title, highlight=()):
    eff = eff.dropna(subset=["dz"]).sort_values("dz")
    y = np.arange(len(eff))
    col = ["#8c8c8c" if f in OUTCOME_LINKED else "#d95f02" if f in LEN_DEP else "#1b6ca8" for f in eff.index]
    ax.hlines(y, eff["dz_lo"], eff["dz_hi"], color=col, lw=1.4)
    ax.scatter(eff["dz"], y, color=col, s=22, zorder=3)
    ax.axvline(0, color="grey", lw=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(eff.index, fontsize=7)
    ax.set_xlabel("paired effect size dz (question level, 95% cluster-bootstrap CI)")
    ax.set_title(title, fontsize=10)

### 8.1 Longest vs shortest correct trace (ALL-CORRECT cells), pooled
Per cell: long minus short; per question: mean over its cells; pooled: mean over questions with a cluster-bootstrap
CI. **Permutation baseline:** because the longest and shortest samples of a cell are *selected* by length, the extra
words must go *somewhere*. The baseline shuffles the role-share vectors (and first-answer positions) among the
samples of each cell while keeping their word counts, which is what the share of extra words per role would be if
composition were unrelated to length. Observed shares are compared with that null.

In [ ]:
ac_tab["log_ratio"] = np.log(ac_tab["ratio"])
b1_cols = (["log_ratio", "extra_words", "seg_count_share"] + ["xshare_" + r for r in ROLES]
           + ["xshare_before_ans", "xshare_after_ans"] + ["d_" + f for f in ALL_FEATS])
ac_q = to_question_level(ac_tab, b1_cols)
eff_ac = paired_summary(ac_q, b1_cols)
save_table(eff_ac, "B1_longest_vs_shortest_pooled.csv")
print("ALL-CORRECT: %d cells, %d questions" % (len(ac_tab), len(ac_q)))
print("pooled length ratio long/short: %.2f (95%% CI %.2f-%.2f)" % tuple(np.exp(eff_ac.loc["log_ratio", ["mean", "ci_lo", "ci_hi"]])))

null_cols = ["share_" + r for r in ROLES] + ["first_ans_rel"]
obs_null, null_mat = None, None      # stay None when there are no ALL-CORRECT cells
if len(ac_pairs):
    cl = list(ac_pairs.index)
    nc, maxn = len(cl), max(len(cell_members[c]) for c in cl)
    V = np.full((nc, maxn, len(null_cols)), np.nan)
    W = np.full((nc, maxn), np.nan)
    for i, c in enumerate(cl):
        ids = list(cell_members[c])
        V[i, :len(ids)] = E.loc[ids, null_cols].to_numpy()
        W[i, :len(ids)] = E.loc[ids, "words"].to_numpy()
    li, si, ar = np.nanargmax(W, 1), np.nanargmin(W, 1), np.arange(nc)
    extra = W[ar, li] - W[ar, si]
    qcode = pd.factorize(cells["qkey"].reindex(cl))[0]
    nq = qcode.max() + 1

    def pooled_xshare(Vp):
        xs = (Vp[ar, li] * W[ar, li][:, None] - Vp[ar, si] * W[ar, si][:, None]) / extra[:, None]
        out = np.empty(Vp.shape[2])
        for j in range(Vp.shape[2]):
            ok = ~np.isnan(xs[:, j])
            n_q = np.bincount(qcode[ok], minlength=nq)
            m_q = np.bincount(qcode[ok], weights=xs[ok, j], minlength=nq) / np.maximum(n_q, 1)
            out[j] = m_q[n_q > 0].mean()
        return out

    rng_n = np.random.default_rng(SEED)
    obs_null = pooled_xshare(V)
    null_mat = np.empty((N_PERM, len(null_cols)))
    for b in range(N_PERM):
        r = rng_n.random((nc, maxn))
        r[np.isnan(W)] = 2.0
        perm = np.argsort(r, axis=1)
        null_mat[b] = pooled_xshare(np.take_along_axis(V, perm[:, :, None], axis=1))
    base = pd.DataFrame({"observed_share_of_extra_words": obs_null,
                         "null_mean": null_mat.mean(0), "null_lo": np.percentile(null_mat, 2.5, axis=0),
                         "null_hi": np.percentile(null_mat, 97.5, axis=0),
                         "p_two_sided": [(1 + (np.abs(null_mat[:, j] - null_mat[:, j].mean()) >=
                                               abs(obs_null[j] - null_mat[:, j].mean())).sum()) / (N_PERM + 1)
                                         for j in range(len(null_cols))]},
                        index=[c.replace("share_", "extra words in role: ").replace("first_ans_rel", "extra words BEFORE first answer")
                               for c in null_cols])
    print("\nshare of the extra words by role / before the first answer: observed vs permutation baseline "
          "(composition unrelated to length), %d permutations" % N_PERM)
    display(base.round(3))
    save_table(base, "B1_extra_words_permutation_baseline.csv")

rel_s = hier_q_array(REL_SH.reshape(nF, -1), ac_pairs["short_id"].to_numpy()) if len(ac_pairs) else None
rel_l = hier_q_array(REL_SH.reshape(nF, -1), ac_pairs["long_id"].to_numpy()) if len(ac_pairs) else None
rel_d_ac = paired_profile_diff(REL_SH.reshape(nF, -1), "long-short") if len(ac_pairs) else None

fig = plt.figure(figsize=(16, 9))
gs = fig.add_gridspec(2, 3)
ax = fig.add_subplot(gs[0, 0])
labs = ROLES + ["BEFORE first answer", "AFTER first answer"]
vals = [eff_ac.loc["xshare_" + r] for r in ROLES] + [eff_ac.loc["xshare_before_ans"], eff_ac.loc["xshare_after_ans"]]
cols_ = [ROLE_COLORS[r] for r in ROLES] + ["#444444", "#999999"]
ax.bar(range(len(vals)), [v["mean"] for v in vals], color=cols_,
       yerr=[[v["mean"] - v["ci_lo"] for v in vals], [v["ci_hi"] - v["mean"] for v in vals]], capsize=3)
if obs_null is not None:
    for j, r in enumerate(ROLES):
        ax.hlines(null_mat[:, j].mean(), j - 0.4, j + 0.4, color="black", lw=2, ls="--")
        ax.vlines(j, np.percentile(null_mat[:, j], 2.5), np.percentile(null_mat[:, j], 97.5), color="black", lw=1)
    ax.hlines(null_mat[:, -1].mean(), 5 - 0.4, 5 + 0.4, color="black", lw=2, ls="--", label="permutation baseline (dashed, 95% band)")
    ax.legend(fontsize=7)
ax.set_xticks(range(len(vals)))
ax.set_xticklabels(labs, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("share of the extra words (long - short)")
ax.set_title("(a) Where the extra words of the longest\ncorrect trace sit (n=%d questions)" % len(ac_q), fontsize=10)

ax = fig.add_subplot(gs[0, 1])
d_items = [("first answer\nposition (rel.)", "d_first_ans_rel"), ("share of text\nafter 1st answer", "d_share_after_first_ans"),
           ("doubt share", "d_share_doubt"), ("verification\nshare", "d_share_verification"),
           ("correction\nshare", "d_share_correction"), ("repeated 4-grams", "d_rep4_share"), ("novelty", "d_novelty")]
ax.errorbar(range(len(d_items)), [eff_ac.at[k, "mean"] for _, k in d_items],
            yerr=[[eff_ac.at[k, "mean"] - eff_ac.at[k, "ci_lo"] for _, k in d_items],
                  [eff_ac.at[k, "ci_hi"] - eff_ac.at[k, "mean"] for _, k in d_items]], fmt="o", capsize=3, color="#1b6ca8")
ax.axhline(0, color="grey", lw=0.8)
ax.set_xticks(range(len(d_items)))
ax.set_xticklabels([n for n, _ in d_items], fontsize=7)
ax.set_ylabel("long minus short (length-free features)")
ax.set_title("(b) Composition differences, long vs short", fontsize=10)

ax = fig.add_subplot(gs[0, 2])
s = eff_ac.loc["seg_count_share"]
ax.bar([0], [s["mean"]], yerr=[[s["mean"] - s["ci_lo"]], [s["ci_hi"] - s["mean"]]], color="#8c6bb1", capsize=4)
ax.axhline(1, color="grey", ls=":")
ax.axhline(0, color="grey", ls=":")
ax.set_xticks([0])
ax.set_xticklabels(["share of the log length gap\nexplained by MORE segments"])
ax.set_ylim(-0.1, 1.2)
ax.set_title("(c) More segments (1) or longer segments (0)?", fontsize=10)

x = (np.arange(REL_BINS) + 0.5) / REL_BINS * 100
for k, (arr, ttl) in enumerate([(rel_s, "(d) role composition along the trace: SHORTEST correct"),
                                (rel_l, "(e) role composition along the trace: LONGEST correct")]):
    ax = fig.add_subplot(gs[1, k])
    if arr is not None:
        m = np.nanmean(arr, axis=0).reshape(5, REL_BINS)
        ax.stackplot(x, m, colors=[ROLE_COLORS[r] for r in ROLES], labels=ROLES)
    ax.set_xlim(x[0], x[-1])
    ax.set_ylim(0, 1)
    ax.set_xlabel("relative position in the trace (%)")
    ax.set_title(ttl, fontsize=9)
    if k == 0:
        ax.legend(fontsize=7, loc="lower left", ncol=2)
ax = fig.add_subplot(gs[1, 2])
if rel_d_ac is not None:
    mean, lo, hi = boot_matrix_ci(rel_d_ac)
    for r in KEY_ROLES:
        j = ROLES.index(r)
        sl = slice(j * REL_BINS, (j + 1) * REL_BINS)
        ax.plot(x, mean[sl], color=ROLE_COLORS[r], label=r, marker="o", ms=3)
        ax.fill_between(x, lo[sl], hi[sl], color=ROLE_COLORS[r], alpha=0.2)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("relative position in the trace (%)")
ax.set_ylabel("role share: long minus short")
ax.set_title("(f) Where the long trace has more doubt / verification / correction", fontsize=9)
ax.legend(fontsize=7)
fig.suptitle("B1 · Longest vs shortest correct trace within the same question and compute level (ALL-CORRECT cells)", fontsize=12)
fig.tight_layout()
save_fig(fig, "B1_longest_vs_shortest.png")
plt.show()
plt.close(fig)

### 8.2 Correct vs incorrect samples (MIXED cells), pooled
Per cell: mean over the incorrect samples minus mean over the correct samples; per question: mean over its MIXED
cells; pooled: effect size `dz` (mean of question-level differences / their SD) with a cluster-bootstrap CI. Blue =
length-free features, orange = length-dependent features (words, segments ...), shown for context only, **grey =
outcome-linked features** (`gold_ever`, `gold_lost` use the gold answer, so they are tied to correctness by
construction and are not treated as discovered patterns).

In [ ]:
mx_tab["incorrect_longer"] = (mx_tab["words"] > 0).astype(float)
b2_cols = ALL_FEATS + ["incorrect_longer"]
mx_q = to_question_level(mx_tab, b2_cols)
eff_mx = paired_summary(mx_q, b2_cols)
save_table(eff_mx, "B2_correct_vs_incorrect_pooled.csv")
print("MIXED: %d cells, %d questions" % (len(mx_tab), len(mx_q)))
print("P(incorrect samples are longer than correct samples of the same cell): %.3f (95%% CI %.3f-%.3f)" %
      tuple(eff_mx.loc["incorrect_longer", ["mean", "ci_lo", "ci_hi"]]))
show = eff_mx.drop(index="incorrect_longer")
display(show.assign(sig_zero_excluded=(show["dz_lo"] > 0) | (show["dz_hi"] < 0)).sort_values("dz", key=abs, ascending=False)
        .head(15)[["n_questions", "mean", "ci_lo", "ci_hi", "dz", "dz_lo", "dz_hi", "p_signflip", "p_wilcoxon"]].round(3))
fig, ax = plt.subplots(figsize=(8, 10))
forest(ax, show, "B2 · MIXED cells: incorrect minus correct (within-cell, per question)\n"
                 "blue = length-free, orange = length-dependent, grey = outcome-linked; exploratory, no multiplicity correction")
fig.tight_layout()
save_fig(fig, "B2_correct_vs_incorrect_forest.png")
plt.show()
plt.close(fig)

### 8.3 Pooled profiles: role share by relative-position decile
Role shares inside each decile of the trace, averaged trace -> cell -> question, with 95% cluster-bootstrap bands.
Short/long come from ALL-CORRECT cells, correct/incorrect from MIXED cells (ALL-WRONG shown as a reference).
Differences are within-cell paired, then averaged per question.

In [ ]:
REL_FLAT = REL_SH.reshape(nF, -1)
prof = {name: hier_q_array(REL_FLAT, ids) for name, ids in COND.items() if len(ids)}
d_ac = paired_profile_diff(REL_FLAT, "long-short") if len(ac_pairs) else None
d_mx = paired_profile_diff(REL_FLAT, "mixed")
xr = (np.arange(REL_BINS) + 0.5) / REL_BINS * 100

fig = plt.figure(figsize=(17, 8.5))
gs = fig.add_gridspec(2, 5)
for k, (name, arr) in enumerate(prof.items()):
    ax = fig.add_subplot(gs[0, k])
    m = np.nanmean(arr, axis=0).reshape(5, REL_BINS)
    ax.stackplot(xr, m, colors=[ROLE_COLORS[r] for r in ROLES], labels=ROLES)
    ax.set_xlim(xr[0], xr[-1]); ax.set_ylim(0, 1)
    ax.set_title("%s\n(n=%d questions)" % (name, arr.shape[0]), fontsize=9, color=COND_COLORS[name])
    ax.set_xlabel("relative position (%)", fontsize=8)
    if k == 0:
        ax.legend(fontsize=6.5, loc="lower left", ncol=2)
        ax.set_ylabel("role share of words")
for k, (arr, ttl) in enumerate([(d_ac, "long minus short (ALL-CORRECT, within cell)"),
                                (d_mx, "incorrect minus correct (MIXED, within cell)")]):
    ax = fig.add_subplot(gs[1, k * 2:(k + 1) * 2])
    if arr is not None:
        mean, lo, hi = boot_matrix_ci(arr)
        for r in KEY_ROLES + ["reasoning"]:
            j = ROLES.index(r)
            sl = slice(j * REL_BINS, (j + 1) * REL_BINS)
            ax.plot(xr, mean[sl], color=ROLE_COLORS[r], label=r, marker="o", ms=3)
            ax.fill_between(xr, lo[sl], hi[sl], color=ROLE_COLORS[r], alpha=0.18)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_title(ttl, fontsize=9)
    ax.set_xlabel("relative position in the trace (%)")
    ax.set_ylabel("difference in role share")
    ax.legend(fontsize=7, ncol=2)
ax = fig.add_subplot(gs[1, 4])
for name in ["MIXED correct", "MIXED incorrect", "ALL-WRONG"]:
    if name in prof:
        m = np.nanmean(prof[name], axis=0).reshape(5, REL_BINS)
        ax.plot(xr, m[ROLES.index("doubt")] + m[ROLES.index("correction")], color=COND_COLORS[name], label=name, marker="o", ms=3)
ax.set_title("doubt + correction share by decile", fontsize=9)
ax.set_xlabel("relative position (%)")
ax.legend(fontsize=7)
fig.suptitle("B3 · Role composition along the trace (relative position), pooled over questions", fontsize=12)
fig.tight_layout()
save_fig(fig, "B3_relative_position_profiles.png")
plt.show()
plt.close(fig)
rows = []
for name, arr in prof.items():
    mean, lo, hi = boot_matrix_ci(arr, B=200)
    for j, r in enumerate(ROLES):
        for b in range(REL_BINS):
            rows.append(dict(condition=name, role=r, decile=b + 1, mean=mean[j * REL_BINS + b], ci_lo=lo[j * REL_BINS + b],
                             ci_hi=hi[j * REL_BINS + b], n_questions=arr.shape[0]))
save_table(pd.DataFrame(rows), "B3_relative_profiles.csv", index=False)

### 8.4 Absolute-position view
Role proportion inside fixed 100-word bins. Traces end at different lengths, so the **number of traces that still
contribute to each bin** is shown (grey line, right axis); a bin is drawn only if at least `ABS_MIN_TRACES` traces reach
it with >= half the bin filled. The composition at large positions is therefore driven by the few longest traces.

In [ ]:
ABS_FLAT = ABS_SH.reshape(nF, -1)
abs_conds = [c for c in ["ALL-CORRECT short", "ALL-CORRECT long", "MIXED correct", "MIXED incorrect"] if len(COND[c])]
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=True)
abs_rows = []
for ax, name in zip(axes.ravel(), abs_conds):
    ids = COND[name]
    pos = F.index.get_indexer(ids)
    n_tr = (ABS_PRESENT[pos] >= ABS_BIN_WORDS / 2).sum(axis=0)
    arr = hier_q_array(ABS_FLAT, ids)
    m = np.nanmean(arr, axis=0).reshape(5, ABS_NB)
    show = n_tr >= ABS_MIN_TRACES
    xs = np.arange(ABS_NB) * ABS_BIN_WORDS
    bottom = np.zeros(ABS_NB)
    for j, r in enumerate(ROLES):
        v = np.where(show, m[j], 0.0)
        ax.bar(xs, v, width=ABS_BIN_WORDS * 0.92, bottom=bottom, color=ROLE_COLORS[r], align="edge", label=r)
        bottom += v
    ax.set_title(name + " (bars drawn where >= %d traces contribute)" % ABS_MIN_TRACES, fontsize=9, color=COND_COLORS[name])
    ax.set_xlabel("absolute word position (bin start)")
    ax2 = ax.twinx()
    ax2.plot(xs + ABS_BIN_WORDS / 2, n_tr, color="black", lw=1.2, marker=".")
    ax2.set_ylabel("traces contributing", fontsize=8)
    ax2.grid(False)
    for b in range(ABS_NB):
        abs_rows.append(dict(condition=name, bin_start=b * ABS_BIN_WORDS, n_traces=int(n_tr[b]), drawn=bool(show[b]),
                             **{"share_" + r: m[j, b] for j, r in enumerate(ROLES)}))
axes[0, 0].legend(fontsize=7, ncol=5, loc="upper center", bbox_to_anchor=(1.0, 1.28))
axes[0, 0].set_ylabel("role proportion in the bin")
axes[1, 0].set_ylabel("role proportion in the bin")
fig.suptitle("B4 · Role proportion in fixed 100-word bins, with the number of contributing traces", fontsize=12, y=1.02)
fig.tight_layout()
save_fig(fig, "B4_absolute_position_roles.png")
plt.show()
plt.close(fig)
save_table(pd.DataFrame(abs_rows), "B4_absolute_profiles.csv", index=False)

### 8.5 When do doubt, verification and correction first start? (ECDFs)
One value per trace (position of the **first** run of that role); each trace is weighted `1 / (number of traces of its
question in that condition that have the role)`, so every question counts once. Traces without the role are excluded
from the curve and reported in the table.

In [ ]:
def q_weighted_values(ids, col):
    d = E.loc[ids, ["qkey", col]].dropna()
    w = 1.0 / d["qkey"].map(d["qkey"].value_counts())
    return d[col].to_numpy(), w.to_numpy()


ecdf_conds = [c for c in COND if len(COND[c])]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
cover = {}
for j, r in enumerate(KEY_ROLES):
    for i, (col, lab) in enumerate([("first_%s_rel" % r, "relative position of first %s (fraction of trace)" % r),
                                    ("first_%s_words" % r, "absolute position of first %s (words)" % r)]):
        ax = axes[i, j]
        for name in ecdf_conds:
            v, w = q_weighted_values(COND[name], col)
            if len(v) == 0:
                continue
            xv, cv = weighted_ecdf(v, w)
            ax.step(xv, cv, where="post", color=COND_COLORS[name], label=name, lw=1.6)
        if i == 1:
            ax.set_xlim(0, E["words"].quantile(0.99))
        ax.set_xlabel(lab, fontsize=8)
        ax.set_ylabel("question-weighted ECDF")
        ax.set_title(r, color=ROLE_COLORS[r], fontsize=11)
    for name in ecdf_conds:
        cover.setdefault(name, {})["has_" + r] = hier_mean(E.loc[COND[name]], ["has_" + r])["has_" + r].mean()
axes[0, 0].legend(fontsize=7)
fig.suptitle("B5 · ECDFs of the start of the first doubt / verification / correction phase", fontsize=12)
fig.tight_layout()
save_fig(fig, "B5_phase_start_ecdfs.png")
plt.show()
plt.close(fig)
print("share of traces that contain the role at all (mean over questions):")
display(pd.DataFrame(cover).T.round(3))

### 8.6 Answer emergence and stability
First candidate / final / stabilisation positions, observed changes (replacement vs refinement/format), words after
the answer value stabilised, gold match (light normalisation) and "gold candidate later replaced". `final` is an
annotation status, **not** correctness. Question-level means with cluster-bootstrap CIs.

In [ ]:
ans_feats = ["has_answer", "first_ans_rel", "first_cand_rel", "first_final_rel", "first_ans_end", "stab_rel",
             "words_after_stab", "share_after_stab", "segs_after_stab", "n_distinct_values", "n_replace", "n_refine",
             "any_replace", "cand_repeat_n", "gold_ever", "gold_lost"]
rows = []
for name in ecdf_conds:
    qdf = hier_mean(E.loc[COND[name]], ans_feats)
    for f in ans_feats:
        m, lo, hi, nq = boot_mean_ci(qdf[f])
        rows.append(dict(condition=name, feature=f, mean=m, ci_lo=lo, ci_hi=hi, n_questions=nq))
ans_tab = pd.DataFrame(rows)
display(ans_tab.pivot(index="feature", columns="condition", values="mean").reindex(ans_feats)[ecdf_conds].round(3))
save_table(ans_tab, "B6_answer_development_by_condition.csv", index=False)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, (col, lab) in zip(axes.ravel()[:3], [("first_ans_rel", "first answer (fraction of trace)"),
                                              ("stab_rel", "answer value stabilised (fraction of trace)"),
                                              ("words_after_stab", "words after the answer stabilised")]):
    for name in ecdf_conds:
        v, w = q_weighted_values(COND[name], col)
        if len(v):
            xv, cv = weighted_ecdf(v, w)
            ax.step(xv, cv, where="post", color=COND_COLORS[name], label=name, lw=1.6)
    if col == "words_after_stab":
        ax.set_xlim(0, E["words_after_stab"].quantile(0.99))
    ax.set_xlabel(lab, fontsize=8)
    ax.set_ylabel("question-weighted ECDF")
axes[0, 0].legend(fontsize=7)
ax = axes[1, 0]
PAT_COLS = ["pat_stable", "pat_replaced", "pat_goldlost", "pat_noanswer"]
pat = pd.DataFrame({name: hier_mean(E.loc[COND[name]], PAT_COLS).mean() for name in ecdf_conds}).T
pat.columns = ["stable", "answer replaced", "gold candidate later replaced", "no answer value"]
pat.plot(kind="barh", stacked=True, ax=ax, color=["#54a24b", "#f58518", "#d62728", "#bbbbbb"], width=0.7)
ax.set_xlabel("share of traces (mean over questions)")
ax.set_title("answer-development pattern", fontsize=9)
ax.legend(fontsize=6, loc="lower right")
ax = axes[1, 1]
sub = ans_tab[ans_tab["feature"].isin(["n_replace", "n_refine", "cand_repeat_n", "n_distinct_values"])]
piv = sub.pivot(index="feature", columns="condition", values="mean")[ecdf_conds]
piv.plot(kind="bar", ax=ax, color=[COND_COLORS[c] for c in ecdf_conds], width=0.8, legend=False)
ax.set_ylabel("mean per trace")
ax.set_title("answer changes and repeated candidates", fontsize=9)
ax.tick_params(axis="x", rotation=20)
ax = axes[1, 2]
sub = ans_tab[ans_tab["feature"].isin(["has_answer", "gold_ever", "gold_lost", "any_replace"])]
piv = sub.pivot(index="feature", columns="condition", values="mean")[ecdf_conds]
piv.plot(kind="bar", ax=ax, color=[COND_COLORS[c] for c in ecdf_conds], width=0.8, legend=False)
ax.set_ylabel("share of traces")
ax.set_title("coverage and gold match (gold_* are outcome-linked)", fontsize=9)
ax.tick_params(axis="x", rotation=20)
fig.suptitle("B6 · Answer emergence, change and stability, pooled over questions", fontsize=12)
fig.tight_layout()
save_fig(fig, "B6_answer_development.png")
plt.show()
plt.close(fig)

### 8.7 Phase transitions
Phases are **merged runs** of consecutive segments with the same role, so *role changes* (`n_runs - 1`) are kept apart
from *segment count*. Transition probabilities are computed per question (counts pooled over the question's traces
of that condition, rows normalised) and then averaged over questions with equal weight. `START`/`END` mark the
beginning and end of a trace. What precedes / follows each key phase, the outcome of each key phase, the position of
the transitions (absolute words and relative %), recurring role n-grams versus correctness, and pooled divergence
candidates follow.

In [ ]:
STATES_FROM, STATES_TO = ["START"] + ROLES, ROLES + ["END"]
e_in = R.assign(frm=R["prev_role"].fillna("START"), to=R["role"])[["trace_id", "frm", "to", "start_word", "rel_start"]]
e_out = R[R["next_role"].isna()].assign(frm=lambda d: d["role"], to="END", start_word=lambda d: d["end_word"],
                                        rel_start=1.0)[["trace_id", "frm", "to", "start_word", "rel_start"]]
EDGES = pd.concat([e_in, e_out], ignore_index=True)
EDGES["qkey"] = EDGES["trace_id"].map(F["qkey"])
EDGES = EDGES[EDGES["trace_id"].isin(E.index)]


def transition_rows(ids):
    """Per-question row-normalised transition probabilities: DataFrame indexed (qkey, from) with columns = to-states."""
    sel = EDGES[EDGES["trace_id"].isin(ids)]
    cnt = sel.groupby(["qkey", "frm", "to"]).size().unstack(fill_value=0).reindex(columns=STATES_TO, fill_value=0)
    return cnt.div(cnt.sum(axis=1), axis=0)


TP = {name: transition_rows(ids) for name, ids in COND.items() if len(ids)}
TM = {name: p.groupby(level="frm").mean().reindex(STATES_FROM) for name, p in TP.items()}
fig, axes = plt.subplots(1, len(TM), figsize=(4.3 * len(TM), 4.3))
for ax, (name, m) in zip(np.atleast_1d(axes), TM.items()):
    sns.heatmap(m, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar=False, ax=ax, annot_kws={"fontsize": 7})
    ax.set_title(name, fontsize=9, color=COND_COLORS[name])
    ax.set_xlabel("next phase")
    ax.set_ylabel("current phase")
fig.suptitle("B7a · Phase-transition probabilities (merged runs; mean over questions of per-question row-normalised matrices)", fontsize=11)
fig.tight_layout()
save_fig(fig, "B7a_transition_matrices.png")
plt.show()
plt.close(fig)
save_table(pd.concat(TM, names=["condition"]), "B7_transition_matrices.csv")


def paired_transition_diff(name_a, name_b, B=300):
    """Within-question paired difference of the transition matrices (a - b), bootstrap CI over questions."""
    pa, pb = TP[name_a], TP[name_b]
    idx = pa.index.intersection(pb.index)
    d = pa.loc[idx] - pb.loc[idx]
    mean = d.groupby(level="frm").mean().reindex(STATES_FROM)
    star = pd.DataFrame("", index=STATES_FROM, columns=STATES_TO)
    for frm in STATES_FROM:
        if frm not in d.index.get_level_values("frm"):
            continue
        sub = d.xs(frm, level="frm")
        for to in STATES_TO:
            m, lo, hi, nq = boot_mean_ci(sub[to], B=B)
            if nq >= 5 and (lo > 0 or hi < 0):
                star.at[frm, to] = "*"
    return mean, star


pairs_diff = [("ALL-CORRECT long", "ALL-CORRECT short"), ("MIXED incorrect", "MIXED correct")]
pairs_diff = [p for p in pairs_diff if p[0] in TP and p[1] in TP]
if pairs_diff:
    fig, axes = plt.subplots(1, len(pairs_diff), figsize=(6.2 * len(pairs_diff), 4.6))
    for ax, (a, b) in zip(np.atleast_1d(axes), pairs_diff):
        mean, star = paired_transition_diff(a, b)
        lab = mean.round(2).astype(str) + star
        sns.heatmap(mean.astype(float), annot=lab, fmt="", cmap="RdBu_r", center=0, vmin=-0.3, vmax=0.3, cbar=True, ax=ax,
                    annot_kws={"fontsize": 7})
        ax.set_title("%s minus %s\n(* = 95%% question-bootstrap CI excludes 0; exploratory)" % (a, b), fontsize=9)
        ax.set_xlabel("next phase")
        ax.set_ylabel("current phase")
    fig.tight_layout()
    save_fig(fig, "B7b_transition_differences.png")
    plt.show()
    plt.close(fig)

# segment-level persistence: P(next segment has the same role | role)
S["next_role_seg"] = S.groupby("trace_id")["role"].shift(-1)
pairs_seg = S[S["next_role_seg"].notna() & S["trace_id"].isin(E.index)]
stay = {}
for name, ids in COND.items():
    if not len(ids):
        continue
    sub = pairs_seg[pairs_seg["trace_id"].isin(ids)].assign(same=lambda d: (d["role"] == d["next_role_seg"]).astype(float))
    sub = sub.assign(qkey=sub["trace_id"].map(F["qkey"]))
    stay[name] = sub.groupby(["qkey", "role"])["same"].mean().groupby(level="role").mean().reindex(ROLES)
print("segment-level persistence P(next segment keeps the same role | role), mean over questions:")
display(pd.DataFrame(stay).round(3))

#### What precedes and what follows doubt, verification and correction? Positions of the transitions into them.

In [ ]:
def neighbour_dist(ids, role, side):
    sel = EDGES[EDGES["trace_id"].isin(ids)]
    if side == "before":
        sel, col, cats = sel[sel["to"] == role], "frm", STATES_FROM
    else:
        sel, col, cats = sel[sel["frm"] == role], "to", STATES_TO
    cnt = sel.groupby(["qkey", col]).size().unstack(fill_value=0).reindex(columns=cats, fill_value=0)
    return cnt.div(cnt.sum(axis=1), axis=0).mean()


NB_COLORS = dict(ROLE_COLORS, START="#e0e0e0", END="#222222")
fig, axes = plt.subplots(3, 2, figsize=(13, 9), sharex=True)
nb_rows = []
for i, role in enumerate(KEY_ROLES):
    for j, side in enumerate(["before", "after"]):
        ax = axes[i, j]
        tab = pd.DataFrame({name: neighbour_dist(ids, role, side) for name, ids in COND.items() if len(ids)}).T
        tab.plot(kind="barh", stacked=True, ax=ax, color=[NB_COLORS[c] for c in tab.columns], width=0.75, legend=False)
        ax.set_title("%s: phase %s it" % (role, "BEFORE" if side == "before" else "AFTER"), fontsize=9, color=ROLE_COLORS[role])
        ax.set_xlabel("share (mean over questions)" if i == 2 else "")
        for name, r in tab.iterrows():
            nb_rows.append(dict(role=role, side=side, condition=name, **r.to_dict()))
handles = [plt.Rectangle((0, 0), 1, 1, color=NB_COLORS[c]) for c in ["START"] + ROLES + ["END"]]
fig.legend(handles, ["START"] + ROLES + ["END"], ncol=7, loc="lower center", fontsize=8)
fig.suptitle("B7c · What precedes and follows doubt / verification / correction phases", fontsize=11)
fig.tight_layout(rect=(0, 0.04, 1, 0.97))
save_fig(fig, "B7c_phase_neighbours.png")
plt.show()
plt.close(fig)
save_table(pd.DataFrame(nb_rows), "B7_phase_neighbours.csv", index=False)

# positions of ALL runs of a key role (not only the first): absolute words and relative %
rows = []
for name, ids in COND.items():
    if not len(ids):
        continue
    sel = EDGES[EDGES["trace_id"].isin(ids) & EDGES["to"].isin(KEY_ROLES)]
    for role in KEY_ROLES:
        s_ = sel[sel["to"] == role]
        qm = s_.groupby("qkey").agg(rel=("rel_start", "median"), abs_=("start_word", "median"), n=("rel_start", "size"))
        rows.append(dict(condition=name, role=role, n_runs=len(s_), n_questions=len(qm),
                         rel_median_of_question_medians=qm["rel"].median(), rel_q25=qm["rel"].quantile(.25),
                         rel_q75=qm["rel"].quantile(.75), words_median_of_question_medians=qm["abs_"].median(),
                         words_q25=qm["abs_"].quantile(.25), words_q75=qm["abs_"].quantile(.75)))
pos_tab = pd.DataFrame(rows)
print("start position of every doubt / verification / correction run (question-level medians):")
display(pos_tab.round(3))
save_table(pos_tab, "B7_phase_start_positions.csv", index=False)

#### Outcome of each doubt / verification / correction phase
Mutually exclusive, in this priority order: **answer changes** (an observed answer *replacement* inside the phase or in the
first segment after it) -> **answer restated** (the phase or the next segment repeats the value that was already stated)
-> **trace ends** (last phase of the trace) -> **followed by another key phase** -> **returns to reasoning/setup**.
The outcome is a description of what happens next, not a claim that the phase caused it.

In [ ]:
VAL = {tid: (g["seg_pos"].to_numpy(), g["ans_norm"].to_numpy()) for tid, g in ev[ev["ans_norm"].notna()].groupby("trace_id")}
OUTCOMES = ["answer changes", "answer restated", "trace ends", "followed by another key phase", "returns to reasoning/setup"]
kr = R[R["role"].isin(KEY_ROLES) & R["trace_id"].isin(E.index)]
out_rows = []
for tid, role, fs, ls, nxt in zip(kr["trace_id"], kr["role"], kr["first_seg"], kr["last_seg"], kr["next_role"]):
    repl = A_lists.at[tid, "replace_segs"] if tid in A_lists.index else []
    win_end = min(ls + 1, F.at[tid, "n_seg"] - 1)
    if any(fs <= p <= win_end for p in repl):
        o = OUTCOMES[0]
    else:
        o = None
        if tid in VAL:
            pos_v, val_v = VAL[tid]
            before = [v for p, v in zip(pos_v, val_v) if p < fs]
            if before and any(v == before[-1] for p, v in zip(pos_v, val_v) if fs <= p <= win_end):
                o = OUTCOMES[1]
        if o is None:
            o = (OUTCOMES[2] if pd.isna(nxt) else OUTCOMES[3] if nxt in KEY_ROLES else OUTCOMES[4])
    out_rows.append((tid, role, o))
RO = pd.DataFrame(out_rows, columns=["trace_id", "role", "outcome"])
RO["qkey"] = RO["trace_id"].map(F["qkey"])
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
out_tabs = []
for ax, role in zip(axes, KEY_ROLES):
    tab = {}
    for name, ids in COND.items():
        if not len(ids):
            continue
        sub = RO[(RO["role"] == role) & RO["trace_id"].isin(ids)]
        cnt = sub.groupby(["qkey", "outcome"]).size().unstack(fill_value=0).reindex(columns=OUTCOMES, fill_value=0)
        tab[name] = cnt.div(cnt.sum(axis=1), axis=0).mean()
    tab = pd.DataFrame(tab).T
    tab.plot(kind="barh", stacked=True, ax=ax, color=["#d62728", "#e6a1a1", "#444444", "#f7b267", "#4c78a8"], width=0.75, legend=False)
    ax.set_title("outcome of a %s phase" % role, fontsize=9, color=ROLE_COLORS[role])
    ax.set_xlabel("share of phases (mean over questions)")
    out_tabs.append(tab.assign(role=role))
fig.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in ["#d62728", "#e6a1a1", "#444444", "#f7b267", "#4c78a8"]], OUTCOMES,
           ncol=5, loc="lower center", fontsize=8)
fig.tight_layout(rect=(0, 0.1, 1, 1))
save_fig(fig, "B7d_phase_outcomes.png")
plt.show()
plt.close(fig)
out_all = pd.concat(out_tabs)
save_table(out_all, "B7_phase_outcomes.csv")
display(out_all.round(3))

#### Recurring role n-grams versus correctness (MIXED cells, question-level aggregation)
n-grams over the sequence of merged phases (e.g. `doubt>verification>reasoning`). Per cell: rate per 100 words in the
incorrect samples minus rate in the correct samples; per question: mean over its cells; then `dz` with bootstrap CI.
Only n-grams that occur in at least `MIN_Q_NGRAM` questions are reported.

In [ ]:
mixed_E = E[E["cell_group"] == "MIXED"]
ng_rows = []
for n_ in [2, 3, 4]:
    cnt = ngram_counts(mixed_E.index, n_)
    rate = cnt.div(mixed_E.loc[cnt.index, "words"], axis=0) * 100
    ck, rk = mixed_E.loc[rate.index, "cell_id"].to_numpy(), mixed_E.loc[rate.index, "correct"].to_numpy()
    g = rate.groupby([ck, rk]).mean()
    wrong_side, right_side = g.xs(0.0, level=1), g.xs(1.0, level=1)
    diff = wrong_side - right_side.reindex(wrong_side.index)
    qd = diff.groupby(cells["qkey"].reindex(diff.index).to_numpy()).mean()
    support = (cnt > 0).groupby(mixed_E.loc[cnt.index, "qkey"].to_numpy()).any().sum()
    keep = support[support >= MIN_Q_NGRAM].index
    eff = paired_summary(qd[keep], B=500)
    eff["n"] = n_
    eff["questions_with_ngram"] = support.reindex(eff.index)
    eff["rate_incorrect"] = rate.loc[rk == 0, eff.index].mean()
    eff["rate_correct"] = rate.loc[rk == 1, eff.index].mean()
    ng_rows.append(eff)
ng_eff = pd.concat(ng_rows)
save_table(ng_eff, "B7_role_ngrams_mixed.csv")
top = pd.concat([ng_eff.sort_values("dz", ascending=False).head(8), ng_eff.sort_values("dz").head(8)])
display(top[["n", "questions_with_ngram", "rate_incorrect", "rate_correct", "mean", "dz", "dz_lo", "dz_hi"]].round(3))
fig, ax = plt.subplots(figsize=(8, 6))
t = top.sort_values("dz")
ax.hlines(range(len(t)), t["dz_lo"], t["dz_hi"], color="#555555")
ax.scatter(t["dz"], range(len(t)), color=["#d62728" if v > 0 else "#2ca02c" for v in t["dz"]], zorder=3)
ax.axvline(0, color="grey", lw=0.8)
ax.set_yticks(range(len(t)))
ax.set_yticklabels(t.index, fontsize=7)
ax.set_xlabel("dz of (incorrect - correct) rate per 100 words, question level")
ax.set_title("B7e · Role n-grams most associated with incorrect (red) / correct (green) samples\nwithin the same cell (exploratory)", fontsize=9)
fig.tight_layout()
save_fig(fig, "B7e_role_ngrams_mixed.png")
plt.show()
plt.close(fig)

#### Pooled divergence candidates
Positions of the algorithmic divergence candidates from 7.2, each question weighted once. *Reminder: candidates for manual
reading, no claim that the divergence caused the outcome.*

In [ ]:
dvq = div_df.assign(qkey=cells["qkey"].reindex(div_df["cell_id"]).to_numpy())
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (col_pairs, ttl) in zip(axes, [([("text_div_rel_a", "MIXED: incorrect trace", "wrong_vs_correct(length-matched)", "#d95f02"),
                                          ("text_div_rel_b", "MIXED: correct trace", "wrong_vs_correct(length-matched)", "#1b9e77"),
                                          ("text_div_rel_a", "ALL-CORRECT: long trace", "long_vs_short", "#08519c"),
                                          ("text_div_rel_b", "ALL-CORRECT: short trace", "long_vs_short", "#9ecae9")],
                                         "text divergence (first segment with no counterpart in the other trace)"),
                                        ([("phase_div_rel_a", "MIXED: incorrect trace", "wrong_vs_correct(length-matched)", "#d95f02"),
                                          ("phase_div_rel_b", "MIXED: correct trace", "wrong_vs_correct(length-matched)", "#1b9e77"),
                                          ("phase_div_rel_a", "ALL-CORRECT: long trace", "long_vs_short", "#08519c"),
                                          ("phase_div_rel_b", "ALL-CORRECT: short trace", "long_vs_short", "#9ecae9")],
                                         "phase divergence (first merged run whose role differs)")]):
    for col, lab, pair, colr in col_pairs:
        d = dvq[(dvq["pair"] == pair)][["qkey", col]].dropna()
        d[col] = d[col].astype(float)
        if len(d) >= 3:
            w = 1.0 / d["qkey"].map(d["qkey"].value_counts())
            xv, cv = weighted_ecdf(d[col].to_numpy(), w.to_numpy())
            ax.step(xv, cv, where="post", color=colr, label="%s (n=%d)" % (lab, len(d)))
    ax.set_xlabel("relative position of the divergence candidate in its own trace")
    ax.set_ylabel("question-weighted ECDF")
    ax.set_title(ttl, fontsize=9)
    ax.legend(fontsize=7)
fig.suptitle("B7f · Where the traces of a compared pair diverge (algorithmic candidates)", fontsize=11)
fig.tight_layout()
save_fig(fig, "B7f_divergence_positions.png")
plt.show()
plt.close(fig)
mx_dv = dvq[dvq["pair"].str.startswith("wrong")]
print("MIXED pairs: text divergence found in the incorrect trace %.0f%%, in the correct trace %.0f%%; incorrect-trace divergence "
      "lies before its first answer in %.0f%% (pairs where both exist)" % (
          100 * mx_dv["text_div_words_a"].notna().mean(), 100 * mx_dv["text_div_words_b"].notna().mean(),
          100 * mx_dv["precedes_first_ans_a"].mean()))

### 8.8 Universal summary: when traces get longer, when they go wrong, when they are right
Within-cell **length rank**: in MIXED cells every sample gets its length percentile inside its own cell (0 = shortest,
1 = longest). If wrong traces simply were longer, accuracy would fall with that percentile. The check compares the
outcome of the *longest* / *shortest* sample of each cell with what is expected from the cell's own accuracy
(`n_wrong / n`), per question, so cells with a different accuracy are not mixed up.

In [ ]:
m = E[E["cell_group"] == "MIXED"].copy()
m["n_in_cell"] = m.groupby("cell_id")["words"].transform("size")
m["len_pct"] = (m.groupby("cell_id")["words"].rank(method="average") - 0.5) / m["n_in_cell"]
m["len_bin"] = pd.cut(m["len_pct"], [0, 0.2, 0.4, 0.6, 0.8, 1.0], labels=False, include_lowest=True)
rank_rows = []
for b in range(5):
    q = hier_mean(m[m["len_bin"] == b], ["correct"])["correct"]
    mean_, lo, hi, nq = boot_mean_ci(q)
    rank_rows.append(dict(len_quintile_in_cell=b + 1, p_correct=mean_, ci_lo=lo, ci_hi=hi, n_questions=nq))
rank_tab = pd.DataFrame(rank_rows)
base_acc = hier_mean(m, ["correct"])["correct"].mean()
cell_ext = (m.sort_values(["cell_id", "words", "sample_index"]).groupby("cell_id")
            .agg(qkey=("qkey", "first"), shortest_wrong=("correct", lambda s: 1.0 - s.iloc[0]),
                 longest_wrong=("correct", lambda s: 1.0 - s.iloc[-1]), exp_wrong=("correct", lambda s: 1.0 - s.mean())))
cell_ext["longest_wrong_minus_expected"] = cell_ext["longest_wrong"] - cell_ext["exp_wrong"]
cell_ext["shortest_wrong_minus_expected"] = cell_ext["shortest_wrong"] - cell_ext["exp_wrong"]
ext_q = cell_ext.groupby("qkey")[["longest_wrong_minus_expected", "shortest_wrong_minus_expected"]].mean()
ext_eff = paired_summary(ext_q)
display(rank_tab.round(3))
print("overall P(correct) in MIXED cells (question-level): %.3f" % base_acc)
print("P(the longest sample of a MIXED cell is wrong) minus its expected value:")
display(ext_eff[["n_questions", "mean", "ci_lo", "ci_hi", "p_signflip"]].round(3))
save_table(rank_tab, "B8_within_cell_length_rank_accuracy.csv", index=False)
save_table(ext_eff, "B8_longest_shortest_wrong_vs_expected.csv")

fig = plt.figure(figsize=(16, 9.5))
gs = fig.add_gridspec(2, 2, hspace=0.38, wspace=0.28)
ax = fig.add_subplot(gs[0, 0])
labs = ROLES + ["BEFORE\nfirst answer", "AFTER\nfirst answer"]
vals = [eff_ac.loc["xshare_" + r] for r in ROLES] + [eff_ac.loc["xshare_before_ans"], eff_ac.loc["xshare_after_ans"]]
ax.bar(range(len(vals)), [v["mean"] for v in vals], color=[ROLE_COLORS[r] for r in ROLES] + ["#444444", "#999999"],
       yerr=[[v["mean"] - v["ci_lo"] for v in vals], [v["ci_hi"] - v["mean"] for v in vals]], capsize=3)
if obs_null is not None:
    for j in range(len(ROLES)):
        ax.hlines(null_mat[:, j].mean(), j - 0.4, j + 0.4, color="black", lw=2, ls="--")
    ax.hlines(null_mat[:, -1].mean(), 5 - 0.4, 5 + 0.4, color="black", lw=2, ls="--")
ax.set_xticks(range(len(vals)))
ax.set_xticklabels(labs, fontsize=8)
ax.set_ylabel("share of the extra words")
ax.set_title("WHEN TRACES GET LONGER: where the extra words of the longest correct\ntrace sit "
             "(dashed = expected if composition were unrelated to length; n=%d questions)" % len(ac_q), fontsize=9)
ax = fig.add_subplot(gs[0, 1])
cand = eff_mx.drop(index=[f for f in eff_mx.index if f in LEN_DEP or f in OUTCOME_LINKED or f == "incorrect_longer"])
t = cand.reindex(cand["dz"].abs().sort_values(ascending=False).index).head(10).sort_values("dz")
ax.hlines(range(len(t)), t["dz_lo"], t["dz_hi"], color="#555555")
ax.scatter(t["dz"], range(len(t)), color=["#d62728" if v > 0 else "#2ca02c" for v in t["dz"]], zorder=3)
ax.axvline(0, color="grey", lw=0.8)
ax.set_yticks(range(len(t)))
ax.set_yticklabels(t.index, fontsize=8)
ax.set_xlabel("dz, incorrect minus correct (same cell)")
ax.set_title("WHEN TRACES GO WRONG: ten largest length-free within-cell differences\n(red: higher in incorrect samples; n=%d questions)" % len(mx_q), fontsize=9)
sub = gs[1, 0].subgridspec(3, 1, hspace=0.15)
for k, name in enumerate(["ALL-CORRECT short", "ALL-CORRECT long", "MIXED incorrect"]):
    axk = fig.add_subplot(sub[k])
    if name in prof:
        mm = np.nanmean(prof[name], axis=0).reshape(5, REL_BINS)
        axk.stackplot(xr, mm, colors=[ROLE_COLORS[r] for r in ROLES], labels=ROLES)
    axk.set_xlim(xr[0], xr[-1]); axk.set_ylim(0, 1)
    axk.set_ylabel(name.replace("ALL-CORRECT ", "correct ").replace("MIXED ", ""), fontsize=7, color=COND_COLORS[name])
    axk.set_yticks([])
    if k < 2:
        axk.set_xticklabels([])
    if k == 0:
        axk.set_title("COMPOSITION ALONG THE TRACE (relative position, role share)", fontsize=9)
        axk.legend(fontsize=6, ncol=5, loc="upper center", bbox_to_anchor=(0.5, 1.9))
axk.set_xlabel("relative position (%)")
ax = fig.add_subplot(gs[1, 1])
ax.errorbar(rank_tab["len_quintile_in_cell"], rank_tab["p_correct"],
            yerr=[rank_tab["p_correct"] - rank_tab["ci_lo"], rank_tab["ci_hi"] - rank_tab["p_correct"]], fmt="o-", capsize=3, color="#1b6ca8")
ax.axhline(base_acc, color="grey", ls="--", label="mean over MIXED cells")
ax.set_xlabel("length quintile inside the cell (1 = shortest samples, 5 = longest)")
ax.set_ylabel("P(correct)")
ax.set_xticks(range(1, 6))
l_ = ext_eff.loc["longest_wrong_minus_expected"]
ax.set_title("WHEN TRACES ARE RIGHT: accuracy by within-cell length rank\n(longest sample wrong minus expected: %+.3f, 95%% CI %.3f to %.3f)"
             % (l_["mean"], l_["ci_lo"], l_["ci_hi"]), fontsize=9)
ax.legend(fontsize=8)
fig.suptitle("B8 · Universal pattern, pooled over questions (within-cell contrasts; observational, exploratory)", fontsize=12)
save_fig(fig, "B8_universal_summary.png")
plt.show()
plt.close(fig)

## 9. Part C: compute-level analysis
Key comparisons repeated per compute level: length distribution, accuracy, role shares, phase structure, answer
development, and the long-vs-short and incorrect-vs-correct effects. Cells of one question at different compute levels
are **not independent**, so differences between levels are descriptive; paired-by-question contrasts between levels
are given where the same questions appear at both levels. Eligible (non-truncated) samples only unless stated.

In [ ]:
LV = [c for c in COMPUTE_ORDER if (E["compute_level"] == c).any()]
lv_rows = []
for lv in LV:
    e = E[E["compute_level"] == lv]
    allr = tr[(~tr["dup_key"]) & tr["labels_ok"] & (tr["compute_level"] == lv)]
    qacc = hier_mean(e, ["correct"])["correct"]
    cg = cells[cells["compute_level"] == lv]["group"].value_counts()
    lv_rows.append(dict(compute_level=lv, eligible_traces=len(e), questions=e["qkey"].nunique(),
                        cells_ALL_CORRECT=cg.get("ALL-CORRECT", 0), cells_MIXED=cg.get("MIXED", 0),
                        cells_ALL_WRONG=cg.get("ALL-WRONG", 0), words_median=e["words"].median(),
                        words_q25=e["words"].quantile(.25), words_q75=e["words"].quantile(.75),
                        accuracy_question_level=qacc.mean(), acc_lo=boot_mean_ci(qacc)[1], acc_hi=boot_mean_ci(qacc)[2],
                        truncation_rate=float((allr["trunc"] == 1).mean()), empty_output_rate=float((e["empty_out"] == 1).mean())))
lv_tab = pd.DataFrame(lv_rows).set_index("compute_level")
display(lv_tab.round(3))
save_table(lv_tab, "C_compute_level_overview.csv")

# paired-by-question change in log length and accuracy relative to the first level
base_lv = LV[0]
qm = {lv: hier_mean(E[E["compute_level"] == lv], ["log_words", "correct"]) for lv in LV}
rows = {}
for lv in LV[1:]:
    idx = qm[lv].index.intersection(qm[base_lv].index)
    rows[lv] = pd.DataFrame({"d_log_words": qm[lv].loc[idx, "log_words"] - qm[base_lv].loc[idx, "log_words"],
                             "d_accuracy": qm[lv].loc[idx, "correct"] - qm[base_lv].loc[idx, "correct"]})
if rows:
    tab = pd.concat({lv: paired_summary(r) for lv, r in rows.items()}, names=["compute_level"])
    print("paired by question, relative to '%s':" % base_lv)
    display(tab[["n_questions", "mean", "ci_lo", "ci_hi", "dz"]].round(3))
    save_table(tab, "C_levels_vs_first_level_paired.csv")

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
ax = axes[0, 0]
dat = [E.loc[(E["compute_level"] == lv), "words"].to_numpy() for lv in LV]
ax.boxplot(dat, tick_labels=LV, showfliers=False)
ax.set_yscale("log")
ax.set_ylabel("words (log scale)")
ax.set_title("length distribution (eligible traces)", fontsize=9)
ax = axes[0, 1]
ax.errorbar(range(len(LV)), lv_tab["accuracy_question_level"], yerr=[lv_tab["accuracy_question_level"] - lv_tab["acc_lo"],
                                                                      lv_tab["acc_hi"] - lv_tab["accuracy_question_level"]], fmt="o-", capsize=3)
ax.set_xticks(range(len(LV)))
ax.set_xticklabels(LV)
ax.set_ylabel("accuracy (question-level mean, eligible traces)")
ax.set_title("accuracy by compute level (95% question-bootstrap CI)", fontsize=9)
ax = axes[0, 2]
grp_share = pd.DataFrame({lv: cells[cells["compute_level"] == lv]["group"].value_counts(normalize=True) for lv in LV}).T
grp_share = grp_share.reindex(columns=[g for g in GROUPS if g in grp_share.columns]).fillna(0)
grp_share.plot(kind="bar", stacked=True, ax=ax, color=["#08519c", "#9ecae9", "#f58518", "#d62728", "#cccccc"][:grp_share.shape[1]], width=0.7)
ax.set_ylabel("share of cells")
ax.set_title("cell groups by compute level", fontsize=9)
ax.tick_params(axis="x", rotation=0)
ax = axes[1, 0]
shares = pd.DataFrame({lv: hier_mean(E[E["compute_level"] == lv], ["share_" + r for r in ROLES]).mean() for lv in LV}).T
shares.columns = ROLES
shares.plot(kind="bar", stacked=True, ax=ax, color=[ROLE_COLORS[r] for r in ROLES], width=0.7)
ax.set_ylabel("share of words (mean over questions)")
ax.set_title("role shares", fontsize=9)
ax.tick_params(axis="x", rotation=0)
ax.legend(fontsize=6, ncol=5, loc="lower center")
ax = axes[1, 1]
phase_feats = ["changes_per100w", "runs100_doubt", "runs100_verification", "runs100_correction", "n_replace_100w", "any_replace",
               "first_ans_rel", "share_after_stab"]
ph = pd.DataFrame({lv: hier_mean(E[E["compute_level"] == lv], phase_feats).mean() for lv in LV}).T
ph_n = (ph - ph.min()) / (ph.max() - ph.min()).replace(0, 1)
sns.heatmap(ph_n.T, annot=ph.T.round(2), fmt="", cmap="Greys", cbar=False, ax=ax, annot_kws={"fontsize": 7})
ax.set_title("phase structure and answer development (cell colour: scaled within row)", fontsize=9)
ax = axes[1, 2]
w_ = pd.DataFrame({lv: E[(E["compute_level"] == lv)].groupby("correct")["words"].median() for lv in LV}).T
w_.columns = ["incorrect" if c == 0 else "correct" for c in w_.columns]
w_.plot(kind="bar", ax=ax, color=["#d62728", "#2ca02c"], width=0.7)
ax.set_ylabel("median words")
ax.set_title("median length by correctness (eligible traces)", fontsize=9)
ax.tick_params(axis="x", rotation=0)
fig.suptitle("C1 · Compute-level overview", fontsize=12)
fig.tight_layout()
save_fig(fig, "C1_compute_overview.png")
plt.show()
plt.close(fig)
save_table(shares, "C_role_shares_by_level.csv")
save_table(ph, "C_phase_and_answer_by_level.csv")

### 9.1 Effects per compute level
The same within-cell contrasts as in 8.1 / 8.2, restricted to one compute level at a time (each question then
contributes at most one cell per level). Heatmap: `dz` of incorrect minus correct (MIXED); `*` = 95% question-bootstrap CI
excludes 0. Right panels: long-vs-short, share of the extra words by role. "Persists" = same sign at all levels with
at least 2 levels excluding 0; "changes" = opposite signs with the CI excluding 0 at two levels; otherwise "inconclusive".

In [ ]:
EFFECT_FEATS = ["share_doubt", "share_verification", "share_correction", "share_reasoning", "share_setup",
                "changes_per100w", "runs100_doubt", "runs100_verification", "runs100_correction", "doubt_to_corr100",
                "first_ans_rel", "share_after_first_ans", "n_replace_100w", "any_replace", "stab_rel", "rep4_share",
                "near_repeat_wshare", "sent_repeat_share", "hedge_100w", "novelty", "comp_ratio", "log_words"]
eff_lv = {}
for lv in LV:
    t_ = mx_tab[mx_tab["compute_level"] == lv]
    if t_["qkey"].nunique() >= 5:
        eff_lv[lv] = paired_summary(to_question_level(t_, EFFECT_FEATS), B=1000)
if eff_lv:
    dz_tab = pd.DataFrame({lv: e["dz"] for lv, e in eff_lv.items()}).reindex(EFFECT_FEATS)
    sig_tab = pd.DataFrame({lv: (e["dz_lo"] > 0) | (e["dz_hi"] < 0) for lv, e in eff_lv.items()}).reindex(EFFECT_FEATS)
    nq_tab = {lv: int(e["n_questions"].max()) for lv, e in eff_lv.items()}

    def verdict(f):
        v, s_ = dz_tab.loc[f].dropna(), sig_tab.loc[f].reindex(dz_tab.loc[f].dropna().index)
        if len(v) < 2:
            return "n/a"
        if (np.sign(v) == np.sign(v.iloc[0])).all() and s_.sum() >= 2:
            return "persists"
        if s_.sum() >= 2 and (np.sign(v[s_]) != np.sign(v[s_].iloc[0])).any():
            return "changes"
        return "inconclusive"

    persist = pd.DataFrame({"verdict": [verdict(f) for f in EFFECT_FEATS]}, index=EFFECT_FEATS).join(dz_tab.round(2))
    display(persist)
    save_table(persist, "C_mixed_effects_persistence.csv")
    ComputeAC = [lv for lv in LV if ac_tab[ac_tab["compute_level"] == lv]["qkey"].nunique() >= 3]
    fig, axes = plt.subplots(1, 2, figsize=(15, 8), gridspec_kw=dict(width_ratios=[1.3, 1]))
    ax = axes[0]
    annot = dz_tab.round(2).astype(str) + sig_tab.map(lambda b: "*" if b else "")
    sns.heatmap(dz_tab.astype(float), annot=annot.where(dz_tab.notna(), ""), fmt="", cmap="RdBu_r", center=0, vmin=-1.5, vmax=1.5,
                ax=ax, annot_kws={"fontsize": 7}, cbar_kws={"label": "dz (incorrect - correct)"})
    ax.set_xticklabels(["%s\n(n=%d q)" % (lv, nq_tab[lv]) for lv in dz_tab.columns], fontsize=8)
    ax.set_title("MIXED cells: incorrect minus correct by compute level (* CI excludes 0)", fontsize=9)
    ax = axes[1]
    xs_rows = {}
    for lv in ComputeAC:
        q_ = to_question_level(ac_tab[ac_tab["compute_level"] == lv], ["xshare_" + r for r in ROLES] + ["xshare_before_ans"])
        xs_rows[lv] = q_.mean()
    if xs_rows:
        xs_df = pd.DataFrame(xs_rows).T
        xs_df.columns = ROLES + ["before 1st answer"]
        xs_df[ROLES].plot(kind="bar", stacked=True, ax=ax, color=[ROLE_COLORS[r] for r in ROLES], width=0.6)
        ax.set_ylabel("share of the extra words (long - short)")
        ax.set_xticklabels(["%s\n(n=%d q)" % (lv, ac_tab[ac_tab['compute_level'] == lv]['qkey'].nunique()) for lv in xs_df.index], rotation=0)
        ax.set_title("ALL-CORRECT: extra words by role, per compute level\n(small n: descriptive only)", fontsize=9)
        save_table(xs_df, "C_allcorrect_extra_words_by_level.csv")
    else:
        ax.text(0.5, 0.5, "fewer than 3 ALL-CORRECT questions per level", ha="center")
    fig.suptitle("C2 · Do the within-cell patterns persist across compute levels?", fontsize=12)
    fig.tight_layout()
    save_fig(fig, "C2_effects_by_compute_level.png")
    plt.show()
    plt.close(fig)
else:
    print("fewer than 5 MIXED questions at every compute level; per-level effects not computed")

## 10. Textual analyses (lightweight, interpretable)
All text measures are **cues and similarity scores**, not labels. TF-IDF cosine uses one global vocabulary fitted on all
segments; "counterpart" means cosine >= `CP_SIM` (0.30), "near repeat" means cosine >= `REP_SIM` (0.60) to an earlier
segment. Question-level aggregation and cluster-bootstrap CIs as everywhere else.

### 10.1 Repetition vs new content
(a) Summary effects (long - short in ALL-CORRECT, incorrect - correct in MIXED) for the repetition/novelty features.
(b) **Counterpart analysis**: every segment of the longer trace (ALL-CORRECT) / of each member of the length-matched
pair (MIXED) is classified as *has a counterpart in the other trace*, *repeats earlier content of its own trace*, *no
counterpart (new relative to the other trace)* or *uninformative (no content words)*.

In [ ]:
REP_FEATS = ["novelty", "rep4_share", "near_repeat_wshare", "rep_sim_mean", "num_repeat_share", "sent_repeat_share",
             "eq_repeat_share", "comp_ratio", "loop_flag"]
t1 = pd.DataFrame({"ALL-CORRECT long-short": eff_ac.reindex(["d_" + f for f in REP_FEATS])["mean"].to_numpy(),
                   "  CI lo": eff_ac.reindex(["d_" + f for f in REP_FEATS])["ci_lo"].to_numpy(),
                   "  CI hi": eff_ac.reindex(["d_" + f for f in REP_FEATS])["ci_hi"].to_numpy(),
                   "MIXED incorrect-correct": eff_mx.reindex(REP_FEATS)["mean"].to_numpy(),
                   "  dz": eff_mx.reindex(REP_FEATS)["dz"].to_numpy(),
                   "  dz CI lo": eff_mx.reindex(REP_FEATS)["dz_lo"].to_numpy(),
                   "  dz CI hi": eff_mx.reindex(REP_FEATS)["dz_hi"].to_numpy()}, index=REP_FEATS)
display(t1.round(3))
print("loop_flag is a coarse threshold (sentence-repeat >= %.2f or repeated-4-gram >= %.2f or a sentence repeated >= %d times); "
      "prevalence in eligible traces: %.0f%%. The continuous measures are the primary ones." % (
          LOOP_SENT, LOOP_REP4, LOOP_MAXREP, 100 * E["loop_flag"].mean()))
save_table(t1, "T1_repetition_effects.csv")


def counterpart_categories(ta, tb):
    """Classify each segment of trace ta relative to trace tb (TF-IDF cosine)."""
    ia, ib = seg_ix[ta], seg_ix[tb]
    Xa, Xb = X[ia[0]:ia[-1] + 1], X[ib[0]:ib[-1] + 1]
    best = np.asarray((Xa @ Xb.T).max(axis=1).todense()).ravel()
    informative = np.asarray(Xa.getnnz(axis=1)) > 0
    own = np.fmax(S["sim_prev"].to_numpy()[ia], S["sim_nonadj_max"].to_numpy()[ia])     # best similarity to earlier own text
    cat = np.select([~informative, best >= CP_SIM, own >= REP_SIM],
                    ["uninformative", "has counterpart in other trace", "repeats earlier own content"],
                    default="no counterpart (new vs other trace)")
    fa = F.at[ta, "first_ans_seg"]
    pos_arr = S["seg_pos"].to_numpy()[ia]
    phase = np.full(len(ia), "no answer annotation") if fa != fa else np.where(pos_arr <= fa, "up to first answer", "after first answer")
    return pd.DataFrame({"trace_id": ta, "seg_pos": pos_arr, "role": S_role[ia], "words": S_words[ia], "cat": cat, "phase": phase})


CATS = ["has counterpart in other trace", "repeats earlier own content", "no counterpart (new vs other trace)", "uninformative"]
cp_parts = []
for _, r in ac_pairs.assign(cell_id=ac_pairs.index).iterrows():
    cp_parts.append(counterpart_categories(r["long_id"], r["short_id"]).assign(set="ALL-CORRECT: long trace vs short", cell_id=r["cell_id"]))
    cp_parts.append(counterpart_categories(r["short_id"], r["long_id"]).assign(set="ALL-CORRECT: short trace vs long", cell_id=r["cell_id"]))
for _, r in div_df[div_df["pair"].str.startswith("wrong")].iterrows():
    cp_parts.append(counterpart_categories(r["id_a"], r["id_b"]).assign(set="MIXED: incorrect trace vs correct", cell_id=r["cell_id"]))
    cp_parts.append(counterpart_categories(r["id_b"], r["id_a"]).assign(set="MIXED: correct trace vs incorrect", cell_id=r["cell_id"]))
CP = pd.concat(cp_parts, ignore_index=True)
CP["qkey"] = CP["cell_id"].map(cells["qkey"])
save_table(CP, "T1_counterpart_segments.csv", index=False)


def cat_shares(by=None):
    keys = ["set", "qkey", "cell_id"] + ([by] if by else [])
    w = CP.groupby(keys + ["cat"])["words"].sum().unstack(fill_value=0).reindex(columns=CATS, fill_value=0)
    w = w.div(w.sum(axis=1).replace(0, np.nan), axis=0)                # word-weighted shares inside each pair side
    cq = w.groupby(keys[:-1] if not by else ["set", "qkey"] + [by]).mean()  # cell -> question
    return cq.groupby(level=[0] + ([2] if by else [])).mean()


cs = cat_shares()
print("share of each trace's words by counterpart category (word-weighted inside a trace, mean over cells, then questions):")
display(cs.round(3))
save_table(cs, "T1_counterpart_shares.csv")
long_cp = CP[CP["set"] == "ALL-CORRECT: long trace vs short"]
if len(long_cp):
    by_role = long_cp.groupby(["role", "cat"])["words"].sum().unstack(fill_value=0).reindex(columns=CATS, fill_value=0)
    print("\nlong correct trace: category shares of the words of each role (pooled words, descriptive):")
    display(by_role.div(by_role.sum(axis=1), axis=0).round(3))
    by_ph = long_cp.groupby(["phase", "cat"])["words"].sum().unstack(fill_value=0).reindex(columns=CATS, fill_value=0)
    print("long correct trace: category shares before vs after the first answer (pooled words, descriptive):")
    display(by_ph.div(by_ph.sum(axis=1), axis=0).round(3))
fig, ax = plt.subplots(figsize=(10, 3.8))
cs.plot(kind="barh", stacked=True, ax=ax, color=["#4c78a8", "#e6a1a1", "#54a24b", "#cccccc"], width=0.7)
ax.set_xlabel("share of the trace's words")
ax.set_title("T1 · Repetition vs new content: each trace compared with its partner trace in the same cell", fontsize=10)
ax.legend(fontsize=7, loc="lower center", bbox_to_anchor=(0.5, -0.45), ncol=4)
fig.tight_layout()
save_fig(fig, "T1_counterpart_categories.png")
plt.show()
plt.close(fig)

### 10.2 Lexical cues (hedges / restart phrases)
Cues per 100 words: `wait, hmm, maybe, actually, let me check, alternatively` (summed as *hedge*) and restart phrases. Where do they occur
(role, relative position, before/after the first answer), and do they differ long vs short and incorrect vs correct
*within the cell*? The role table doubles as a **sanity check of the discourse_role labels**: do doubt segments really
carry more hedges than reasoning segments?

In [ ]:
Se = S[S["trace_id"].isin(E.index)].assign(qkey=lambda d: d["trace_id"].map(F["qkey"]))
cue_cols = ["cue_" + k for k in CUES] + ["cue_hedge"]
g = Se.groupby(["qkey", "role"])[cue_cols + ["words"]].sum()
rate_role = (g[cue_cols].div(g["words"], axis=0) * 100).groupby(level="role").mean().reindex(ROLES)
rate_role.columns = [c.replace("cue_", "") for c in rate_role.columns]
print("cue rate per 100 words by annotated role (mean over questions):")
display(rate_role.round(3))
save_table(rate_role, "T2_cues_by_role.csv")

seg_bin = np.minimum((S["rel_mid"].to_numpy() * REL_BINS).astype(int), REL_BINS - 1)


def bin_ratio(num, den):
    d = pd.DataFrame({"t": S["trace_id"].to_numpy(), "b": seg_bin, "n": np.asarray(num, float), "d": np.asarray(den, float)})
    gg = d.groupby(["t", "b"])[["n", "d"]].sum()
    r = (gg["n"] / gg["d"].replace(0, np.nan)).unstack("b").reindex(columns=range(REL_BINS))
    arr = np.full((nF, REL_BINS), np.nan)
    arr[F.index.get_indexer(r.index)] = r.to_numpy()
    return arr


BIN_HEDGE = bin_ratio(S["cue_hedge"], S["words"]) * 100
BIN_REPEAT4 = bin_ratio(S["n4_rep"], S["n4"])
BIN_NOVEL = bin_ratio(S["n_new"], S["n_content"])
BIN_COMP = bin_ratio(S["inc_comp"].fillna(0).to_numpy(), S["inc_comp"].notna().to_numpy().astype(float))

# before / after the first answer
fa = S["trace_id"].map(F["first_ans_seg"])
S["phase_ans"] = np.where(fa.isna(), "no answer", np.where(S["seg_pos"] <= fa, "up to first answer", "after first answer"))
ph = (S[S["trace_id"].isin(E.index) & (S["phase_ans"] != "no answer")].groupby(["trace_id", "phase_ans"])[["cue_hedge", "words"]].sum())
ph["rate"] = ph["cue_hedge"] / ph["words"].replace(0, np.nan) * 100
ph_arr = ph["rate"].unstack("phase_ans").reindex(F.index)
rows = []
for name, ids in COND.items():
    if not len(ids):
        continue
    for col in ["up to first answer", "after first answer"]:
        arr_ = np.full((nF, 1), np.nan)
        arr_[:, 0] = ph_arr[col].to_numpy()
        qv = hier_q_array(arr_, ids)[:, 0]
        m_, lo, hi, nq = boot_mean_ci(qv)
        rows.append(dict(condition=name, segment_part=col, hedge_per100w=m_, ci_lo=lo, ci_hi=hi, n_questions=nq))
ph_tab = pd.DataFrame(rows)
display(ph_tab.pivot(index="condition", columns="segment_part", values="hedge_per100w").round(3))
save_table(ph_tab, "T2_hedge_before_after_first_answer.csv", index=False)
cue_eff = pd.concat({"long-short (ALL-CORRECT)": eff_ac.reindex(["d_cue_%s_100w" % k for k in CUES] + ["d_hedge_100w"]).rename(index=lambda s: s[2:]),
                     "incorrect-correct (MIXED)": eff_mx.reindex(["cue_%s_100w" % k for k in CUES] + ["hedge_100w"])})
print("within-cell cue differences (per 100 words):")
display(cue_eff[["n_questions", "mean", "ci_lo", "ci_hi", "dz"]].round(3))
save_table(cue_eff, "T2_cue_effects.csv")

### 10.3 Extra textual profiles along the trace (additional analyses)
Hedge rate, share of repeated 4-grams, content-word novelty and **incremental compression** (extra compressed bytes a
segment adds to a `zlib` stream of everything before it; low = predictable/repetitive; 32 KB window; segments >= 80 bytes)
by relative position, per condition (question-level mean, 95% band). Compression is a cheap, model-free repetition proxy.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 4.2))
for ax, (arr, ttl) in zip(axes, [(BIN_HEDGE, "hedge cues per 100 words"), (BIN_REPEAT4, "share of repeated 4-grams"),
                                 (BIN_NOVEL, "content-word novelty"), (BIN_COMP, "incremental compression ratio")]):
    for name, ids in COND.items():
        if len(ids):
            qa = hier_q_array(arr, ids)
            mean, lo, hi = boot_matrix_ci(qa, B=200)
            ax.plot(xr, mean, color=COND_COLORS[name], label=name, marker="o", ms=3)
            ax.fill_between(xr, lo, hi, color=COND_COLORS[name], alpha=0.12)
    ax.set_title(ttl, fontsize=9)
    ax.set_xlabel("relative position (%)")
axes[0].legend(fontsize=6.5)
fig.suptitle("T3 · Text profiles along the trace", fontsize=11)
fig.tight_layout()
save_fig(fig, "T3_text_profiles.png")
plt.show()
plt.close(fig)
rows = []
for name, arr in [("hedge_per100w", BIN_HEDGE), ("repeat4_share", BIN_REPEAT4), ("novelty", BIN_NOVEL), ("incremental_compression", BIN_COMP)]:
    for cname, ids in COND.items():
        if len(ids):
            mean, lo, hi = boot_matrix_ci(hier_q_array(arr, ids), B=200)
            rows += [dict(measure=name, condition=cname, decile=b + 1, mean=mean[b], ci_lo=lo[b], ci_hi=hi[b]) for b in range(REL_BINS)]
save_table(pd.DataFrame(rows), "T3_text_profiles.csv", index=False)

### 10.4 Loops: repeated sentences, equations, numbers and answer candidates

In [ ]:
loop_feats = ["sent_repeat_share", "eq_repeat_share", "num_repeat_share", "cand_repeat_n", "max_sent_repeat", "loop_flag"]
loop_tab = pd.DataFrame({name: hier_mean(E.loc[ids], loop_feats).mean() for name, ids in COND.items() if len(ids)}).T
display(loop_tab.round(3))
save_table(loop_tab, "T4_loops_by_condition.csv")
print("most repeated sentences (inspection list; condition = cell group of the trace):")
ex_loop = E.sort_values("max_sent_repeat", ascending=False).head(5)
for tid, r in ex_loop.iterrows():
    print("  [%s | %s | correct=%s] repeated %dx: %s" % (r["cell_id"], r["cell_group"], r["correct"], r["max_sent_repeat"], r["top_sentence"][:140]))

### 10.5 Post-answer behaviour: what do the segments after the first `final` do?
Each segment after the first `final` segment is assigned (in this priority order): **new different answer** (its answer value
differs from the final value), **restates the answer** (same value annotated, or the value occurs in the text), **verifies**,
**doubts**, **corrects**, or **continues reasoning**.

In [ ]:
def mentions(value, text):
    return value is not None and re.search(r"(?<![\w.])" + re.escape(value) + r"(?!\w|\.\d)", text.lower().replace(" ", "")) is not None


pf_rows = []
for tid in E.index[E["first_final_seg"].notna()]:
    ix = seg_ix[tid]
    ff = int(E.at[tid, "first_final_seg"])
    vals_ = S["ans_norm"].to_numpy()[ix]
    fv = vals_[ff] if vals_[ff] is not None and vals_[ff] == vals_[ff] else None
    for p in range(ff + 1, len(ix)):
        v = vals_[p]
        txt = S["text_clean"].to_numpy()[ix[p]]
        role = S_role[ix[p]]
        if v is not None and v == v and fv is not None and v != fv:
            c = "new different answer"
        elif (v is not None and v == v and v == fv) or mentions(fv, txt):
            c = "restates the answer"
        elif role == "verification":
            c = "verifies"
        elif role == "doubt":
            c = "doubts"
        elif role == "correction":
            c = "corrects"
        else:
            c = "continues reasoning"
        pf_rows.append((tid, c, S_words[ix[p]]))
PF = pd.DataFrame(pf_rows, columns=["trace_id", "cat", "words"])
PF["qkey"] = PF["trace_id"].map(F["qkey"])
PF_CATS = ["new different answer", "restates the answer", "verifies", "doubts", "corrects", "continues reasoning"]
rows, ends = {}, {}
for name, ids in COND.items():
    if not len(ids):
        continue
    has_final = E.loc[ids]
    has_final = has_final[has_final["first_final_seg"].notna()]
    sub = PF[PF["trace_id"].isin(ids)]
    w = sub.groupby(["qkey", "cat"])["words"].sum().unstack(fill_value=0).reindex(columns=PF_CATS, fill_value=0)
    rows[name] = w.div(w.sum(axis=1).replace(0, np.nan), axis=0).mean()
    ends[name] = dict(traces_with_final=len(has_final),
                      share_ending_at_final=float(hier_mean(has_final.assign(e=(has_final["words_after_first_final"] == 0).astype(float)), ["e"])["e"].mean()),
                      median_words_after_final=has_final["words_after_first_final"].median(),
                      share_text_after_final=float(hier_mean(has_final, ["share_after_first_final"])["share_after_first_final"].mean()))
post_tab = pd.DataFrame(rows).T
display(post_tab.round(3))
display(pd.DataFrame(ends).T.round(3))
save_table(post_tab, "T5_post_final_segment_types.csv")
fig, ax = plt.subplots(figsize=(10, 3.8))
post_tab.plot(kind="barh", stacked=True, ax=ax, width=0.7,
              color=["#d62728", "#9e9ac8", "#54a24b", "#f58518", "#e45756", "#4c78a8"])
ax.set_xlabel("share of the words after the first final (mean over questions)")
ax.set_title("T5 · What the text after the first `final` answer does", fontsize=10)
ax.legend(fontsize=7, loc="lower center", bbox_to_anchor=(0.5, -0.5), ncol=3)
fig.tight_layout()
save_fig(fig, "T5_post_final_behaviour.png")
plt.show()
plt.close(fig)

### 10.6 Thinking vs output; how traces end
The output answer is **heuristically extracted** from `raw_model_output` (last `\boxed{}` or "answer is ..." phrase after
`</think>` if present); coverage and agreement with `is_correct` were reported in section 5. Here: does the output answer
equal the last answer candidate of the thinking?

In [ ]:
t_o = E[E["out_ans"].notna() & E["last_val"].notna()]
tab = pd.DataFrame({name: hier_mean(t_o.loc[t_o.index.intersection(ids)], ["out_matches_last_cand", "out_matches_gold", "last_matches_gold"]).mean()
                    for name, ids in COND.items() if len(ids)}).T
tab["n_traces"] = [len(t_o.index.intersection(ids)) for ids in COND.values() if len(ids)]
print("output answer vs last thinking candidate (traces with both available):")
display(tab.round(3))
mis = t_o[t_o["out_matches_last_cand"] == 0]
print("traces where output != last thinking candidate: %d of %d; among them output==gold: %.2f, last candidate==gold: %.2f" % (
    len(mis), len(t_o), mis["out_matches_gold"].mean() if len(mis) else np.nan, mis["last_matches_gold"].mean() if len(mis) else np.nan))
save_table(tab, "T6_output_vs_thinking_answer.csv")

last = S[S["trace_id"].isin(E.index)].groupby("trace_id").tail(1).set_index("trace_id")
E_end = E.assign(last_status=last["answer_status"].reindex(E.index), last_role_=last["role"].reindex(E.index))
E_end["ends_with_final"] = (E_end["last_status"] == "final").astype(float)
end_tab = {}
for name, ids in COND.items():
    if len(ids):
        sub = E_end.loc[ids]
        for r in ROLES:
            sub = sub.assign(**{"end_" + r: (sub["last_role_"] == r).astype(float)})
        end_tab[name] = hier_mean(sub, ["end_" + r for r in ROLES] + ["ends_with_final"]).mean()
end_tab = pd.DataFrame(end_tab).T
print("how traces end (role of the last segment; share whose last segment is the `final` one):")
display(end_tab.round(3))
save_table(end_tab, "T6_how_traces_end.csv")

### 10.7 Segment length vs role; `new` vs `continuation` tested against the text
Segment length (words) by role. Then the annotation `temporal_role` is **tested against text** instead of being read as
novelty: if `continuation` really continues, its cosine with the previous segment should be higher; if `new` really adds
something, its content-word novelty should be higher. AUC = P(score of a continuation segment > score of a new segment);
0.5 = no relation. Segment-level numbers are descriptive (segments are not independent); the question-level AUC uses only
questions with >= 3 segments of each type.

In [ ]:
sl = Se.groupby(["qkey", "role"])["words"].mean().groupby(level="role").agg(["mean", "median"]).reindex(ROLES)
sl["segment_words_median_pooled"] = Se.groupby("role")["words"].median().reindex(ROLES)
sl["q25"], sl["q75"] = Se.groupby("role")["words"].quantile(.25).reindex(ROLES), Se.groupby("role")["words"].quantile(.75).reindex(ROLES)
sl.columns = ["mean of question means", "median of question means", "median (pooled segments)", "q25", "q75"]
display(sl.round(1))
save_table(sl, "T7_segment_length_by_role.csv")
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([Se.loc[Se["role"] == r, "words"].clip(lower=1) for r in ROLES], tick_labels=ROLES, showfliers=False, patch_artist=True,
           boxprops=dict(facecolor="#dddddd"))
ax.set_yscale("log")
ax.set_ylabel("words per segment (log)")
ax.set_title("T7 · Segment length by role", fontsize=10)
fig.tight_layout()
save_fig(fig, "T7_segment_length_by_role.png")
plt.show()
plt.close(fig)

tdf = Se[Se["temporal_role"].isin(TEMPORAL) & (Se["seg_pos"] > 0)]
print("mean text measures by temporal_role (segments after the first one):")
display(tdf.groupby("temporal_role")[["sim_prev", "sim_nonadj_max", "novelty", "rep4_share", "words"]].mean().round(3))
rows = []
for scope, sub in [("all roles", tdf)] + [(r, tdf[tdf["role"] == r]) for r in ROLES]:
    for score, sign in [("sim_prev", 1), ("novelty", -1)]:
        d = sub[[score, "temporal_role", "qkey"]].dropna()
        if d["temporal_role"].nunique() < 2:
            continue
        y = (d["temporal_role"] == "continuation").astype(int)
        auc = roc_auc_score(y, sign * d[score])
        aucs = []
        for q, dq_ in d.groupby("qkey"):
            yq = (dq_["temporal_role"] == "continuation").astype(int)
            if yq.sum() >= 3 and (1 - yq).sum() >= 3:
                aucs.append(roc_auc_score(yq, sign * dq_[score]))
        m_, lo, hi, nq = boot_mean_ci(aucs)
        rows.append(dict(scope=scope, score=("cosine with previous segment (higher => continuation)" if score == "sim_prev" else
                                              "content-word novelty (lower => continuation)"),
                         segment_level_auc=auc, question_level_auc=m_, ci_lo=lo, ci_hi=hi, n_questions=nq,
                         n_continuation=int(y.sum()), n_new=int((1 - y).sum())))
auc_tab = pd.DataFrame(rows)
display(auc_tab.round(3))
save_table(auc_tab, "T7_temporal_role_vs_text.csv", index=False)

### 10.8 Data-driven phrases (additional analysis)
Which words/bigrams are more frequent in the incorrect than in the correct samples *of the same cell*? Rates per 100
words, within-cell differences, averaged per question. To limit multiple looks, terms are first pre-screened by a simple
question-level t statistic (top 400 by |t|), and only terms present in >= `MIN_Q_NGRAM` questions are kept. Ranked by dz.
**Cues only** - an invitation to read text, not a finding about causes; no multiplicity correction.

In [ ]:
mx_ids = E.index[E["cell_group"] == "MIXED"]
trace_text = S[S["trace_id"].isin(mx_ids)].groupby("trace_id")["text_clean"].apply(" ".join)
cv = CountVectorizer(ngram_range=(1, 2), min_df=max(10, int(0.01 * len(trace_text))), max_features=6000, lowercase=True,
                     token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z']+\b", dtype=np.float32)
Cm = cv.fit_transform(trace_text.reindex(mx_ids))
terms = np.array(cv.get_feature_names_out())
rate = pd.DataFrame(Cm.toarray(), index=mx_ids, columns=terms).div(E.loc[mx_ids, "words"], axis=0) * 100
cell_key, corr_key = E.loc[mx_ids, "cell_id"].to_numpy(), E.loc[mx_ids, "correct"].to_numpy()
gg = rate.groupby([cell_key, corr_key]).mean()                    # keys passed as arrays: a term may be called "correct"
wrong_side, right_side = gg.xs(0.0, level=1), gg.xs(1.0, level=1)
diff = wrong_side - right_side.reindex(wrong_side.index)
qd = diff.groupby(cells["qkey"].reindex(diff.index).to_numpy()).mean()
present = pd.DataFrame(Cm.toarray() > 0, index=mx_ids, columns=terms).groupby(E.loc[mx_ids, "qkey"].to_numpy()).any().sum()
qd = qd.loc[:, present[present >= MIN_Q_NGRAM].index]
tstat = (qd.mean() / (qd.std(ddof=1) / np.sqrt(qd.notna().sum()))).abs().sort_values(ascending=False)
cand = tstat.head(400).index
term_eff = paired_summary(qd[cand], B=300)
term_eff["questions_with_term"] = present.reindex(term_eff.index)
term_eff["rate_incorrect"] = rate.loc[corr_key == 0, term_eff.index].mean()
term_eff["rate_correct"] = rate.loc[corr_key == 1, term_eff.index].mean()
save_table(term_eff, "T8_discriminative_terms_mixed.csv")
cols = ["questions_with_term", "rate_incorrect", "rate_correct", "dz", "dz_lo", "dz_hi"]
print("more frequent in INCORRECT samples (within cell):")
display(term_eff.sort_values("dz", ascending=False).head(15)[cols].round(3))
print("more frequent in CORRECT samples (within cell):")
display(term_eff.sort_values("dz").head(15)[cols].round(3))

### 10.9 Lexical signature of each role label (additional sanity check of the annotations)
Log-odds ratio with an informative Dirichlet prior (Monroe et al.): the words most specific to each `discourse_role`
against all other roles (unigrams, z-scored). If the labels are reliable, the signatures should look like what the role
names promise.

In [ ]:
cvu = CountVectorizer(lowercase=True, min_df=20, max_features=8000, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z']+\b", dtype=np.float32)
seg_text = Se["text_clean"]
Cu = cvu.fit_transform(seg_text)
vocab = np.array(cvu.get_feature_names_out())
role_idx = Se["role"].to_numpy()
tot = np.asarray(Cu.sum(axis=0)).ravel()
alpha0 = 500.0
alpha = alpha0 * tot / tot.sum()
sig_rows = {}
for r in ROLES:
    yi = np.asarray(Cu[role_idx == r].sum(axis=0)).ravel()
    yj = tot - yi
    ni, nj = yi.sum(), yj.sum()
    delta = (np.log((yi + alpha) / (ni + alpha0 - yi - alpha)) - np.log((yj + alpha) / (nj + alpha0 - yj - alpha)))
    z = delta / np.sqrt(1 / (yi + alpha) + 1 / (yj + alpha))
    sig_rows[r] = ", ".join(vocab[np.argsort(-z)[:10]])
sig = pd.Series(sig_rows, name="top-10 role-specific words (log-odds z)")
with pd.option_context("display.max_colwidth", 200):
    display(sig.to_frame())
save_table(sig.to_frame(), "T9_role_lexical_signatures.csv")

## 11. Statistics and robustness
**Hierarchy.** Segments are never independent units and samples of a cell are not independent either. Every pooled number
is *cell -> question (equal weight) -> pooled*; CIs come from a cluster bootstrap that resamples **questions**
(`N_BOOT` = %d); p-values (sign-flip permutation and Wilcoxon on the question-level differences) are reported for
orientation only. Everything is exploratory: no multiplicity correction and no causal language. `n` (questions) is shown in
every table.

### 11.1 Permutation baseline: shuffle correctness within cells
Samples of a cell always differ somewhat (the min and max of 5 samples are never identical), so a "difference" can arise
without any link to correctness. Here the correct/incorrect labels are shuffled **within each MIXED cell** (keeping the number
of correct and incorrect samples), the within-cell differences are recomputed and pooled exactly as for the real labels.
The null distribution shows the size of difference that pure sampling noise produces.

In [ ]:
PERM_FEATS = ["share_doubt", "share_verification", "share_correction", "share_reasoning", "changes_per100w",
              "first_ans_rel", "share_after_first_ans", "n_replace_100w", "rep4_share", "near_repeat_wshare",
              "hedge_100w", "novelty", "comp_ratio", "log_words"]
mx_cells = list(cells.index[cells["group"] == "MIXED"])
nc, maxn = len(mx_cells), max(len(cell_members[c]) for c in mx_cells)
VV = np.full((nc, maxn, len(PERM_FEATS)), np.nan)
LL = np.full((nc, maxn), np.nan)
for i, c in enumerate(mx_cells):
    ids = list(cell_members[c])
    VV[i, :len(ids)] = E.loc[ids, PERM_FEATS].to_numpy()
    LL[i, :len(ids)] = E.loc[ids, "correct"].to_numpy()
qc = pd.factorize(cells["qkey"].reindex(mx_cells))[0]
nq_ = qc.max() + 1
valid = ~np.isnan(VV)


def pooled_diff(labels):
    w, c = (labels == 0)[:, :, None] & valid, (labels == 1)[:, :, None] & valid
    nw, ncor = w.sum(1), c.sum(1)
    with np.errstate(divide="ignore", invalid="ignore"):
        d = np.where(w, VV, 0).sum(1) / nw - np.where(c, VV, 0).sum(1) / ncor
    d[(nw == 0) | (ncor == 0)] = np.nan
    out = np.empty(d.shape[1])
    for j in range(d.shape[1]):
        ok = ~np.isnan(d[:, j])
        n_q = np.bincount(qc[ok], minlength=nq_)
        m_q = np.bincount(qc[ok], weights=d[ok, j], minlength=nq_) / np.maximum(n_q, 1)
        out[j] = m_q[n_q > 0].mean()
    return out


obs = pooled_diff(LL)
rng_p = np.random.default_rng(SEED + 1)
null = np.empty((N_PERM, len(PERM_FEATS)))
for b in range(N_PERM):
    r = rng_p.random((nc, maxn))
    r[np.isnan(LL)] = 2.0                                  # padding stays at the end; valid labels are permuted among valid samples
    null[b] = pooled_diff(np.take_along_axis(LL, np.argsort(r, axis=1), axis=1))
perm_tab = pd.DataFrame({"observed_incorrect_minus_correct": obs, "null_mean": null.mean(0), "null_sd": null.std(0),
                         "null_lo": np.percentile(null, 2.5, axis=0), "null_hi": np.percentile(null, 97.5, axis=0)}, index=PERM_FEATS)
perm_tab["z_vs_null"] = (perm_tab["observed_incorrect_minus_correct"] - perm_tab["null_mean"]) / perm_tab["null_sd"]
perm_tab["p_two_sided"] = [(1 + (np.abs(null[:, j] - null[:, j].mean()) >= abs(obs[j] - null[:, j].mean())).sum()) / (N_PERM + 1)
                           for j in range(len(PERM_FEATS))]
chk_mean = eff_mx.reindex(PERM_FEATS)["mean"].to_numpy()
assert np.allclose(obs, chk_mean, atol=1e-9, equal_nan=True), "permutation code must reproduce the pooled observed differences"
display(perm_tab.round(4))
save_table(perm_tab, "S11_permutation_baseline_mixed.csv")
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.hlines(range(len(perm_tab)), perm_tab["null_lo"] / perm_tab["null_sd"], perm_tab["null_hi"] / perm_tab["null_sd"], color="#bbbbbb", lw=6)
ax.scatter(perm_tab["observed_incorrect_minus_correct"] / perm_tab["null_sd"], range(len(perm_tab)), color="#d62728", zorder=3)
ax.axvline(0, color="grey", lw=0.8)
ax.set_yticks(range(len(perm_tab)))
ax.set_yticklabels(perm_tab.index, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("pooled within-cell difference in units of the permutation SD (grey band = null 95% range)")
ax.set_title("S11 · Observed incorrect - correct differences vs the shuffle-within-cell baseline (%d permutations)" % N_PERM, fontsize=9)
fig.tight_layout()
save_fig(fig, "S11_permutation_baseline.png")
plt.show()
plt.close(fig)

### 11.2 Length control
(a) Features are length-normalised by construction (shares, per-100-word rates, relative positions). (b) **Length-matched
pairs**: per MIXED cell, only the incorrect/correct pair with the closest word counts, kept if the counts differ by less than
15% (caliper). (c) **Regression with cell fixed effects and a log-length covariate**: for every feature,
`y = cell effect + b * incorrect + g * log(words)`, estimated on within-cell deviations; CIs by bootstrapping questions.
Cells are finer than questions (question x compute level), which only makes the fixed effects more conservative.

In [ ]:
CALIPER = 0.15
dvm = div_df[div_df["pair"].str.startswith("wrong")].copy()
dvm["abs_log_ratio"] = np.abs(np.log(dvm["words_a"].astype(float) / dvm["words_b"].astype(float)))
matched = dvm[dvm["abs_log_ratio"] < np.log(1 + CALIPER)]
LC_FEATS = [f for f in PERM_FEATS if f != "log_words"]
Fa, Fb = E.loc[matched["id_a"], LC_FEATS].reset_index(drop=True), E.loc[matched["id_b"], LC_FEATS].reset_index(drop=True)
md = (Fa - Fb)
md["qkey"] = cells["qkey"].reindex(matched["cell_id"]).to_numpy()
eff_matched = paired_summary(md.groupby("qkey").mean(), LC_FEATS)
cmp_ = pd.DataFrame({"dz_all_cells": eff_mx.reindex(LC_FEATS)["dz"], "n_q_all": eff_mx.reindex(LC_FEATS)["n_questions"],
                     "dz_length_matched": eff_matched["dz"], "dz_lm_lo": eff_matched["dz_lo"], "dz_lm_hi": eff_matched["dz_hi"],
                     "n_q_matched": eff_matched["n_questions"]})
print("length-matched pairs (|log ratio| < log(1+%.2f)): %d of %d MIXED cells, %d questions" % (
    CALIPER, len(matched), len(dvm), matched["cell_id"].map(cells["qkey"]).nunique()))
display(cmp_.round(3))
save_table(cmp_, "S11_length_matched_vs_all.csv")

mxE = E[E["cell_group"] == "MIXED"]
cid_, qcode, quniq = mxE["cell_id"].to_numpy(), *pd.factorize(mxE["qkey"].to_numpy())
w_ = (mxE["correct"] == 0).to_numpy(float)
lw_ = mxE["log_words"].to_numpy(float)
# Within-cell demeaned cross-products are additive over traces, so they are accumulated per question once and the question
# bootstrap only re-adds them. Each feature is demeaned over the traces where that feature is defined.
PAIRS = [("w", "w"), ("w", "l"), ("l", "l"), ("w", "y"), ("l", "y")]
sums = np.zeros((len(quniq), len(LC_FEATS), len(PAIRS)))
sd_y = np.zeros(len(LC_FEATS))
for j, f in enumerate(LC_FEATS):
    y = mxE[f].to_numpy(float)
    ok = ~np.isnan(y)
    d0 = pd.DataFrame({"c": cid_[ok], "y": y[ok], "w": w_[ok], "l": lw_[ok]})
    dm = d0[["y", "w", "l"]] - d0.groupby("c")[["y", "w", "l"]].transform("mean")
    sd_y[j] = np.sqrt((dm["y"] ** 2).mean())
    for k, (a, b) in enumerate(PAIRS):
        sums[:, j, k] = np.bincount(qcode[ok], weights=(dm[a] * dm[b]).to_numpy(), minlength=len(quniq))


def solve(S):
    Sww, Swl, Sll, Swy, Sly = S.T
    with np.errstate(divide="ignore", invalid="ignore"):
        return Swy / Sww, (Sll * Swy - Swl * Sly) / (Sww * Sll - Swl ** 2)      # without / with the log-length covariate


b1, b2 = solve(sums.sum(0))
rng_r = np.random.default_rng(SEED + 2)
boots = [solve(sums[rng_r.integers(0, len(quniq), len(quniq))].sum(0)) for _ in range(N_BOOT)]
B1, B2 = np.array([b[0] for b in boots]), np.array([b[1] for b in boots])
reg = pd.DataFrame({"coef_no_length_control": b1, "lo1": np.nanpercentile(B1, 2.5, axis=0), "hi1": np.nanpercentile(B1, 97.5, axis=0),
                    "coef_with_log_length": b2, "lo2": np.nanpercentile(B2, 2.5, axis=0), "hi2": np.nanpercentile(B2, 97.5, axis=0)}, index=LC_FEATS)
reg["std_coef_no_control"] = reg["coef_no_length_control"] / sd_y
reg["std_coef_with_control"] = reg["coef_with_log_length"] / sd_y
reg["still_excludes_0"] = (reg["lo2"] > 0) | (reg["hi2"] < 0)
print("cell fixed effects regression (incorrect coefficient; %d traces, %d cells, %d questions; %d question-bootstrap resamples):" % (
    len(mxE), mxE["cell_id"].nunique(), len(quniq), N_BOOT))
display(reg.round(4))
save_table(reg, "S11_regression_fixed_effects.csv")

### 11.3 Sensitivity analyses
The key effects are recomputed under: (1) the main definition, (2) truncated traces **included**, (3) traces with an empty
extracted output **excluded**, (4) stricter / looser ALL-CORRECT spread thresholds, (5) only cells with exactly 5 eligible
samples, (6) only cells with exactly 4 eligible samples, (7) length-matched pairs only (MIXED). Reported: `dz` of incorrect minus
correct for MIXED, and the share of the extra words by role / before the first answer for ALL-CORRECT.

In [ ]:
SENS_MIXED = ["share_doubt", "share_verification", "share_correction", "changes_per100w", "n_replace_100w", "rep4_share", "hedge_100w", "first_ans_rel"]
SENS_AC = ["xshare_" + r for r in ROLES] + ["xshare_before_ans"]


def scenario(label, Fe, cells_df):
    out = {"scenario": label, "n_cells_MIXED": int((cells_df["group"] == "MIXED").sum()),
           "n_questions_MIXED": cells_df.loc[cells_df["group"] == "MIXED", "qkey"].nunique(),
           "n_cells_ALLCORRECT": int((cells_df["group"] == "ALL-CORRECT").sum())}
    mx_ = mixed_cell_diffs(Fe, cells_df, SENS_MIXED)
    e_ = paired_summary(to_question_level(mx_, SENS_MIXED), B=500)
    for f in SENS_MIXED:
        out["dz:" + f] = e_.at[f, "dz"] if f in e_.index else np.nan
    ids = cells_df.index[cells_df["group"] == "ALL-CORRECT"]
    if len(ids) >= 3:
        pr = make_pairs(Fe, ids)
        t_ = long_short_table(Fe, pr, ["share_doubt"])
        t_["qkey"] = cells_df["qkey"].reindex(t_.index).to_numpy()
        qm_ = t_.groupby("qkey")[SENS_AC].mean().mean()
        for f in SENS_AC:
            out["xs:" + f.replace("xshare_", "")] = qm_[f]
    return out


scen = [scenario("1 main", E, cells)]
scen.append(scenario("2 truncated traces included", F, build_cells(F)))
Fe_ne = E[E["empty_out"] != 1]
scen.append(scenario("3 empty-output traces excluded", Fe_ne, build_cells(Fe_ne)))
scen.append(scenario("4a ALL-CORRECT spread: 1.0 SD & ratio 1.25", E, build_cells(E, spread_sd=1.0, min_ratio=1.25)))
scen.append(scenario("4b ALL-CORRECT spread: 2.0 SD & ratio 2.0", E, build_cells(E, spread_sd=2.0, min_ratio=2.0)))
c5 = build_cells(E, min_elig=5)
scen.append(scenario("5 cells with exactly 5 eligible samples", E, c5[c5["n_elig"] == 5]))
c4 = build_cells(E, min_elig=4)
scen.append(scenario("6 cells with exactly 4 eligible samples", E, c4[c4["n_elig"] == 4]))
sens = pd.DataFrame(scen).set_index("scenario")
lm = eff_matched.reindex(SENS_MIXED)
sens.loc["7 length-matched pairs only (MIXED)"] = [matched["cell_id"].nunique(), matched["cell_id"].map(cells["qkey"]).nunique(), np.nan] + \
    [lm.at[f, "dz"] for f in SENS_MIXED] + [np.nan] * len(SENS_AC)
display(sens.round(3))
save_table(sens, "S11_sensitivity.csv")
fig, axes = plt.subplots(1, 2, figsize=(16, 5.2))
dzc = [c for c in sens.columns if c.startswith("dz:")]
for k, sc in enumerate(sens.index):
    axes[0].scatter(sens.loc[sc, dzc].to_numpy(float), np.arange(len(dzc)) + (k - 3) * 0.09, s=22, label=sc)
axes[0].axvline(0, color="grey", lw=0.8)
axes[0].set_yticks(range(len(dzc)))
axes[0].set_yticklabels([c[3:] for c in dzc], fontsize=8)
axes[0].set_xlabel("dz (incorrect - correct, MIXED cells)")
axes[0].set_title("MIXED effects across scenarios", fontsize=9)
axes[0].legend(fontsize=6, loc="lower right")
xsc = [c for c in sens.columns if c.startswith("xs:")]
for k, sc in enumerate(sens.index[:-1]):
    axes[1].scatter(np.arange(len(xsc)) + (k - 3) * 0.09, sens.loc[sc, xsc].to_numpy(float), s=22, label=sc)
axes[1].set_xticks(range(len(xsc)))
axes[1].set_xticklabels([c[3:] for c in xsc], rotation=20, fontsize=8)
axes[1].set_ylabel("share of the extra words (long - short)")
axes[1].set_title("ALL-CORRECT extra-word shares across scenarios", fontsize=9)
fig.tight_layout()
save_fig(fig, "S11_sensitivity.png")
plt.show()
plt.close(fig)

### 11.4 Truncated traces, analysed separately
Truncated traces are excluded from every main analysis (their length is cut by the budget, not by the model's own stopping).
Here they are compared with complete traces: how often, at which compute level, how accurate, and *within the same cell*.

In [ ]:
Fall = F.copy()
tr_tab = Fall.groupby("compute_level").agg(rows=("words", "size"), truncated=("trunc", "sum"), mean_words=("words", "mean"))
tr_tab["truncation_rate"] = tr_tab["truncated"] / tr_tab["rows"]
tr_tab["accuracy_truncated"] = Fall[Fall["trunc"] == 1].groupby("compute_level")["correct"].mean()
tr_tab["accuracy_complete"] = Fall[Fall["trunc"] == 0].groupby("compute_level")["correct"].mean()
tr_tab["has_answer_truncated"] = Fall[Fall["trunc"] == 1].groupby("compute_level")["has_answer"].mean()
tr_tab["has_answer_complete"] = Fall[Fall["trunc"] == 0].groupby("compute_level")["has_answer"].mean()
display(tr_tab.reindex(COMPUTE_ORDER).round(3))
save_table(tr_tab.reindex(COMPUTE_ORDER), "S11_truncated_overview.csv")
tf = ["words", "share_doubt", "share_verification", "share_correction", "share_reasoning", "rep4_share", "hedge_100w", "has_answer"]
g_ = Fall.groupby(["cell_id", "trunc"])[tf].mean()
both = g_.xs(1.0, level="trunc").index.intersection(g_.xs(0.0, level="trunc").index)
dd = g_.xs(1.0, level="trunc").loc[both] - g_.xs(0.0, level="trunc").loc[both]
dd["qkey"] = Fall.drop_duplicates("cell_id").set_index("cell_id")["qkey"].reindex(dd.index).to_numpy()
eff_tr = paired_summary(dd.groupby("qkey").mean(), tf)
print("within-cell truncated minus complete (%d cells with both, %d questions):" % (len(both), dd["qkey"].nunique()))
display(eff_tr[["n_questions", "mean", "ci_lo", "ci_hi", "dz"]].round(3))
save_table(eff_tr, "S11_truncated_vs_complete_within_cell.csv")

### 11.5 Composition and heterogeneity
The composition tables (benchmark / difficulty / compute level) are in section 6. Do the MIXED effects look similar in every
benchmark and difficulty tier? (`dz`, `*` = CI excludes 0; tiers with fewer than 8 questions are skipped.)

In [ ]:
HET_FEATS = ["share_doubt", "share_verification", "share_correction", "share_reasoning", "changes_per100w", "n_replace_100w",
             "first_ans_rel", "rep4_share", "hedge_100w", "log_words"]
mxh = mx_tab.assign(benchmark=cells["benchmark"].reindex(mx_tab.index).to_numpy(),
                    difficulty_tier=cells["difficulty_tier"].reindex(mx_tab.index).to_numpy())
het = {}
for dim in ["benchmark", "difficulty_tier"]:
    for lab, sub in mxh.groupby(dim):
        if sub["qkey"].nunique() >= 8:
            het["%s=%s (n=%d q)" % (dim, lab, sub["qkey"].nunique())] = paired_summary(to_question_level(sub, HET_FEATS), B=500)
het_dz = pd.DataFrame({k: v["dz"] for k, v in het.items()}).reindex(HET_FEATS)
het_sig = pd.DataFrame({k: (v["dz_lo"] > 0) | (v["dz_hi"] < 0) for k, v in het.items()}).reindex(HET_FEATS)
if het:
    fig, ax = plt.subplots(figsize=(max(7, 1.5 * len(het_dz.columns)), 5))
    sns.heatmap(het_dz.astype(float), annot=het_dz.round(2).astype(str) + het_sig.map(lambda b: "*" if b else ""), fmt="", cmap="RdBu_r",
                center=0, vmin=-1.5, vmax=1.5, ax=ax, annot_kws={"fontsize": 7}, cbar_kws={"label": "dz (incorrect - correct)"})
    ax.set_title("S11 · MIXED effects by benchmark and difficulty tier (* CI excludes 0)", fontsize=9)
    ax.tick_params(axis="x", labelsize=7)
    fig.tight_layout()
    save_fig(fig, "S11_heterogeneity.png")
    plt.show()
    plt.close(fig)
    save_table(het_dz, "S11_heterogeneity_dz.csv")

## 12. Findings
Every sentence below is **generated from the result tables of this run** (nothing is typed in by hand), so the text always matches
the numbers. The findings are split into **Observed** (what the tables show, with question-level n and 95% CIs), **Interpretation**
(conditional readings, emitted only when their premise holds), **Unresolved** (what the data cannot decide) and **Limitations**.
A MIXED effect is called *robust* only if it (1) has a bootstrap CI that excludes 0, (2) lies outside the shuffle-within-cell
baseline (p < 0.05), (3) keeps its sign with a CI excluding 0 in length-matched pairs, (4) keeps it in the cell-fixed-effects
regression with a log-length covariate, and (5) keeps its sign in every sensitivity scenario. Fewer passed checks give
"partly robust"; none gives "fragile". This is a descriptive label, not a test.

In [ ]:
def f2(x):
    return "n/a" if pd.isna(x) else "%.2f" % x


def f3(x):
    return "n/a" if pd.isna(x) else "%.3f" % x


def pc(x):
    return "n/a" if pd.isna(x) else "%.1f%%" % (100 * x)


def ci_txt(r, d=3):
    return "%.*f [%.*f, %.*f]" % (d, r["mean"], d, r["ci_lo"], d, r["ci_hi"])


def direction(lo, hi):
    return 1 if lo > 0 else (-1 if hi < 0 else 0)


def md_table(df, index=True):
    d = df.reset_index() if index else df
    head = "| " + " | ".join(str(c) for c in d.columns) + " |"
    sep = "|" + "|".join("---" for _ in d.columns) + "|"
    body = ["| " + " | ".join("" if (isinstance(v, float) and np.isnan(v)) else str(v) for v in row) + " |" for row in d.to_numpy()]
    return "\n".join([head, sep] + body)


def mixed_verdict(f):
    """Robustness label for one MIXED feature (incorrect minus correct); returns (label, direction, passed, available)."""
    if f not in eff_mx.index or pd.isna(eff_mx.at[f, "ci_lo"]):
        return "not estimable", 0, 0, 0
    r = eff_mx.loc[f]
    s = direction(r["ci_lo"], r["ci_hi"])
    if s == 0:
        return "no clear difference", 0, 0, 0
    checks = []
    if f in perm_tab.index:
        checks.append(perm_tab.at[f, "p_two_sided"] < 0.05 and np.sign(perm_tab.at[f, "observed_incorrect_minus_correct"]) == s)
    if f in eff_matched.index and not pd.isna(eff_matched.at[f, "ci_lo"]):
        checks.append(direction(eff_matched.at[f, "ci_lo"], eff_matched.at[f, "ci_hi"]) == s)
    if f in reg.index and not pd.isna(reg.at[f, "lo2"]):
        checks.append(direction(reg.at[f, "lo2"], reg.at[f, "hi2"]) == s)
    col = "dz:" + f
    if col in sens.columns:
        v = sens[col].dropna()
        checks.append(bool((np.sign(v) == s).all()))
    k, m = int(sum(checks)), len(checks)
    label = "robust" if k == m else ("partly robust (%d of %d checks)" % (k, m) if k > 0 else "fragile (0 of %d checks)" % m)
    return label, s, k, m


OBS, INT, UNR = [], [], []
UP = {1: "higher", -1: "lower", 0: "no clear difference"}

# ---- data basis ---------------------------------------------------------------------------------------------------
n_rows, n_eligible = int(funnel["remaining"].iloc[0]), int(E.shape[0])
n_trunc = int((F["trunc"] == 1).sum())
grp = overall["cells"]
OBS.append("**Data basis.** %d CSV rows; %d traces remain after the exclusion funnel and are analysed (%d truncated traces are kept out of the main "
           "analyses and treated separately in 11.4). Cells by group: ALL-CORRECT %d (spread too narrow: %d), MIXED %d, ALL-WRONG %d, too few "
           "eligible samples %d. Questions: ALL-CORRECT %d, MIXED %d, ALL-WRONG %d." % (
               n_rows, n_eligible, n_trunc, grp["ALL-CORRECT"], grp["ALL-CORRECT-NARROW"], grp["MIXED"], grp["ALL-WRONG"], grp["TOO-FEW"],
               overall.at["ALL-CORRECT", "questions"], overall.at["MIXED", "questions"], overall.at["ALL-WRONG", "questions"]))
rd = recon_report.set_index("check")["n_traces"]
OBS.append("**Label quality checks.** %d traces (%s) have a segment count or id set in the text that differs from the annotation JSON and are excluded; "
           "%d traces have an unexpected discourse role. The last labelled candidate answer equals the gold answer in %d of %d correct traces "
           "and differs from it in %d of %d incorrect traces (a consistency check of the labelled answer values against is_correct)." % (
               rd["segment-id set in text != annotation id set"], pc(rd["segment-id set in text != annotation id set"] / rd["rows"]),
               rd["traces with unexpected discourse_role value"],
               agree.at["is_correct=1", "last answer == gold"], agree.loc["is_correct=1"].sum(),
               agree.at["is_correct=0", "last answer != gold"], agree.loc["is_correct=0"].sum()))

# ---- ALL-CORRECT --------------------------------------------------------------------------------------------------
if "log_ratio" in eff_ac.index and not pd.isna(eff_ac.at["log_ratio", "mean"]):
    lr = eff_ac.loc["log_ratio"]
    ew = eff_ac.loc["extra_words"]
    OBS.append("**ALL-CORRECT cells (long vs short correct trace, n = %d questions).** The long trace is %.2fx the short one in words (95%% CI %.2f-%.2f) "
               "and adds %.0f words [%.0f, %.0f]." % (lr["n_questions"], np.exp(lr["mean"]), np.exp(lr["ci_lo"]), np.exp(lr["ci_hi"]),
                                                      ew["mean"], ew["ci_lo"], ew["ci_hi"]))
    parts = []
    for lab, r in base.iterrows():
        side = "above" if r["observed_share_of_extra_words"] > r["null_hi"] else ("below" if r["observed_share_of_extra_words"] < r["null_lo"] else "within")
        parts.append("%s %s (%s the chance band %s-%s)" % (lab.replace("extra words in role: ", "").replace("extra words ", ""),
                                                           pc(r["observed_share_of_extra_words"]), side, pc(r["null_lo"]), pc(r["null_hi"])))
    OBS.append("Share of the **extra words** by role / position, against the baseline obtained by shuffling the role shares within each cell: " + "; ".join(parts) + ".")
    over = [lab.replace("extra words in role: ", "") for lab, r in base.iterrows()
            if r["observed_share_of_extra_words"] > r["null_hi"] and lab.startswith("extra words in role")]
    under = [lab.replace("extra words in role: ", "") for lab, r in base.iterrows()
             if r["observed_share_of_extra_words"] < r["null_lo"] and lab.startswith("extra words in role")]
    if over or under:
        INT.append("In long correct traces the extra length is %s%s relative to the shuffle baseline. This says where the additional words sit, "
                   "not why the model produced them." % (("over-represented in " + ", ".join(over)) if over else "",
                                                          ((" and " if over else "") + "under-represented in " + ", ".join(under)) if under else ""))
    else:
        INT.append("In long correct traces no role takes a share of the extra words that clearly differs from the shuffle baseline; the extra length is "
                   "spread roughly in proportion to the role mix.")
else:
    OBS.append("**ALL-CORRECT cells.** Too few cells with a real length spread for question-level estimates in this run.")

# ---- MIXED ---------------------------------------------------------------------------------------------------------
rows = []
for f in KEY_FEATS:
    if f not in eff_mx.index or pd.isna(eff_mx.at[f, "ci_lo"]):
        continue
    r = eff_mx.loc[f]
    lab, s, k, m = mixed_verdict(f)
    rows.append(dict(feature=f, n_questions=int(r["n_questions"]), incorrect_minus_correct=ci_txt(r, 3), dz=f2(r["dz"]), direction=UP[s], verdict=lab))
mx_summary = pd.DataFrame(rows).set_index("feature")
save_table(mx_summary, "S12_mixed_key_effects_verdicts.csv")
OBS.append("**MIXED cells (incorrect minus correct sample, within the same cell; mean over cells, then over questions; 95% question-bootstrap CI)** - table below.")
OBS.append(md_table(mx_summary))
robust_up = [f for f in mx_summary.index if mx_summary.at[f, "verdict"] == "robust" and mx_summary.at[f, "direction"] == "higher"]
robust_dn = [f for f in mx_summary.index if mx_summary.at[f, "verdict"] == "robust" and mx_summary.at[f, "direction"] == "lower"]
fragile = [f for f in mx_summary.index if mx_summary.at[f, "verdict"].startswith(("fragile", "partly"))]
OBS.append("Robust differences in incorrect samples - higher: %s; lower: %s. Partly robust or fragile: %s. No clear difference: %s." % (
    ", ".join(robust_up) or "none", ", ".join(robust_dn) or "none", ", ".join(fragile) or "none",
    ", ".join(f for f in mx_summary.index if mx_summary.at[f, "direction"] == "no clear difference") or "none"))
if "log_words" in mx_summary.index:
    OBS.append("Length itself: incorrect samples differ from correct ones in log-words by %s (dz %s, %s); the length-matched and length-controlled estimates above "
               "are the ones to read for the other features." % (mx_summary.at["log_words", "incorrect_minus_correct"], mx_summary.at["log_words", "dz"],
                                                                 mx_summary.at["log_words", "verdict"]))
ol = [f for f in OUTCOME_LINKED if f in eff_mx.index and not pd.isna(eff_mx.at[f, "ci_lo"])]
if ol:
    OBS.append("Outcome-linked features (tied to correctness by construction, therefore not evidence of a pattern): " + "; ".join(
        "%s %s" % (f, ci_txt(eff_mx.loc[f], 3)) for f in ol) + ".")
if {"share_doubt", "share_correction"} <= set(robust_up) and ({"share_reasoning", "share_verification"} & set(robust_dn)):
    INT.append("Incorrect samples of the same cell put a larger share of their words into doubt and correction and a smaller share into plain reasoning / "
               "verification, and this survives length matching and the shuffle baseline. A reading consistent with this is that unsuccessful samples are "
               "less settled and revise more, but the data show co-occurrence only: doubt may be a consequence of a failing line of reasoning as much as a cause.")
elif robust_up or robust_dn:
    INT.append("Only part of the expected doubt / correction profile is robust (see the verdict column); the robust differences are those listed above and "
               "nothing more should be read into the others.")
else:
    INT.append("No MIXED difference passes all robustness checks in this run, so no trajectory feature separates incorrect from correct samples "
               "reliably once length and sampling noise are accounted for.")

# ---- ALL-WRONG transfer --------------------------------------------------------------------------------------------
tf_ = transfer.drop(index="log_words", errors="ignore")
tf_ = tf_.loc[[f for f in tf_.index if f in PATTERN_FEATS]]
clear = tf_[(tf_["dz_lo"] > 0) | (tf_["dz_hi"] < 0)]
if len(clear) and tf_["n_q_allwrong"].max() >= 5 and tf_["n_q_allcorrect"].max() >= 5:
    same = clear[np.sign(clear["dz_mixed"]) == np.sign(clear["d_allwrong_vs_allcorrect"])]
    same_ci = same[(same["d_lo"] > 0) | (same["d_hi"] < 0)]
    pos = tf_["allwrong_position_between_mixed_means"].dropna()
    OBS.append("**ALL-WRONG vs ALL-CORRECT cells (pattern transfer from MIXED).** Of %d pattern features with a clear MIXED effect, %d have the same sign "
               "in ALL-WRONG minus ALL-CORRECT cells and %d of those also have a CI that excludes 0 (%d questions with ALL-WRONG cells, %d with ALL-CORRECT cells at most). "
               "%s" % (len(clear), len(same), len(same_ci), tf_["n_q_allwrong"].max(), tf_["n_q_allcorrect"].max(),
                       ("Where the MIXED gap is large enough to define it, ALL-WRONG cells sit at a median %.2f on the scale 0 = MIXED-correct mean, "
                        "1 = MIXED-incorrect mean (%d features)." % (pos.median(), len(pos))) if len(pos) else ""))
    if len(same_ci) >= max(1, len(clear) // 2):
        INT.append("The within-cell pattern also shows up between cell types, which argues against it being a pure within-cell sampling artefact. It does not "
                   "separate the effect from question difficulty: ALL-WRONG cells are probably harder questions.")
    else:
        INT.append("The within-cell MIXED pattern is only weakly mirrored between ALL-WRONG and ALL-CORRECT cells, so it may be specific to cells where the "
                   "model can succeed, or the ALL-WRONG / ALL-CORRECT groups may be too small in this run to say.")
else:
    OBS.append("**ALL-WRONG vs ALL-CORRECT cells.** Too few questions with both cell types for a transfer comparison in this run.")

# ---- answer development --------------------------------------------------------------------------------------------
if {"correct", "incorrect"} <= set(pat_q.index):
    OBS.append("**Answer development in MIXED cells (question-level mean share of samples).** " + "; ".join(
        "%s samples: %s" % (k, ", ".join("%s %s" % (c, pc(pat_q.at[k, c])) for c in pat_q.columns)) for k in ["correct", "incorrect"]) +
        ". 'gold candidate later replaced' is outcome-linked by construction.")
    if pat_q.at["incorrect", "answer replaced"] > pat_q.at["correct", "answer replaced"] + 0.1:
        INT.append("Incorrect samples change their stated answer much more often; a considerable part of them had the gold value as a candidate earlier and "
                   "moved away from it. This describes what the failure looks like in the text; it cannot be used as a predictor because it is defined with the outcome.")

# ---- phase transitions ----------------------------------------------------------------------------------------------
ng_ok = ng_eff.dropna(subset=["ci_lo"])
ng_ok = ng_ok[(ng_ok["ci_lo"] > 0) | (ng_ok["ci_hi"] < 0)]
if len(ng_ok):
    top_up = ng_ok[ng_ok["mean"] > 0].sort_values("dz", ascending=False).head(4)
    top_dn = ng_ok[ng_ok["mean"] < 0].sort_values("dz").head(4)
    OBS.append("**Role transitions (rate per 100 segments, incorrect minus correct, MIXED).** More frequent in incorrect samples: %s. Less frequent: %s." % (
        "; ".join("%s (%s, dz %s)" % (i, ci_txt(r, 2), f2(r["dz"])) for i, r in top_up.iterrows()) or "none",
        "; ".join("%s (%s, dz %s)" % (i, ci_txt(r, 2), f2(r["dz"])) for i, r in top_dn.iterrows()) or "none"))
else:
    OBS.append("**Role transitions.** No role bigram differs clearly between incorrect and correct MIXED samples.")
ro = out_all[out_all["role"] == "doubt"] if "role" in out_all.columns else out_all.iloc[0:0]
if len(ro) and {"MIXED correct", "MIXED incorrect"} <= set(ro.index):
    OBS.append("**What follows a doubt phase (MIXED).** The stated answer changes in %s of incorrect samples' doubt phases vs %s in correct samples; the trace returns to "
               "plain reasoning/setup in %s vs %s." % (pc(ro.at["MIXED incorrect", "answer changes"]), pc(ro.at["MIXED correct", "answer changes"]),
                                                      pc(ro.at["MIXED incorrect", "returns to reasoning/setup"]), pc(ro.at["MIXED correct", "returns to reasoning/setup"])))

# ---- divergence candidates -------------------------------------------------------------------------------------------
pa, pb = mx_dv["phase_div_rel_a"].dropna(), mx_dv["phase_div_rel_b"].dropna()
OBS.append("**Divergence candidates (length-matched incorrect vs correct pair per MIXED cell, %d pairs).** First phase at which the role sequence differs: "
           "median at %s of the incorrect trace and %s of the correct trace (IQR %s-%s and %s-%s). A text-based divergence (no counterpart segment in the other "
           "trace) could be located in %d incorrect-side and %d correct-side traces. These are **algorithmic candidates that need manual reading** "
           "(tables/divergence_candidates.csv, with context)." % (
               len(mx_dv), pc(pa.median()), pc(pb.median()), pc(pa.quantile(.25)), pc(pa.quantile(.75)), pc(pb.quantile(.25)), pc(pb.quantile(.75)),
               mx_dv["text_div_rel_a"].notna().sum(), mx_dv["text_div_rel_b"].notna().sum()))

# ---- compute level -----------------------------------------------------------------------------------------------------
lvl = " ; ".join("%s: accuracy %s [%s, %s], median %.0f words, truncation %s" % (
    lv, pc(lv_tab.at[lv, "accuracy_question_level"]), pc(lv_tab.at[lv, "acc_lo"]), pc(lv_tab.at[lv, "acc_hi"]), lv_tab.at[lv, "words_median"],
    pc(lv_tab.at[lv, "truncation_rate"])) for lv in COMPUTE_ORDER if lv in lv_tab.index)
OBS.append("**Compute levels (Part C).** " + lvl + ".")
vc = persist["verdict"].value_counts()
pers = [f for f in persist.index if persist.at[f, "verdict"] == "persists"]
OBS.append("Of %d effect features, %d keep the same sign with a CI that excludes 0 at every compute level ('persists': %s); %d are inconclusive or differ between levels." % (
    len(persist), vc.get("persists", 0), ", ".join(pers[:8]) + ("..." if len(pers) > 8 else "") if pers else "none", len(persist) - vc.get("persists", 0)))
if vc.get("persists", 0) >= len(persist) / 2:
    INT.append("The incorrect-vs-correct profile is largely the same at every compute level, so it is not specific to short or long reasoning budgets.")
else:
    INT.append("The incorrect-vs-correct profile is not stable across compute levels for most features; pooled estimates mix level-specific patterns.")

# ---- textual analyses ---------------------------------------------------------------------------------------------------
tx = [f for f in ["rep4_share", "near_repeat_wshare", "novelty", "comp_ratio", "hedge_100w"] if f in eff_mx.index and not pd.isna(eff_mx.at[f, "ci_lo"])]
OBS.append("**Repetition and novelty (MIXED, incorrect minus correct).** " + "; ".join(
    "%s %s (dz %s; %s)" % (f, ci_txt(eff_mx.loc[f], 3), f2(eff_mx.at[f, "dz"]), mixed_verdict(f)[0]) for f in tx) + ".")
lt = loop_tab.reindex(["MIXED correct", "MIXED incorrect", "ALL-CORRECT short", "ALL-CORRECT long"]).dropna(how="all")
OBS.append("**Loops (condition means; no CIs, descriptive).** " + "; ".join("%s: repeated-sentence share %s, repeated-equation share %s" % (
    i, pc(r["sent_repeat_share"]), pc(r["eq_repeat_share"])) for i, r in lt.iterrows()) +
    ". The binary loop flag is almost always set, so the continuous shares are the primary measures.")
cue = rate_role.drop(columns=["hedge"], errors="ignore")
top_roles = ["%s -> %s" % (c, cue[c].idxmax()) for c in cue.columns]
OBS.append("**Lexical cues vs discourse roles (cues per 100 words, role with the highest rate).** " + "; ".join(top_roles) + ". This is a plausibility check "
           "of the role labels against the text, not independent evidence, because the labeller may have used the same words.")
ce = cue_eff.loc["incorrect-correct (MIXED)"] if "incorrect-correct (MIXED)" in cue_eff.index.get_level_values(0) else cue_eff.iloc[0:0]
ce_clear = ce[(ce["ci_lo"] > 0) | (ce["ci_hi"] < 0)]
OBS.append("Cue rates that differ between incorrect and correct MIXED samples (CI excludes 0): " + ("; ".join(
    "%s %s" % (i, ci_txt(r, 2)) for i, r in ce_clear.iterrows()) if len(ce_clear) else "none") + ".")
te = term_eff[(term_eff["ci_lo"] > 0) | (term_eff["ci_hi"] < 0)]
t_up, t_dn = te[te["mean"] > 0].sort_values("dz", ascending=False).head(8), te[te["mean"] < 0].sort_values("dz").head(8)
OBS.append("**Cell-controlled discriminative words/phrases (rate per 1000 words, incorrect vs correct, MIXED; tested at question level, no multiplicity correction).** "
           "Higher in incorrect: %s. Higher in correct: %s." % (
               ", ".join("'%s' (dz %s)" % (i, f2(r["dz"])) for i, r in t_up.iterrows()) or "none",
               ", ".join("'%s' (dz %s)" % (i, f2(r["dz"])) for i, r in t_dn.iterrows()) or "none"))
if {"MIXED correct", "MIXED incorrect"} <= set(post_tab.index):
    OBS.append("**After the first answer (share of later segments by role-based category, MIXED).** incorrect: " + ", ".join(
        "%s %s" % (c, pc(post_tab.at["MIXED incorrect", c])) for c in post_tab.columns) + "; correct: " + ", ".join(
        "%s %s" % (c, pc(post_tab.at["MIXED correct", c])) for c in post_tab.columns) + ". Traces end with a final-status segment in %s (incorrect) vs %s (correct)." % (
            pc(end_tab.at["MIXED incorrect", "ends_with_final"]), pc(end_tab.at["MIXED correct", "ends_with_final"])))
au = auc_tab[auc_tab["scope"] == "all roles"].set_index("score")
OBS.append("**temporal_role against text.** Can the labelled continuation/new distinction be recovered from lexical overlap? Question-level AUC: " + "; ".join(
    "%s = %s [%s, %s]" % (i, f3(r["question_level_auc"]), f3(r["ci_lo"]), f3(r["ci_hi"])) for i, r in au.iterrows()) + " (0.5 = no relation).")
if len(au) and (au["question_level_auc"].sub(0.5).abs() < 0.05).all():
    INT.append("temporal_role is almost unrelated to lexical overlap with the previous segment. Either the label encodes something other than textual continuation, "
               "or lexical overlap is a poor proxy for it; this analysis cannot tell which.")
elif len(au):
    INT.append("temporal_role is detectably related to lexical overlap with the previous segment, so the label partly tracks textual continuation.")

# ---- heterogeneity, truncated traces ------------------------------------------------------------------------------------------
if het:
    flips = [f for f in het_dz.index if (het_sig.loc[f] & (het_dz.loc[f] > 0)).any() and (het_sig.loc[f] & (het_dz.loc[f] < 0)).any()]
    OBS.append("**Heterogeneity.** Among the %d benchmark / difficulty subgroups with at least 8 questions, the MIXED effect changes direction with a CI excluding 0 on both "
               "sides for: %s." % (len(het_dz.columns), ", ".join(flips) if flips else "no feature"))
    if flips:
        UNR.append("The effect of %s points in opposite directions in different subgroups, so its pooled estimate is not a single phenomenon." % ", ".join(flips))
OBS.append("**Truncated traces (kept out of the main analyses).** Truncation rate %s overall; within the same cell, truncated minus complete traces differ in words by %s." % (
    pc(Fall["trunc"].mean()), ci_txt(eff_tr.loc["words"], 1) if "words" in eff_tr.index and not pd.isna(eff_tr.at["words", "ci_lo"]) else "n/a (too few cells with both)"))
sd_ = sens.filter(like="dz:").dropna(how="all")
sign_stable = [c[3:] for c in sd_.columns if (np.sign(sd_[c].dropna()) == np.sign(sd_[c].dropna().iloc[0])).all()]
OBS.append("**Sensitivity.** MIXED effects keep their sign in every scenario (truncated traces included, empty outputs excluded, 4- or 5-sample cells only, length-matched pairs) for: %s." % (
    ", ".join(sign_stable) if sign_stable else "no feature"))

# ---- unresolved ------------------------------------------------------------------------------------------------------------------
unclear = [f for f in mx_summary.index if mx_summary.at[f, "direction"] == "no clear difference"]
if unclear:
    UNR.append("No clear MIXED difference for: %s. Absence of a clear difference with this many questions is not evidence of absence." % ", ".join(unclear))
UNR += [
    "Direction of influence: whether doubt, correction and answer changes lead to errors, or arise because a sample is already going wrong, cannot be decided from observational traces.",
    "Question difficulty: within-cell comparisons hold the question and compute level fixed, but two samples of the same cell can still differ in which solution path they sampled; "
    "ALL-WRONG vs ALL-CORRECT comparisons additionally mix easy and hard questions.",
    "Label validity: roles, answer status and temporal_role come from an automatic labeller. The data-quality checks show internal consistency, not agreement with human judgment.",
    "Divergence candidates are heuristics (first differing phase run; first segment without a counterpart). Whether they mark the point where a trace goes wrong needs manual reading of the context tables.",
]

LIM = [
    "Observational data only: no model was run and no intervention was made; all statements describe association.",
    "Length is measured in words of the thinking text (tokens only supplementary); truncated traces are excluded from the main analyses, which removes the longest traces at low budgets.",
    "MIXED cells are selected on the outcome (both a correct and an incorrect sample exist), which favours questions of intermediate difficulty; ALL-CORRECT cells with a real length spread are few.",
    "At most 5 samples per cell, so within-cell contrasts are noisy; cells are aggregated to questions with equal weight and CIs resample questions, but benchmarks and tiers are unevenly represented.",
    "Many features and terms were examined, p-values are not corrected for multiplicity, and the robustness label is descriptive; treat single isolated effects as hypotheses.",
    "Lexical features (cues, novelty, repetition, compression) are crude proxies; the cue words overlap with the vocabulary that defines the doubt/correction labels.",
    "Outcome-linked answer features (gold_ever, gold_lost, last_matches_gold) are defined through correctness and are excluded from pattern-transfer and not shown as evidence.",
    "Answer extraction from the model output is heuristic (last boxed value or 'answer is' after the thinking), and the agreement with is_correct is reported, not assumed.",
]
FUT = [
    "Read the divergence-candidate contexts for a random and for an extreme sample of pairs and record whether the flagged point is a real change of approach.",
    "Validate a random sample of segment labels (especially doubt vs correction and temporal_role) against human judgment and re-run the key effects on the validated subset.",
    "Add question difficulty (e.g. pooled accuracy over models or samples) as a covariate to separate trajectory effects from difficulty.",
    "Test whether the early part of a trace (first 25% of words) already predicts the outcome within a cell, using cross-validation by question.",
]

report = "\n\n".join(["# Findings (generated from this run)", "## Observed"] + [(s if s.startswith("|") else "- " + s) for s in OBS] +
                     ["## Interpretation (conditional, descriptive)"] + ["- " + s for s in INT] +
                     ["## Unresolved"] + ["- " + s for s in UNR] + ["## Limitations"] + ["- " + s for s in LIM] +
                     ["## Further questions"] + ["- " + s for s in FUT])
with open(os.path.join(TAB_DIR, "findings.md"), "w", encoding="utf-8") as fh:
    fh.write(report + "\n")
display(Markdown(report))
print(report)